# Heat Pump Stock Rally Prediction — V10
Heterogeneous Stacking: XGB + LGB + RF → XGB Meta-Learner

In [51]:
# Cell 1 — Imports (Kernel: diese Zelle zuerst ausführen, dann Cell 2 ff.)
# Standardbibliothek
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone
from pathlib import Path
import base64
import io
import json
import pickle
import subprocess
import time
import warnings

# Daten & HTTP
import numpy as np
import pandas as pd
import requests
import yfinance as yf

# Technische Indikatoren (z. B. add_technical_indicators, RSI in Optuna)
import ta

# ML / Optimierung / Erklärbarkeit
import lightgbm as lgb
import optuna
import shap
import xgboost as xgb
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Visualisierung (Holdout-Plots, SHAP, Auswertungen)
import matplotlib.dates as mdates
import matplotlib.pyplot as plt


In [52]:
# Cell 2 — Configuration

import datetime
import os
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

# ── Google Cloud / BigQuery: ADC + Projekt (vor BQ_PROJECT_ID) ────────────
ADC_PATH = r"C:\Users\HP\AppData\Roaming\gcloud\application_default_credentials.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = ADC_PATH
os.environ["GOOGLE_CLOUD_PROJECT"] = "gedelt-calls"
print(f"Credentials geladen von: {ADC_PATH}")
if not os.path.isfile(ADC_PATH):
    print("WARNUNG: Datei nicht gefunden — Pfad prüfen oder gcloud auth application-default login")

# ── Dates ──────────────────────────────────────────────────────────────────
# Walk-forward: BASE → META → THRESHOLD → FINAL = disjunkte Zeitfenster (Purge zwischen Stufen).
# FINAL = letztes Fenster (Final-Test); TRAIN_END_DATE = END_DATE = Kalenderende des Laufs (aktueller Tag).
START_DATE = '2018-01-01'  # Kurs-Download / Labels; News siehe NEWS_EXTRA_HISTORY_* und NEWS_BQ_*
END_DATE        = datetime.date.today().strftime('%Y-%m-%d')           # Download bis heute
TRAIN_END_DATE  = datetime.date.today().strftime('%Y-%m-%d')           # gleiches Kalenderende wie END_DATE
print(f'Download:  {START_DATE} → {END_DATE}')
print(f'Training:  {START_DATE} → {TRAIN_END_DATE}  (bis zum aktuellen Datum)')


# ── SCORING_ONLY: gespeicherte Modelle + Threshold, ohne Neu-Training ───────
# True  → Zellen 1,2, dann 3–8, 11, 17 (12–16 überspringen Training).
# False → Training; Artefakt wird automatisch nach Phase 5 (Cell 14) geschrieben, ggf. Sicherung in Cell 17.
SCORING_ONLY = True
SCORING_ARTIFACT_PATH = Path("models") / "scoring_artifacts.joblib"
# Wird von save_scoring_artifacts / load_scoring_artifacts auf True gesetzt (Cell 17 speichert nach, falls nötig).
_SCORING_ARTIFACT_SAVED_THIS_SESSION = False


import lib.scoring_persist as scoring_persist


def save_scoring_artifacts(path=None):
    """Speichert Base-Modelle, Meta-Classifier, kalibrierten Threshold und Metadaten."""
    p = Path(path) if path is not None else None
    out = scoring_persist.save_scoring_artifacts(globals(), p)
    globals()["_SCORING_ARTIFACT_SAVED_THIS_SESSION"] = True
    return out


def load_scoring_artifacts(path=None):
    """Lädt Artefakt; setzt u.a. base_models, meta_clf, best_threshold, FEAT_COLS, topk_idx."""
    p = Path(path) if path is not None else None
    r = scoring_persist.load_scoring_artifacts(globals(), p)
    globals()["_SCORING_ARTIFACT_SAVED_THIS_SESSION"] = True
    return r


# ── Ticker universe + sector mapping ───────────────────────────────────────
# DAX40 / MDAX50 / SDAX70 vollständig + internationale Referenzwerte
TICKERS_BY_SECTOR = {
    # ── Wärmepumpen & HVAC ─────────────────────────────────────────────────
    'heat_pump':  ['NIBE-B.ST', '6367.T', 'CARR', 'TT', 'JCI', 'LII',
                   'AOS', 'WSO', '6503.T', 'AALB.AS'],

    # ── Technologie ────────────────────────────────────────────────────────
    # DAX: SAP, IFX  |  MDAX: NEM(Nemetschek), BC8  |  SDAX: TMV, COK, YSN, GFT, NA9, AOF, ELG, AIXA, SMHN, SANT
    'tech':       ['AAPL', 'MSFT', 'NVDA', 'CRM', 'CSCO', 'INTC', 'IBM', 'ASML',
                   'SAP.DE', 'IFX.DE',
                   'NEM.DE', 'BC8.DE',
                   'TMV.DE', 'COK.DE', 'YSN.DE', 'GFT.DE', 'NA9.DE',
                   'AOF.DE', 'ELG.DE', 'AIXA.DE', 'SMHN.DE', 'SANT.DE'],

    # ── Finanzen ───────────────────────────────────────────────────────────
    # DAX: DB1, HNR1  |  MDAX: TLX  |  SDAX: MLP, GLJ, PBB, HABA, WUW, DBAG, HYQ
    'finance':    ['JPM', 'GS', 'AXP', 'V',
                   'ALV.DE', 'MUV2.DE', 'DBK.DE', 'CBK.DE', 'BNP.PA',
                   'DB1.DE', 'HNR1.DE',
                   'TLX.DE',
                   'MLP.DE', 'GLJ.DE', 'PBB.DE', 'WUW.DE', 'DBAG.DE', 'HYQ.DE'],

    # ── Gesundheit ─────────────────────────────────────────────────────────
    # DAX: FME, SHL  |  MDAX: SRT3  |  SDAX: AFX, EUZ, DMP, GXI, DRW3, O2BC, EVT
    'healthcare': ['JNJ', 'UNH', 'MRK', 'AMGN', 'NVS',
                   'BAYN.DE', 'FRE.DE', 'MRK.DE', 'FME.DE', 'SHL.DE',
                   'SRT3.DE',
                   'AFX.DE', 'EUZ.DE', 'DMP.DE', 'GXI.DE', 'DRW3.DE', 'O2BC.DE', 'EVT.DE'],

    # ── Konsumgüter ────────────────────────────────────────────────────────
    # DAX: BEI  |  SDAX: FIE, HBH, DOU, SZU, HFG, KWS, EVD, DHER
    'consumer':   ['KO', 'MCD', 'NKE', 'PG', 'WMT', 'HD', 'DIS',
                   'ADS.DE', 'BOSS.DE', 'HEN3.DE', 'PUM.DE', 'ZAL.DE', 'LVMH.PA',
                   'BEI.DE',
                   'FIE.DE', 'HBH.DE', 'DOU.DE', 'SZU.DE', 'HFG.DE', 'KWS.DE',
                   'EVD.DE', 'DHER.DE'],

    # ── Industrie ──────────────────────────────────────────────────────────
    # DAX: SIE, RHM, MTX, AIR, DHL, G1A  |  MDAX: KBX, HOC, KGX, JUN3, TKA, HAG, R3NK, LHA, FRA, HLAG, RAA
    # SDAX: DUE, JST, SFQ, NOEJ, VOS, WAC, INH, MUX, STM, KSB, HDD
    'industrial': ['CAT', 'BA', 'HON', 'MMM', 'SHW', 'TRV', 'DOW',
                   'SIE.DE', 'RHM.DE', 'MTX.DE', 'AIR.DE', 'DHL.DE', 'G1A.DE',
                   'KBX.DE', 'HOC.DE', 'KGX.DE', 'JUN3.DE', 'TKA.DE', 'HAG.DE',
                   'R3NK.DE', 'LHA.DE', 'FRA.DE', 'HLAG.DE', 'RAA.DE',
                   'DUE.DE', 'JST.DE', 'SFQ.DE', 'NOEJ.DE', 'VOS.DE',
                   'WAC.DE', 'INH.DE', 'MUX.DE', 'STM.DE', 'KSB.DE', 'HDD.DE'],

    # ── Energie ────────────────────────────────────────────────────────────
    # DAX: ENR  |  MDAX: NDX1  |  SDAX: S92, VBK, EKT, PNE3, F3C, VH2
    'energy':     ['CVX', 'XOM', 'BP',
                   'RWE.DE', 'EOAN.DE', 'NEE', 'DUK', 'IBE.MC',
                   'ENR.DE',
                   'NDX1.DE',
                   'S92.DE', 'VBK.DE', 'EKT.DE', 'PNE3.DE', 'F3C.DE', 'VH2.DE'],

    # ── Krypto ─────────────────────────────────────────────────────────────
    'crypto':     ['BTC-USD', 'ETH-USD', 'BNB-USD', 'SOL-USD', 'XRP-USD', 'ADA-USD'],

    # ── Automotive ─────────────────────────────────────────────────────────
    # DAX: VOW3, BMW, MBG, CON, PAH3, P911, DTG  |  MDAX: TGR, SHA
    'automotive': ['VOW3.DE', 'BMW.DE', 'MBG.DE', 'CON.DE', 'PAH3.DE', 'P911.DE',
                   'DTG.DE',
                   'TM', 'STLA', 'GM', 'F',
                   'TGR.DE', 'SHA.DE'],

    # ── Rohstoffe & Chemie ─────────────────────────────────────────────────
    # DAX: BAS, BNR, HEI, SY1  |  MDAX: EVK, LXS, NDA, SDO, FPE3  |  SDAX: ACT, WAF, BFSA, KCO
    'materials':  ['BAS.DE', 'LIN', 'WCH.DE', 'APD', 'LYB', 'BNR.DE', 'NEM',
                   'HEI.DE', 'SY1.DE',
                   'EVK.DE', 'LXS.DE', 'NDA.DE', 'SDO.DE', 'FPE3.DE',
                   'ACT.DE', 'WAF.DE', 'BFSA.DE', 'KCO.DE'],

    # ── Immobilien ─────────────────────────────────────────────────────────
    # DAX: VNA  |  MDAX: LEG  |  SDAX: GYC, DEQ, HABA, PAT
    'real_estate':['VNA.DE', 'LEG.DE', 'O', 'AMT', 'SPG', 'WDP.BR',
                   'GYC.DE', 'DEQ.DE', 'HABA.DE', 'PAT.DE'],

    # ── Telekommunikation ──────────────────────────────────────────────────
    # DAX: DTE  |  MDAX: O2D, UTDI, FNTN  |  SDAX: 1U1
    'telecom':    ['DTE.DE', 'T', 'VZ', 'TMUS', 'ORAN',
                   'O2D.DE', 'UTDI.DE', 'FNTN.DE', '1U1.DE'],

    # ── Medien (neu) ───────────────────────────────────────────────────────
    # DAX: G24  |  MDAX: RRTL  |  SDAX: SPG, BVB, PSM, CWC
    'media':      ['RRTL.DE', 'SPG.DE', 'BVB.DE', 'PSM.DE', 'CWC.DE', 'G24.DE'],
}

# Alphabetically sorted sector labels (0-based)
SECTOR_LABELS = {s: i for i, s in enumerate(sorted(TICKERS_BY_SECTOR.keys()))}
# automotive=0, consumer=1, crypto=2, energy=3, finance=4, healthcare=5,
# heat_pump=6, industrial=7, materials=8, media=9, real_estate=10, tech=11, telecom=12

TICKER_TO_SECTOR = {t: s for s, tl in TICKERS_BY_SECTOR.items() for t in tl}
ALL_TICKERS = [t for tl in TICKERS_BY_SECTOR.values() for t in tl]

# Company display names
COMPANY_NAMES = {
    # Heat pump
    'NIBE-B.ST': 'NIBE', '6367.T': 'Daikin', 'CARR': 'Carrier', 'TT': 'Trane Tech',
    'JCI': 'Johnson Controls', 'LII': 'Lennox', 'AOS': 'A.O. Smith', 'WSO': 'Watsco',
    '6503.T': 'Mitsubishi', 'AALB.AS': 'Aalberts',
    # Tech
    'AAPL': 'Apple', 'MSFT': 'Microsoft', 'NVDA': 'NVIDIA', 'CRM': 'Salesforce',
    'CSCO': 'Cisco', 'INTC': 'Intel', 'IBM': 'IBM',
    'SAP.DE': 'SAP', 'IFX.DE': 'Infineon', 'ASML': 'ASML',
    # Finance
    'JPM': 'JPMorgan', 'GS': 'Goldman Sachs', 'AXP': 'AmEx', 'V': 'Visa',
    'ALV.DE': 'Allianz', 'MUV2.DE': 'Munich Re', 'DBK.DE': 'Deutsche Bank',
    'CBK.DE': 'Commerzbank', 'BNP.PA': 'BNP Paribas',
    # Healthcare
    'JNJ': 'J&J', 'UNH': 'UnitedHealth', 'MRK': 'Merck US', 'AMGN': 'Amgen',
    'BAYN.DE': 'Bayer', 'FRE.DE': 'Fresenius', 'MRK.DE': 'Merck KGaA',
    'SRT3.DE': 'Sartorius', 'NVS': 'Novartis',
    # Consumer
    'KO': 'Coca-Cola', 'MCD': "McDonald's", 'NKE': 'Nike', 'PG': 'P&G',
    'WMT': 'Walmart', 'HD': 'Home Depot', 'DIS': 'Disney',
    'ADS.DE': 'Adidas', 'BOSS.DE': 'Hugo Boss', 'HEN3.DE': 'Henkel',
    'PUM.DE': 'PUMA', 'ZAL.DE': 'Zalando', 'LVMH.PA': 'LVMH',
    # Industrial (DAX)
    'CAT': 'Caterpillar', 'BA': 'Boeing', 'HON': 'Honeywell', 'MMM': '3M',
    'SHW': 'Sherwin-Williams', 'TRV': 'Travelers', 'DOW': 'Dow Inc.',
    'SIE.DE': 'Siemens', 'RHM.DE': 'Rheinmetall', 'MTX.DE': 'MTU Aero',
    'AIR.DE': 'Airbus', 'DHL.DE': 'DHL Group', 'G1A.DE': 'GEA Group',
    # Industrial (MDAX)
    'KBX.DE': 'Knorr-Bremse', 'HOC.DE': 'Hochtief', 'KGX.DE': 'KION Group',
    'JUN3.DE': 'Jungheinrich', 'TKA.DE': 'Thyssenkrupp', 'HAG.DE': 'Hensoldt',
    'R3NK.DE': 'RENK Group', 'LHA.DE': 'Lufthansa', 'FRA.DE': 'Fraport',
    'HLAG.DE': 'Hapag-Lloyd', 'RAA.DE': 'Rational',
    # Industrial (SDAX)
    'DUE.DE': 'Dürr', 'JST.DE': 'JOST Werke', 'SFQ.DE': 'SAF-Holland',
    'NOEJ.DE': 'NORMA Group', 'VOS.DE': 'Vossloh', 'WAC.DE': 'Wacker Neuson',
    'INH.DE': 'INDUS Holding', 'MUX.DE': 'Mutares', 'STM.DE': 'Stabilus',
    'KSB.DE': 'KSB', 'HDD.DE': 'Heidelberger Druck',
    # Energy
    'CVX': 'Chevron', 'XOM': 'ExxonMobil', 'BP': 'BP',
    'RWE.DE': 'RWE', 'EOAN.DE': 'E.ON', 'NEE': 'NextEra', 'DUK': 'Duke Energy',
    'IBE.MC': 'Iberdrola',
    'ENR.DE': 'Siemens Energy', 'NDX1.DE': 'Nordex',
    'S92.DE': 'SMA Solar', 'VBK.DE': 'Verbio', 'EKT.DE': 'Energiekontor',
    'PNE3.DE': 'PNE', 'F3C.DE': 'SFC Energy', 'VH2.DE': 'Friedrich Vorwerk',
    # Crypto
    'BTC-USD': 'Bitcoin', 'ETH-USD': 'Ethereum', 'BNB-USD': 'BNB',
    'SOL-USD': 'Solana', 'XRP-USD': 'XRP', 'ADA-USD': 'Cardano',
    # Automotive (DAX)
    'VOW3.DE': 'Volkswagen', 'BMW.DE': 'BMW', 'MBG.DE': 'Mercedes-Benz',
    'CON.DE': 'Continental', 'PAH3.DE': 'Porsche SE', 'P911.DE': 'Porsche AG',
    'DTG.DE': 'Daimler Truck',
    'TM': 'Toyota', 'STLA': 'Stellantis', 'GM': 'General Motors', 'F': 'Ford',
    # Automotive (MDAX)
    'TGR.DE': 'Traton', 'SHA.DE': 'Schaeffler',
    # Materials (DAX)
    'BAS.DE': 'BASF', 'LIN': 'Linde', 'WCH.DE': 'Wacker Chemie',
    'APD': 'Air Products', 'LYB': 'LyondellBasell', 'BNR.DE': 'Brenntag',
    'NEM': 'Newmont', 'HEI.DE': 'Heidelberg Materials', 'SY1.DE': 'Symrise',
    # Materials (MDAX)
    'EVK.DE': 'Evonik', 'LXS.DE': 'LANXESS', 'NDA.DE': 'Aurubis',
    'SDO.DE': 'K+S', 'FPE3.DE': 'Fuchs SE',
    # Materials (SDAX)
    'ACT.DE': 'AlzChem', 'WAF.DE': 'Siltronic', 'BFSA.DE': 'Befesa',
    'KCO.DE': 'Klöckner & Co',
    # Real Estate
    'VNA.DE': 'Vonovia', 'LEG.DE': 'LEG Immobilien', 'O': 'Realty Income',
    'AMT': 'American Tower', 'SPG': 'Simon Property', 'WDP.BR': 'WDP',
    'GYC.DE': 'Grand City Prop.', 'DEQ.DE': 'Deutsche EuroShop',
    'HABA.DE': 'Hamborner REIT', 'PAT.DE': 'PATRIZIA',
    # Telecom
    'DTE.DE': 'Deutsche Telekom', 'T': 'AT&T', 'VZ': 'Verizon',
    'TMUS': 'T-Mobile', 'ORAN': 'Orange',
    'O2D.DE': 'Telefónica Dtl.', 'UTDI.DE': 'United Internet',
    'FNTN.DE': 'freenet', '1U1.DE': '1&1',
    # Media
    'RRTL.DE': 'RTL Group', 'SPG.DE': 'Springer Nature', 'BVB.DE': 'Borussia Dortmund',
    'PSM.DE': 'ProSieben', 'CWC.DE': 'CEWE', 'G24.DE': 'Scout24',
    # Finance (DAX additions)
    'DB1.DE': 'Deutsche Börse', 'HNR1.DE': 'Hannover Rück',
    # Finance (MDAX)
    'TLX.DE': 'Talanx',
    # Finance (SDAX)
    'MLP.DE': 'MLP', 'GLJ.DE': 'GRENKE', 'PBB.DE': 'Deutsche Pfandbriefbank',
    'HABA.DE': 'Hamborner REIT', 'WUW.DE': 'Wüstenrot & W.',
    'DBAG.DE': 'Deutsche Beteiligungs', 'HYQ.DE': 'Hypoport',
    # Healthcare (DAX additions)
    'FME.DE': 'Fresenius Med.', 'SHL.DE': 'Siemens Healthineers',
    # Healthcare (SDAX)
    'AFX.DE': 'Carl Zeiss Meditec', 'EUZ.DE': 'Eckert & Ziegler',
    'DMP.DE': 'Dermapharm', 'GXI.DE': 'Gerresheimer',
    'DRW3.DE': 'Drägerwerk', 'O2BC.DE': 'Ottobock', 'EVT.DE': 'Evotec',
    # Consumer (DAX additions)
    'BEI.DE': 'Beiersdorf',
    # Consumer (SDAX)
    'FIE.DE': 'Fielmann', 'HBH.DE': 'Hornbach', 'DOU.DE': 'Douglas',
    'SZU.DE': 'Südzucker', 'HFG.DE': 'HelloFresh', 'KWS.DE': 'KWS Saat',
    'EVD.DE': 'CTS Eventim', 'DHER.DE': 'Delivery Hero',
    # Tech (MDAX)
    'NEM.DE': 'Nemetschek', 'BC8.DE': 'Bechtle',
    # Tech (SDAX)
    'TMV.DE': 'TeamViewer', 'COK.DE': 'CANCOM', 'YSN.DE': 'secunet',
    'GFT.DE': 'GFT Technologies', 'NA9.DE': 'Nagarro', 'AOF.DE': 'ATOSS Software',
    'ELG.DE': 'Elmos Semiconductor', 'AIXA.DE': 'Aixtron',
    'SMHN.DE': 'SÜSS MicroTec', 'SANT.DE': 'Kontron',
}

# ── Fixed pipeline constants ─────────────────────────────────────────────────────
N_WF_SPLITS           = 5     # Walk-forward CV folds
GDELT_CHUNK_DAYS      = 45    # GDELT API: kürzere Sub-Chunks (vorher fest 90d)
OPT_MIN_PRECISION_BASE   = 0.6   # Phase 1 Base-XGB
OPT_MIN_PRECISION_META   = 0.7   # Phase 4 Meta-Learner
OPT_MIN_PRECISION_THRESHOLD = 0.95  # Phase 5 Threshold-Kalibrierung / PR-Plots
OPT_MIN_PRECISION = OPT_MIN_PRECISION_THRESHOLD  # Alias: find_precision_threshold, Reports
# Phase 5: Mindestanzahl positiver Roh-Vorhersagen (prob>=t), damit Precision nicht trivial ist
PHASE5_MIN_SIGNALS    = 5

# ── Optuna mode toggle ────────────────────────────────────────────────────────
# True  (Option A): Optuna also searches XGBoost model hyperparameters.
#                   More expressive search, but only XGBoost is tuned — other
#                   base models (LGB/RF/ET/LR) are not — asymmetric optimisation.
# False (Option B): Optuna only searches target/filter params (ohne threshold — der kommt in Phase 5).
#                   All base models use fixed SEED_PARAMS — consistent and faster.
OPT_MODEL_HYPERPARAMS = True
# True: Phase-1-Optuna optimiert Rally-/Label-Parameter (return_window, rally_threshold, …).
# False: feste Regel — grüner Bereich = Rendite > FIXED_Y_RALLY_THRESHOLD innerhalb von
# FIXED_Y_WINDOW_MIN..FIXED_Y_WINDOW_MAX Handelstagen; Targets nur nach fester Vorschrift.
OPT_OPTIMIZE_Y_TARGETS = False
FIXED_Y_WINDOW_MIN = 3
FIXED_Y_WINDOW_MAX = 10
FIXED_Y_RALLY_THRESHOLD = 0.06
FIXED_Y_SEGMENT_SPLIT = 5
EARLY_STOPPING_ROUNDS = 30
N_OPTUNA_TRIALS       = 300  # erhöht wegen News-Feature-Grid
# Nur Phase-1-Studie (Base-XGB): hier drosseln, wenn Optuna trotz kleinem Universum langsam ist.
# UNIVERSE_FRACTION verkürzt Datenpipeline; jeder Trial kostet noch rebuild_target + WF×XGB.
OPTUNA_WF_SPLITS      = None  # None → N_WF_SPLITS; z.B. 3 weniger Folds pro Trial
N_META_TRIALS         = 300
# Cell 13a: SHAP → Meta-Stacking — Top-K Roh-Features neben Base-Probabilities (Cell 14)
META_SHAP_TOP_K       = 10
RANDOM_STATE          = 42
N_WORKERS             = os.cpu_count()

# ── Optuna initial values (überschrieben durch Cell 13 nach Optimierung) ─────────
# create_target() in Cell 11 läuft VOR Optuna — Werte sollten mit SEED_PARAMS konsistent sein.
RETURN_WINDOW        = 4     # zuletzt: Optuna trial 154 (Base-XGB)
RALLY_THRESHOLD      = 0.07423891180366396
# „Early-only“-Label: positives = Vorlauf (LEAD_DAYS) + kurze Einstiegszone (ENTRY_DAYS),
# nicht die gesamte grüne Rally. Optuna sucht entry_days in [1, 3] (fest in optuna_train).
LEAD_DAYS            = 4
ENTRY_DAYS           = 3
# Positives Label nur, wenn ab diesem Tag noch mind. so viele Rally-Handelstage „übrig“ sind
# (inkl. heute bis Segmentende) — kurze grüne Reste / Rally-Ende werden nicht als Ziel markiert.
MIN_RALLY_TAIL_DAYS  = 5
CONSECUTIVE_DAYS     = 2
SIGNAL_COOLDOWN_DAYS = 4

# ── Anti-Peak (nur apply_signal_filters / Website — nicht in Optuna-CV) ─────
# Viele Fehlsignale sitzen direkt am lokalen Kurs-High; Filter verlangt Abstand zum N-Tage-High.
SIGNAL_SKIP_NEAR_PEAK = True
PEAK_LOOKBACK_DAYS = 20
PEAK_MIN_DIST_FROM_HIGH_PCT = 0.012   # mind. 1.2 % unter Rolling-High (tunable 0.008–0.02)
SIGNAL_MAX_RSI = 78.0                 # kein Kauf-Signal wenn RSI darüber; None = aus

# ── Trainings-Universum ─────────────────────────────────────────────────────
# Maximale Anzahl Wertpapiere im TRAIN_BASE-Set — nur bei SPLIT_MODE="ticker" (Legacy).
# Bei SPLIT_MODE="time" nutzen alle Stufen dieselben Ticker (TIME_SPLIT_FRAC_*).
# Empfehlung: 30–60. Höher = langsamer, aber potenziell besseres Modell.
MAX_TRAIN_TICKERS   = 45
# Schneller Run: nur einen zufälligen Anteil der geladenen Ticker (nach assemble_features)
UNIVERSE_FRACTION = 1  # 1.0 = alle; <1 wählt Tickers VOR Download/Features (nicht erst danach)
UNIVERSE_SAMPLE_SEED = 42
# Train/Test-Split: "time" = gleiches Ticker-Universum, disjunkte Zeiträume (empfohlen).
# "ticker" = Legacy: verschiedene Ticker pro Stufe, gleicher Kalender bis TRAIN_END_DATE.
SPLIT_MODE = "time"
# Anteile auf allen Handelstagen [START … TRAIN_END_DATE]; walk-forward, keine Überlappung der Stufen (Purge Base→Meta).
TIME_SPLIT_FRAC_BASE = 0.45
TIME_SPLIT_FRAC_META = 0.25
TIME_SPLIT_FRAC_THRESHOLD = 0.15
# Rest = FINAL: letztes Fenster = eigentlicher Final-Test bis zum Kalenderende (s. TRAIN_END_DATE).
TIME_PURGE_TRADING_DAYS = 5

# ── Indicator parameter grids ───────────────────────────────────────────────
RSI_WINDOWS  = [7, 10, 14, 21]
BB_WINDOWS   = [15, 20, 25]
SMA_WINDOWS  = [30, 50, 70]

# ── Seed params für enqueue_trial (Optuna Base-XGB) ─────────────────────────
# Zuletzt aus Notebook-Lauf: bester Trial 154 (score≈49, 280 Trials, Option A).
SEED_PARAMS = dict(
    return_window=4,
    rally_threshold=0.07423891180366396,
    lead_days=4,
    entry_days=3,
    min_rally_tail_days=5,
    consecutive_days=2,
    signal_cooldown_days=4,
    threshold=0.6338651351186617,
    rsi_window=14,
    bb_window=25,
    sma_window=50,
    news_mom_w=3,
    news_vol_ma=20,
    news_tone_roll=0,
    news_extra_zscore_w=20,
    news_extra_tone_accel=True,
    news_extra_macro_sec_diff=True,
    grow_policy='depthwise',
    max_leaves=910,
    max_bin=64,
    max_depth=6,
    min_child_weight=9,
    gamma=5.7204046577916365,
    reg_alpha=0.00044267707418928947,
    reg_lambda=1.1663163488001458,
    learning_rate=0.013316606543583374,
    n_estimators=1263,
    subsample=0.5896905968497552,
    colsample_bytree=0.5602711933350929,
    focal_gamma=0.5063350144294034,
    focal_alpha=0.38191769324282254,
    signal_skip_near_peak=True,
    peak_lookback_days=20,
    peak_min_dist_from_high_pct=0.012,
    signal_max_rsi=78.0,
)

# ── News / GDELT grids (Optuna wählt ein Tag-Tripel) ─────────────────────────
USE_NEWS_SENTIMENT = True
# Momentum auf geglätteter Ton-Serie; Vol-MA für Spike; zusätzliche Ton-Glättung vor Momentum
NEWS_MOM_WINDOWS = [3, 5, 7]
NEWS_VOL_MA_WINDOWS = [10, 20]
NEWS_TONE_ROLL_WINDOWS = [0, 3]  # 0 = keine zusätzliche Ton-Glättung

# Zusätzliche News-Spalten — Optuna Phase 1 wählt je Trial ein Tripel (wie news_mom_w …)
NEWS_EXTRA_ZSCORE_WINDOWS = [0, 10, 20, 30]   # 0 = keine tone_z/vol_z; >0 → Spalten …_tone_z_w{w}
NEWS_EXTRA_TONE_ACCEL_OPTIONS = [False, True]
NEWS_EXTRA_MACRO_SEC_DIFF_OPTIONS = [False, True]

NEWS_CACHE_DIR = os.path.join(os.getcwd(), 'data')
NEWS_CACHE_FILE = os.path.join(NEWS_CACHE_DIR, 'news_gdelt_cache.pkl')
# News: BigQuery (GDELT events, kein API-Rate-Limit) oder Legacy HTTP-Doc-API
NEWS_SOURCE = "bigquery"  # "bigquery" | "gdelt_api"
# GCP-Projekt mit BigQuery-Billing. Auth: gcloud auth application-default login
# oder GOOGLE_APPLICATION_CREDENTIALS. Wenn leer: Notebook ermittelt Projekt (Auth/gcloud).
BQ_PROJECT_ID = "gedelt-calls"
os.environ["GOOGLE_CLOUD_PROJECT"] = "gedelt-calls"
# Öffentliches GDELT (Projekt gdelt-bq). **events** hat keine Theme-Spalten — News-Features nutzen **GKG**.
BQ_USE_GKG_TABLE = True
GDELT_BQ_EVENTS_TABLE = "`gdelt-bq.gdeltv2.gkg_partitioned`"
# Kosten: GKG + BQ_SINGLE_SCAN → 1 Query pro Lauf (Vereinigung der Lücken); sonst 1 Query pro Kanal/Lücke.
# News vs. Kurs: optional länger (Kalenderjahre). Überschreibt NEWS_EXTRA_* wenn gesetzt.
# Vor Kurs-Start: Warmup für Rollings/Lags/Z-Scores am ersten Kurstag (empfohlen: 1).
NEWS_EXTRA_HISTORY_YEARS_BEFORE = 1
# Nach Kurs-Ende: selten nötig; z. B. wenn Nachrichtenlage mit Verzug zur letzten Kurszeile passen soll.
NEWS_EXTRA_HISTORY_YEARS_AFTER = 0
# Explizite Grenzen (setzt NEWS_EXTRA_* außer Kraft): None = automatisch wie oben + Kurs-START/END
NEWS_BQ_START_DATE = None  # z.B. "2023-01-01" — kürzer = weniger gescannte GB
NEWS_BQ_END_DATE = None    # z.B. None oder gleiches END_DATE
# Nur wenn BQ_USE_GKG_TABLE False und du eine andere Tabelle meinst (Legacy):
BQ_EVENT_DATE_FIELD = "SQLDATE"
BQ_THEMES_COLUMN = "V2Themes"
BQ_USE_PARTITION_FILTER = True  # Pflicht: _PARTITIONTIME — sonst Full-Table-Scan
BQ_SINGLE_SCAN = True          # True: 1× GKG-Query mit IF/COUNTIF pro Kanal (nicht 14×)
# Makro: meist weltweit. Sektor: EU+US (Tipps) — None oder [] = kein Länderfilter
BQ_MACRO_GEO_COUNTRIES = None
BQ_SECTOR_USE_GEO_FILTER = True
BQ_SECTOR_GEO_COUNTRIES = ["EU", "US"]
GDELT_REQUEST_DELAY_SEC = 6.0  # nur gdelt_api: ~1 Request / 5s
NEWS_GDELT_SECONDS_PER_CALL = 6.0  # nur gdelt_api: Throttle + Zeit-Schätzung

MACRO_NEWS_QUERY = "heat pump OR HVAC OR refrigerant OR air conditioning"
SECTOR_KEYWORDS = {
    'heat_pump':  ['heat pump', 'HVAC', 'refrigerant', 'air conditioning', 'Daikin', 'Carrier'],
    'tech':       ['semiconductor', 'AI chip', 'cloud computing', 'software'],
    'finance':    ['interest rate', 'bank earnings', 'Fed', 'credit'],
    'healthcare': ['FDA approval', 'clinical trial', 'pharma', 'biotech'],
    'consumer':   ['consumer spending', 'retail sales', 'inflation'],
    'industrial': ['manufacturing PMI', 'industrial output', 'capex'],
    'energy':     ['oil price', 'crude', 'OPEC', 'natural gas'],
    'crypto':     ['Bitcoin', 'Ethereum', 'crypto regulation', 'DeFi'],
    'automotive': ['automotive industry', 'electric vehicle', 'car manufacturing'],
    'materials':  ['chemical industry', 'steel', 'commodities'],
    'real_estate': ['real estate', 'mortgage rates', 'housing market'],
    'telecom':    ['telecom', '5G', 'broadband'],
    'media':      ['media earnings', 'streaming', 'advertising'],
}

# Makro-Kanal: Zinsen/Inflation (alle Ticker) — eigene WHERE-Clause
MACRO_BQ_THEME_WHERE = (
    "(V2Themes LIKE '%ECON_INTERESTRATES%' OR V2Themes LIKE '%ECON_INFLATION%' "
    "OR V2Themes LIKE '%MONETARY%' OR V2Themes LIKE '%ECON_CENTRALBANK%')"
)

# V2Themes-Filter pro Sektor-Kanal (ohne macro)
SECTOR_BQ_THEME_WHERE = {
    'heat_pump': (
        "(V2Themes LIKE '%ENV_GREEN_ENERGY%' OR V2Themes LIKE '%ECON_SUBSIDIES%' "
        "OR V2Themes LIKE '%ENV_ENERGY%')"
    ),
    'tech': (
        "(V2Themes LIKE '%TECH%' OR V2Themes LIKE '%ECON_%' OR V2Themes LIKE '%AI_%' "
        "OR V2Themes LIKE '%SEMICONDUCTOR%')"
    ),
    'finance': (
        "(V2Themes LIKE '%ECON_%' OR V2Themes LIKE '%FINANCE%' OR V2Themes LIKE '%BANK%' "
        "OR V2Themes LIKE '%MONETARY%' OR V2Themes LIKE '%INFLATION%')"
    ),
    'healthcare': (
        "(V2Themes LIKE '%HEALTH%' OR V2Themes LIKE '%MEDICAL%' OR V2Themes LIKE '%FDA%' "
        "OR V2Themes LIKE '%PHARMACEUTICAL%')"
    ),
    'consumer': (
        "(V2Themes LIKE '%CONSUMER%' OR V2Themes LIKE '%RETAIL%' OR V2Themes LIKE '%ECON_%' "
        "OR V2Themes LIKE '%INFLATION%')"
    ),
    'industrial': (
        "(V2Themes LIKE '%MANUFACTURING%' OR V2Themes LIKE '%ECON_%' OR V2Themes LIKE '%INDUSTRY%')"
    ),
    'energy': (
        "(V2Themes LIKE '%ENERGY%' OR V2Themes LIKE '%OIL%' OR V2Themes LIKE '%GAS%' "
        "OR V2Themes LIKE '%PETROLEUM%' OR V2Themes LIKE '%OPEC%')"
    ),
    'crypto': (
        "(V2Themes LIKE '%CRYPTO%' OR V2Themes LIKE '%BITCOIN%' OR V2Themes LIKE '%BLOCKCHAIN%' "
        "OR V2Themes LIKE '%ETHEREUM%')"
    ),
    'automotive': (
        "(V2Themes LIKE '%AUTO%' OR V2Themes LIKE '%VEHICLE%' OR V2Themes LIKE '%ELECTRIC%' "
        "OR V2Themes LIKE '%AUTOMOTIVE%')"
    ),
    'materials': (
        "(V2Themes LIKE '%STEEL%' OR V2Themes LIKE '%COMMODITY%' OR V2Themes LIKE '%CHEMICAL%')"
    ),
    'real_estate': (
        "(V2Themes LIKE '%REALESTATE%' OR V2Themes LIKE '%HOUSING%' OR V2Themes LIKE '%MORTGAGE%' "
        "OR V2Themes LIKE '%PROPERTY%')"
    ),
    'telecom': (
        "(V2Themes LIKE '%TELECOM%' OR V2Themes LIKE '%5G%' OR V2Themes LIKE '%BROADBAND%')"
    ),
    'media': (
        "(V2Themes LIKE '%MEDIA%' OR V2Themes LIKE '%STREAMING%' OR V2Themes LIKE '%ADVERTISING%')"
    ),
}

# Optional: Firmen-Attention (V2Organizations) z. B. Nibe/Daikin — eigene Query, nicht im Merge,
# bis Kanäle/Ticker-Join im Modell vorgesehen sind (kleine SDAX-Namen: oft zu wenig Volumen).

def news_feat_tag(mom_w, vol_ma, tone_roll):
    return f"{int(mom_w)}_{int(vol_ma)}_{int(tone_roll)}"

def _news_base_cols(tag):
    return [
        f'news_macro_{tag}_tone',
        f'news_macro_{tag}_vol',
        f'news_macro_{tag}_tone_mom',
        f'news_macro_{tag}_vol_spike',
        f'news_macro_{tag}_tone_l1', f'news_macro_{tag}_tone_l3', f'news_macro_{tag}_tone_l5',
        f'news_macro_{tag}_vol_l1', f'news_macro_{tag}_vol_l3', f'news_macro_{tag}_vol_l5',
        f'news_sec_{tag}_tone',
        f'news_sec_{tag}_vol',
        f'news_sec_{tag}_tone_mom',
        f'news_sec_{tag}_vol_spike',
        f'news_sec_{tag}_tone_l1', f'news_sec_{tag}_tone_l3', f'news_sec_{tag}_tone_l5',
        f'news_sec_{tag}_vol_l1', f'news_sec_{tag}_vol_l3', f'news_sec_{tag}_vol_l5',
    ]

def build_news_model_cols(tag, zscore_w=0, use_accel=False, use_cross=False):
    """Spaltenliste für ein gewähltes Tag-Tripel; zscore_w=0 schließt tone_z/vol_z aus."""
    out = _news_base_cols(tag)
    zw = int(zscore_w) if zscore_w is not None else 0
    if zw > 0:
        sw = f'_w{zw}'
        out.extend([
            f'news_macro_{tag}_tone_z{sw}', f'news_macro_{tag}_vol_z{sw}',
            f'news_sec_{tag}_tone_z{sw}', f'news_sec_{tag}_vol_z{sw}',
        ])
    if use_accel:
        out.extend([f'news_macro_{tag}_tone_accel', f'news_sec_{tag}_tone_accel'])
    if use_cross:
        out.append(f'news_cross_{tag}_macro_minus_sec_tone')
    return out

def all_news_model_cols():
    cols = []
    for m in NEWS_MOM_WINDOWS:
        for v in NEWS_VOL_MA_WINDOWS:
            for r in NEWS_TONE_ROLL_WINDOWS:
                tag = news_feat_tag(m, v, r)
                cols.extend(_news_base_cols(tag))
                for z in NEWS_EXTRA_ZSCORE_WINDOWS:
                    if z > 0:
                        sw = f'_w{int(z)}'
                        cols.extend([
                            f'news_macro_{tag}_tone_z{sw}', f'news_macro_{tag}_vol_z{sw}',
                            f'news_sec_{tag}_tone_z{sw}', f'news_sec_{tag}_vol_z{sw}',
                        ])
                if True in NEWS_EXTRA_TONE_ACCEL_OPTIONS:
                    cols.extend([f'news_macro_{tag}_tone_accel', f'news_sec_{tag}_tone_accel'])
                if True in NEWS_EXTRA_MACRO_SEC_DIFF_OPTIONS:
                    cols.append(f'news_cross_{tag}_macro_minus_sec_tone')
    return cols

def build_technical_cols(rsi_w, bb_w, sma_w):
    return [
        f'rsi_{rsi_w}',
        f'bb_pband_{bb_w}',
        'macd_diff',
        'vol_stress',
        'drawdown',
        'adx',
        'vol_ratio',
        f'bb_x_rsi_{bb_w}_{rsi_w}',
        f'rsi_delta_3d_{rsi_w}',
        f'bb_slope_5d_{bb_w}',
        f'bb_delta_3d_{bb_w}',
        'adx_delta_3d',
        'momentum_accel',
        f'rsi_weekly_{rsi_w}',
        f'sma_cross_20_{sma_w}',
        'close_vs_sma200',
        'sma200_delta_5d',
        'drawdown_252d',
        f'market_breadth_{sma_w}',
        'volume_zscore',
        f'sector_avg_rsi_{rsi_w}',
        'btc_momentum',
        'month',
        'sector_id',
    ]

def build_feature_cols(
    rsi_w, bb_w, sma_w,
    news_mom_w=None, news_vol_ma=None, news_tone_roll=None,
    news_extra_zscore_w=None, news_extra_tone_accel=None, news_extra_macro_sec_diff=None,
):
    out = build_technical_cols(rsi_w, bb_w, sma_w)
    if USE_NEWS_SENTIMENT:
        if news_mom_w is None:
            news_mom_w = NEWS_MOM_WINDOWS[len(NEWS_MOM_WINDOWS) // 2]
            news_vol_ma = NEWS_VOL_MA_WINDOWS[len(NEWS_VOL_MA_WINDOWS) // 2]
            news_tone_roll = NEWS_TONE_ROLL_WINDOWS[0]
        if news_extra_zscore_w is None:
            nz = [z for z in NEWS_EXTRA_ZSCORE_WINDOWS if z > 0]
            news_extra_zscore_w = nz[len(nz) // 2] if nz else 0
        if news_extra_tone_accel is None:
            news_extra_tone_accel = True if True in NEWS_EXTRA_TONE_ACCEL_OPTIONS else False
        if news_extra_macro_sec_diff is None:
            news_extra_macro_sec_diff = True if True in NEWS_EXTRA_MACRO_SEC_DIFF_OPTIONS else False
        tag = news_feat_tag(news_mom_w, news_vol_ma, news_tone_roll)
        out = out + build_news_model_cols(
            tag,
            zscore_w=news_extra_zscore_w,
            use_accel=bool(news_extra_tone_accel),
            use_cross=bool(news_extra_macro_sec_diff),
        )
    return out

def build_rename_map(
    rsi_w, bb_w, sma_w,
    news_mom_w=None, news_vol_ma=None, news_tone_roll=None,
    news_extra_zscore_w=None, news_extra_tone_accel=None, news_extra_macro_sec_diff=None,
):
    m = {
        f'rsi_{rsi_w}':              f'RSI ({rsi_w}d)',
        f'bb_pband_{bb_w}':          f'Bollinger %B ({bb_w}d)',
        f'bb_x_rsi_{bb_w}_{rsi_w}': f'BB({bb_w}) × RSI({rsi_w})',
        f'sma_cross_20_{sma_w}':     f'SMA20 / SMA{sma_w}',
        f'market_breadth_{sma_w}':   f'Breadth (SMA{sma_w})',
        f'sector_avg_rsi_{rsi_w}':   f'Sector Avg RSI ({rsi_w}d)',
        f'rsi_delta_3d_{rsi_w}':     f'RSI Δ3d ({rsi_w}d)',
        f'bb_slope_5d_{bb_w}':       f'BB Slope 5d ({bb_w}d)',
        f'bb_delta_3d_{bb_w}':       f'BB Δ3d ({bb_w}d)',
        f'rsi_weekly_{rsi_w}':       f'RSI Weekly ({rsi_w}d)',
        'macd_diff':       'MACD Histogram',
        'vol_stress':      'Vol Stress',
        'drawdown':        'Drawdown 60d',
        'adx':             'ADX',
        'vol_ratio':       'Vol Ratio 5/20d',
        'adx_delta_3d':    'ADX Δ3d',
        'momentum_accel':  'Momentum Accel',
        'close_vs_sma200': 'Close / SMA200',
        'sma200_delta_5d': 'SMA200 Ratio Δ 5d',
        'drawdown_252d':   'Drawdown 252d',
        'volume_zscore':   'Volume Z-Score',
        'btc_momentum':    'BTC Momentum',
        'month':           'Month',
        'sector_id':       'Sector ID',
    }
    if USE_NEWS_SENTIMENT and news_mom_w is not None and news_vol_ma is not None and news_tone_roll is not None:
        if news_extra_zscore_w is None:
            nz = [z for z in NEWS_EXTRA_ZSCORE_WINDOWS if z > 0]
            news_extra_zscore_w = nz[len(nz) // 2] if nz else 0
        if news_extra_tone_accel is None:
            news_extra_tone_accel = True if True in NEWS_EXTRA_TONE_ACCEL_OPTIONS else False
        if news_extra_macro_sec_diff is None:
            news_extra_macro_sec_diff = True if True in NEWS_EXTRA_MACRO_SEC_DIFF_OPTIONS else False
        tag = news_feat_tag(news_mom_w, news_vol_ma, news_tone_roll)
        for c in build_news_model_cols(
            tag,
            zscore_w=news_extra_zscore_w,
            use_accel=bool(news_extra_tone_accel),
            use_cross=bool(news_extra_macro_sec_diff),
        ):
            m[c] = c.replace('_', ' ')
    return m

print('Configuration loaded.')
print(f'Total tickers: {len(ALL_TICKERS)}')
print(f'Sector labels: {SECTOR_LABELS}')


Credentials geladen von: C:\Users\HP\AppData\Roaming\gcloud\application_default_credentials.json
Download:  2018-01-01 → 2026-04-10
Training:  2018-01-01 → 2026-04-10  (bis zum aktuellen Datum)
Configuration loaded.
Total tickers: 203
Sector labels: {'automotive': 0, 'consumer': 1, 'crypto': 2, 'energy': 3, 'finance': 4, 'healthcare': 5, 'heat_pump': 6, 'industrial': 7, 'materials': 8, 'media': 9, 'real_estate': 10, 'tech': 11, 'telecom': 12}


In [53]:
# Cell 3 — Helpers

def _strip_tz(series):
    """Normalize DatetimeIndex or datetime Series to tz-naive midnight."""
    if hasattr(series, 'dt'):
        if series.dt.tz is not None:
            series = series.dt.tz_convert(None)
        return series.dt.normalize()
    if isinstance(series, pd.DatetimeIndex):
        if series.tz is not None:
            series = series.tz_convert(None)
        return series.normalize()
    return series


def make_focal_objective(gamma, alpha):
    """
    Returns a custom objective function for XGBoost/LightGBM implementing
    Focal Loss with given focusing parameter gamma and class weight alpha.

    L = -alpha_t * (1 - p_t)^gamma * log(p_t)
    where p_t = p if y=1 else (1-p), alpha_t = alpha if y=1 else (1-alpha)
    """
    def focal_obj(y_pred, dtrain):
        y_true = dtrain.get_label()
        p = 1.0 / (1.0 + np.exp(-y_pred))          # sigmoid
        p = np.clip(p, 1e-7, 1 - 1e-7)

        # Gradient
        grad = (
            y_true * alpha * (1 - p) ** gamma
            * (gamma * p * np.log(p) + p - 1)
            + (1 - y_true) * (1 - alpha) * p ** gamma
            * (p - gamma * (1 - p) * np.log(1 - p))
        )

        # Hessian (weighted logistic approximation)
        p_t   = np.where(y_true == 1, p, 1 - p)
        a_t   = np.where(y_true == 1, alpha, 1 - alpha)
        hess  = np.maximum(a_t * (1 - p_t) ** gamma * p * (1 - p), 1e-7)
        return grad, hess

    return focal_obj


def make_focal_objective_lgb(gamma, alpha):
    """LightGBM version of focal loss objective (raw logit input)."""
    def focal_obj_lgb(y_pred, dataset):
        y_true = dataset.get_label()
        p = 1.0 / (1.0 + np.exp(-y_pred))
        p = np.clip(p, 1e-7, 1 - 1e-7)

        grad = (
            y_true * alpha * (1 - p) ** gamma
            * (gamma * p * np.log(p) + p - 1)
            + (1 - y_true) * (1 - alpha) * p ** gamma
            * (p - gamma * (1 - p) * np.log(1 - p))
        )

        p_t  = np.where(y_true == 1, p, 1 - p)
        a_t  = np.where(y_true == 1, alpha, 1 - alpha)
        hess = np.maximum(a_t * (1 - p_t) ** gamma * p * (1 - p), 1e-7)
        return grad, hess

    return focal_obj_lgb


print('Helpers defined.')

Helpers defined.


In [54]:
# Cell 4 — 1: Load stock data
import yfinance as yf
import pandas as pd  # falls Cell 1 (Imports) noch nicht gelaufen ist

def load_stock_data(tickers=ALL_TICKERS, start=START_DATE, end=END_DATE):
    """
    Single bulk yfinance download with threads=False to avoid data corruption bug.
    Returns a DataFrame with columns [Date, close, volume, ticker, company].
    """
    print(f'Downloading {len(tickers)} tickers from {start} to {end}...')
    raw = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        threads=False,   # CRITICAL: threads=True corrupts multi-ticker data
        progress=False,
    )

    frames = []
    for ticker in tickers:
        try:
            if isinstance(raw.columns, pd.MultiIndex):
                close  = raw['Close'][ticker].dropna()
                volume = raw['Volume'][ticker].reindex(close.index).fillna(0)
            else:
                close  = raw['Close'].dropna()
                volume = raw['Volume'].reindex(close.index).fillna(0)

            if len(close) < 100:
                print(f'  Skipping {ticker}: only {len(close)} rows')
                continue

            df = pd.DataFrame({'close': close, 'volume': volume})
            df.index = _strip_tz(df.index)
            df = df.reset_index().rename(columns={'index': 'Date', 'Price': 'Date'})
            if 'Date' not in df.columns:
                df = df.rename(columns={df.columns[0]: 'Date'})
            df['Date']    = _strip_tz(df['Date'])
            df['ticker']  = ticker
            df['company'] = COMPANY_NAMES.get(ticker, ticker)
            frames.append(df)
        except Exception as e:
            print(f'  Error {ticker}: {e}')

    result = pd.concat(frames, ignore_index=True)
    result['Date'] = pd.to_datetime(result['Date'])
    print(f'Loaded {result["ticker"].nunique()} tickers, {len(result):,} rows.')
    return result

In [55]:
# Cell 5 — 2: Target vector

def _create_target_one_ticker(df_ticker, lead_days=None, entry_days=None,
                              return_window=None, rally_threshold=None,
                              min_rally_tail_days=None):
    """
    For a single ticker DataFrame (sorted by Date), compute:
      - rally[]: 1 where the return_window-day cumulative return >= rally_threshold
      - target[]: 1 in Vorlauf + Einstiegszone nur, wenn die Rally-Segmentlänge es hergibt:
        ab dem jeweiligen Tag müssen noch mindestens min_rally_tail_days Rally-Handelstage
        bis zum Segmentende folgen (damit nach Einstieg noch genug grüne Phase übrig ist).
    Params default to globals when None.
    """
    ld = LEAD_DAYS       if lead_days       is None else lead_days
    ed = ENTRY_DAYS      if entry_days      is None else entry_days
    mt = MIN_RALLY_TAIL_DAYS if min_rally_tail_days is None else int(min_rally_tail_days)
    rw = RETURN_WINDOW   if return_window   is None else return_window
    rt = RALLY_THRESHOLD if rally_threshold is None else rally_threshold

    close = df_ticker['close'].values.astype(np.float64)
    n = len(close)
    window = rw

    # Rolling cumulative return (product of (1+daily_ret) over window)
    daily_ret = np.full(n, np.nan)
    daily_ret[1:] = close[1:] / close[:-1] - 1.0

    cum_ret = np.full(n, np.nan)
    for i in range(window - 1, n):
        product = 1.0
        for j in range(i - window + 1, i + 1):
            if not np.isnan(daily_ret[j]):
                product *= (1.0 + daily_ret[j])
        cum_ret[i] = product - 1.0

    # Mark rally periods (backwards from qualifying end-day)
    rally = np.zeros(n, dtype=np.int8)
    for end_idx in range(n):
        if not np.isnan(cum_ret[end_idx]) and cum_ret[end_idx] >= rt:
            start_idx = max(0, end_idx - window + 1)
            rally[start_idx:end_idx + 1] = 1

    # Vorlauf + Einstiegszone nur, wenn genug Rally „Restlauf“ (kein positives Label am Rally-Ende)
    target = np.zeros(n, dtype=np.int8)
    i = 0
    while i < n:
        if rally[i] != 1:
            i += 1
            continue
        if i > 0 and rally[i - 1] == 1:
            i += 1
            continue
        start = i
        j = i
        while j < n and rally[j] == 1:
            j += 1
        end = j - 1
        seg_len = end - start + 1
        pre_start = max(0, start - ld)
        if seg_len >= mt:
            for k in range(pre_start, start):
                target[k] = 1
        for k in range(start, min(n, end + 1, start + ed)):
            if end - k + 1 >= mt:
                target[k] = 1
        i = j

    return rally, target


def _create_target_one_ticker_fixed_bands(df_ticker):
    """
    Feste Label-Regel (wenn OPT_OPTIMIZE_Y_TARGETS False):
      Grüner Bereich: Es gibt ein Fenster w in [FIXED_Y_WINDOW_MIN, FIXED_Y_WINDOW_MAX],
      über das die kumulierte Rendite (Produkt der Tagesrenditen) >= FIXED_Y_RALLY_THRESHOLD ist;
      alle Tage dieses Fensters gehören zur grünen Phase (Vereinigung wie bisher).

      Pro zusammenhängendem grünen Segment [start, end] mit Länge L:
        L < FIXED_Y_SEGMENT_SPLIT: Target = 3 Tage vor start + erste 2 grüne Tage.
        L >= FIXED_Y_SEGMENT_SPLIT: Target = 3 Tage vor start + alle grünen Tage außer den letzten 3.
    """
    w_lo = int(globals().get("FIXED_Y_WINDOW_MIN", 3))
    w_hi = int(globals().get("FIXED_Y_WINDOW_MAX", 10))
    rt = float(globals().get("FIXED_Y_RALLY_THRESHOLD", 0.06))
    split = int(globals().get("FIXED_Y_SEGMENT_SPLIT", 5))

    close = df_ticker["close"].values.astype(np.float64)
    n = len(close)
    daily_ret = np.full(n, np.nan)
    daily_ret[1:] = close[1:] / close[:-1] - 1.0

    rally = np.zeros(n, dtype=np.int8)
    for end_idx in range(n):
        for w in range(w_lo, w_hi + 1):
            if end_idx < w - 1:
                continue
            product = 1.0
            bad = False
            for j in range(end_idx - w + 1, end_idx + 1):
                dr = daily_ret[j]
                if np.isnan(dr):
                    bad = True
                    break
                product *= 1.0 + dr
            if bad:
                continue
            if product - 1.0 >= rt:
                start_idx = max(0, end_idx - w + 1)
                rally[start_idx : end_idx + 1] = 1

    target = np.zeros(n, dtype=np.int8)
    i = 0
    while i < n:
        if rally[i] != 1:
            i += 1
            continue
        if i > 0 and rally[i - 1] == 1:
            i += 1
            continue
        start = i
        j = i
        while j < n and rally[j] == 1:
            j += 1
        end = j - 1
        L = end - start + 1

        for k in range(max(0, start - 3), start):
            target[k] = 1

        if L < split:
            for k in range(start, min(n, start + 2)):
                target[k] = 1
        else:
            last_ok = end - 3
            if last_ok >= start:
                for k in range(start, last_ok + 1):
                    target[k] = 1
        i = j

    return rally, target


def rebuild_target_for_train(df, lead_days, entry_days,
                             return_window=None, rally_threshold=None,
                             min_rally_tail_days=None):
    """
    Re-compute 'target' and 'rally' columns for every ticker in df.
    Wenn OPT_OPTIMIZE_Y_TARGETS False: feste Band-Regel (_create_target_one_ticker_fixed_bands),
    lead_days/entry_days/return_window-Argumente werden ignoriert.
    Sonst: parametrisierte _create_target_one_ticker.
    """
    df = df.copy()
    use_opt_y = globals().get("OPT_OPTIMIZE_Y_TARGETS", True)
    for ticker, sub in df.groupby('ticker'):
        sub_r = sub.reset_index(drop=True)
        if not use_opt_y:
            r, t = _create_target_one_ticker_fixed_bands(sub_r)
        else:
            r, t = _create_target_one_ticker(
                sub_r,
                lead_days=lead_days, entry_days=entry_days,
                return_window=return_window, rally_threshold=rally_threshold,
                min_rally_tail_days=min_rally_tail_days,
            )
        df.loc[sub.index, 'rally']  = r
        df.loc[sub.index, 'target'] = t
    return df


def create_target(df):
    """Add 'rally' and 'target' columns to full DataFrame, parallelised per ticker."""
    df = df.sort_values(['ticker', 'Date']).copy()
    rally_col  = np.zeros(len(df), dtype=np.int8)
    target_col = np.zeros(len(df), dtype=np.int8)

    def process(args):
        idx, sub = args
        use_opt_y = globals().get("OPT_OPTIMIZE_Y_TARGETS", True)
        if use_opt_y:
            r, t = _create_target_one_ticker(sub.reset_index(drop=True))
        else:
            r, t = _create_target_one_ticker_fixed_bands(sub.reset_index(drop=True))
        return idx, r, t

    groups = list(df.groupby('ticker'))
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        results = list(ex.map(process, groups))

    for ticker, sub in groups:
        pass  # just to reuse the loop below

    # Re-assign in correct row order
    for ticker_name, sub in df.groupby('ticker'):
        use_opt_y = globals().get("OPT_OPTIMIZE_Y_TARGETS", True)
        if use_opt_y:
            r, t = _create_target_one_ticker(sub.reset_index(drop=True))
        else:
            r, t = _create_target_one_ticker_fixed_bands(sub.reset_index(drop=True))
        df.loc[sub.index, 'rally']  = r
        df.loc[sub.index, 'target'] = t

    pos_rate = df['target'].mean()
    print(f'Target created. Positive rate: {pos_rate:.1%}')
    return df

In [56]:
# Cell 6 — 3: Technical indicators

def _linreg_slope(values):
    """Slope of OLS line through 'values' (length 5 expected)."""
    n = len(values)
    x = np.arange(n, dtype=float) - np.mean(np.arange(n, dtype=float))
    return np.dot(x, values) / np.dot(x, x)


def _compute_indicators_one_ticker(df_t):
    """
    Compute all indicator variants for one ticker.
    Returns a DataFrame with all precomputed columns.
    """
    df = df_t.sort_values('Date').copy().reset_index(drop=True)
    close  = df['close']
    volume = df['volume']

    # ── Log return for vol_stress ───────────────────────────────────────────
    log_ret = np.log(close / close.shift(1))

    # ── Vol stress ─────────────────────────────────────────────────────────
    df['vol_stress'] = (
        log_ret.rolling(10, min_periods=5).std() /
        log_ret.rolling(60, min_periods=20).std()
    )

    # ── Drawdown 60d ───────────────────────────────────────────────────────
    rolling_max_60 = close.rolling(60, min_periods=1).max()
    df['drawdown'] = (close - rolling_max_60) / rolling_max_60

    # ── MACD ───────────────────────────────────────────────────────────────
    df['macd_diff'] = ta.trend.MACD(
        close, window_slow=26, window_fast=12, window_sign=9
    ).macd_diff()

    # ── ADX (synthetic H/L from close) ────────────────────────────────────
    df['adx'] = ta.trend.ADXIndicator(
        high=close * 1.01, low=close * 0.99, close=close, window=14
    ).adx()
    df['adx_delta_3d'] = df['adx'].diff(3)

    # ── Volume features ────────────────────────────────────────────────────
    vol_5d_mean  = volume.rolling(5,  min_periods=1).mean()
    vol_20d_mean = volume.rolling(20, min_periods=5).mean()
    vol_60d_mean = volume.rolling(60, min_periods=20).mean()
    vol_60d_std  = volume.rolling(60, min_periods=20).std()

    df['vol_ratio']     = vol_5d_mean / (vol_20d_mean + 1e-10)
    df['volume_zscore'] = (vol_5d_mean - vol_60d_mean) / (vol_60d_std + 1e-10)

    # ── Momentum 20d (needed for momentum_accel + cross-asset) ─────────────
    df['momentum_20d'] = close.pct_change(20)
    df['momentum_accel'] = df['momentum_20d'].diff(5)

    # ── Regime features ────────────────────────────────────────────────────
    sma200 = close.rolling(200, min_periods=150).mean()
    df['close_vs_sma200'] = close / sma200
    df['sma200_delta_5d'] = df['close_vs_sma200'].diff(5)

    rolling_max_252 = close.rolling(252, min_periods=60).max()
    df['drawdown_252d'] = (close - rolling_max_252) / (rolling_max_252 + 1e-10)

    # ── RSI — all windows ──────────────────────────────────────────────────
    for w in RSI_WINDOWS:
        df[f'rsi_{w}'] = ta.momentum.rsi(close, window=w)
        df[f'rsi_delta_3d_{w}'] = df[f'rsi_{w}'].diff(3)

        # Weekly RSI (resample → ffill)
        try:
            weekly = close.copy()
            weekly.index = df['Date']
            weekly_c = weekly.resample('W-FRI').last().dropna()
            if len(weekly_c) >= w + 5:
                rsi_wk = ta.momentum.rsi(weekly_c, window=w)
                rsi_wk = rsi_wk.reindex(df['Date'], method='ffill')
                df[f'rsi_weekly_{w}'] = rsi_wk.values
            else:
                df[f'rsi_weekly_{w}'] = np.nan
        except Exception:
            df[f'rsi_weekly_{w}'] = np.nan

    # ── Bollinger Bands — all windows ──────────────────────────────────────
    for w in BB_WINDOWS:
        bb = ta.volatility.BollingerBands(close, window=w, window_dev=2)
        df[f'bb_pband_{w}'] = bb.bollinger_pband()
        df[f'bb_delta_3d_{w}'] = df[f'bb_pband_{w}'].diff(3)

        # Bollinger slope (rolling OLS over 5 points)
        df[f'bb_slope_5d_{w}'] = (
            df[f'bb_pband_{w}']
            .rolling(5)
            .apply(_linreg_slope, raw=True)
        )

    # ── Interaction: BB × RSI (all combinations) ──────────────────────────
    for bw in BB_WINDOWS:
        for rw in RSI_WINDOWS:
            df[f'bb_x_rsi_{bw}_{rw}'] = (
                df[f'bb_pband_{bw}'] * (df[f'rsi_{rw}'] / 100.0)
            )

    # ── SMA cross + sma_ratio (for market breadth) ─────────────────────────
    sma20 = close.rolling(20, min_periods=15).mean()
    for sw in SMA_WINDOWS:
        sma_sw = close.rolling(sw, min_periods=int(0.6 * sw)).mean()
        df[f'sma_cross_20_{sw}'] = sma20 / (sma_sw + 1e-10)
        df[f'sma_ratio_{sw}']    = close / (sma_sw + 1e-10)

    return df


def _compute_indicators_one_ticker_for_meta(df_t, rsi_w, bb_w, sma_w):
    """Nur Indikatoren für die trainierten Fenster rsi_w/bb_w/sma_w (entspricht FEAT_COLS-Technik)."""
    df = df_t.sort_values("Date").copy().reset_index(drop=True)
    close = df["close"]
    volume = df["volume"]
    log_ret = np.log(close / close.shift(1))
    df["vol_stress"] = (
        log_ret.rolling(10, min_periods=5).std() / log_ret.rolling(60, min_periods=20).std()
    )
    rolling_max_60 = close.rolling(60, min_periods=1).max()
    df["drawdown"] = (close - rolling_max_60) / rolling_max_60
    df["macd_diff"] = ta.trend.MACD(
        close, window_slow=26, window_fast=12, window_sign=9
    ).macd_diff()
    df["adx"] = ta.trend.ADXIndicator(
        high=close * 1.01, low=close * 0.99, close=close, window=14
    ).adx()
    df["adx_delta_3d"] = df["adx"].diff(3)
    vol_5d_mean = volume.rolling(5, min_periods=1).mean()
    vol_20d_mean = volume.rolling(20, min_periods=5).mean()
    vol_60d_mean = volume.rolling(60, min_periods=20).mean()
    vol_60d_std = volume.rolling(60, min_periods=20).std()
    df["vol_ratio"] = vol_5d_mean / (vol_20d_mean + 1e-10)
    df["volume_zscore"] = (vol_5d_mean - vol_60d_mean) / (vol_60d_std + 1e-10)
    df["momentum_20d"] = close.pct_change(20)
    df["momentum_accel"] = df["momentum_20d"].diff(5)
    sma200 = close.rolling(200, min_periods=150).mean()
    df["close_vs_sma200"] = close / sma200
    df["sma200_delta_5d"] = df["close_vs_sma200"].diff(5)
    rolling_max_252 = close.rolling(252, min_periods=60).max()
    df["drawdown_252d"] = (close - rolling_max_252) / (rolling_max_252 + 1e-10)
    rw = int(rsi_w)
    df[f"rsi_{rw}"] = ta.momentum.rsi(close, window=rw)
    df[f"rsi_delta_3d_{rw}"] = df[f"rsi_{rw}"].diff(3)
    try:
        weekly = close.copy()
        weekly.index = df["Date"]
        weekly_c = weekly.resample("W-FRI").last().dropna()
        if len(weekly_c) >= rw + 5:
            rsi_wk = ta.momentum.rsi(weekly_c, window=rw)
            rsi_wk = rsi_wk.reindex(df["Date"], method="ffill")
            df[f"rsi_weekly_{rw}"] = rsi_wk.values
        else:
            df[f"rsi_weekly_{rw}"] = np.nan
    except Exception:
        df[f"rsi_weekly_{rw}"] = np.nan
    bw = int(bb_w)
    bb = ta.volatility.BollingerBands(close, window=bw, window_dev=2)
    df[f"bb_pband_{bw}"] = bb.bollinger_pband()
    df[f"bb_delta_3d_{bw}"] = df[f"bb_pband_{bw}"].diff(3)
    df[f"bb_slope_5d_{bw}"] = df[f"bb_pband_{bw}"].rolling(5).apply(_linreg_slope, raw=True)
    df[f"bb_x_rsi_{bw}_{rw}"] = df[f"bb_pband_{bw}"] * (df[f"rsi_{rw}"] / 100.0)
    sma20 = close.rolling(20, min_periods=15).mean()
    sw = int(sma_w)
    sma_sw = close.rolling(sw, min_periods=int(0.6 * sw)).mean()
    df[f"sma_cross_20_{sw}"] = sma20 / (sma_sw + 1e-10)
    df[f"sma_ratio_{sw}"] = close / (sma_sw + 1e-10)
    return df



def add_technical_indicators(df, meta_only=False):
    """Compute all indicators for all tickers in parallel.
    meta_only=True: nur rsi_w/bb_w/sma_w (wie FEAT_COLS) — für OOS neue Ticker ohne volle RSI/BB/SMA-Raster.
    """
    groups = list(df.groupby("ticker"))
    if meta_only:
        _rw = globals().get("rsi_w")
        _bw = globals().get("bb_w")
        _sw = globals().get("sma_w")
        if _rw is None or _bw is None or _sw is None:
            raise ValueError("meta_only=True: rsi_w, bb_w, sma_w müssen gesetzt sein (Cell 13+).")

        def _fn(g):
            return _compute_indicators_one_ticker_for_meta(g[1], _rw, _bw, _sw)

        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            results = list(ex.map(_fn, groups))
        out = pd.concat(results, ignore_index=True)
        print(
            f"Indicators computed (meta-only windows RSI={_rw}, BB={_bw}, SMA={_sw}). Shape: {out.shape}"
        )
        return out
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        results = list(ex.map(lambda g: _compute_indicators_one_ticker(g[1]), groups))
    out = pd.concat(results, ignore_index=True)
    print(f"Indicators computed. Shape: {out.shape}")
    return out

In [57]:
# Cell 7 — 4: News sentiment (BigQuery gdeltv2.events oder Legacy GDELT Doc API)

def _sector_query(sector):
    kws = SECTOR_KEYWORDS.get(sector)
    if kws:
        return " OR ".join(kws[:5])
    return sector.replace("_", " ")


def _current_channels():
    return ["macro"] + list(TICKERS_BY_SECTOR.keys())


# Mindestabstand zwischen GDELT-HTTP-Calls (öffentliche API ist streng rate-limited).
_GDELT_THROTTLE_LAST = [0.0]


def _gdelt_throttle():
    import time
    v = globals().get("GDELT_REQUEST_DELAY_SEC")
    if v is None:
        v = globals().get("NEWS_GDELT_SECONDS_PER_CALL", 6.0)
    gap = max(float(v), 5.0)  # unterhalb ~5s laut GDELT praktisch immer 429
    now = time.monotonic()
    if _GDELT_THROTTLE_LAST[0] > 0.0:
        w = gap - (now - _GDELT_THROTTLE_LAST[0])
        if w > 0:
            time.sleep(w)
    _GDELT_THROTTLE_LAST[0] = time.monotonic()


def _fetch_doc_json(query, date_from, date_to, mode):
    import json
    import time
    import requests
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {
        "query": query,
        "mode": mode,
        "format": "json",
        "maxrecords": 250,
        "startdatetime": date_from.strftime("%Y%m%d%H%M%S"),
        "enddatetime": date_to.strftime("%Y%m%d%H%M%S"),
    }
    headers = {
        "User-Agent": "stock_rally/1.0 (research; GDELT doc API)",
        "Accept": "application/json",
    }
    max_attempts = 8
    for attempt in range(max_attempts):
        try:
            _gdelt_throttle()
            r = requests.get(url, params=params, headers=headers, timeout=90)
            if r.status_code == 429:
                ra = r.headers.get("Retry-After")
                try:
                    wait = float(ra) if ra is not None else None
                except (TypeError, ValueError):
                    wait = None
                if wait is None or wait <= 0:
                    wait = min(180.0, 30.0 * (2**attempt))
                else:
                    wait = min(300.0, wait)
                print(
                    f"[GDELT] HTTP 429 (Rate limit) mode={mode} — warte {wait:.0f}s "
                    f"(Versuch {attempt + 1}/{max_attempts}) …",
                    flush=True,
                )
                time.sleep(wait)
                continue
            if r.status_code >= 500:
                wait = min(90, 15 * (attempt + 1))
                print(
                    f"[GDELT] HTTP {r.status_code} mode={mode} — warte {wait}s "
                    f"(Versuch {attempt + 1}/{max_attempts}) …",
                    flush=True,
                )
                time.sleep(wait)
                continue
            if r.status_code != 200:
                print(
                    f"[GDELT] HTTP {r.status_code} mode={mode} — kein Retry, leeres Ergebnis.",
                    flush=True,
                )
                return {}
            text = (r.text or "").strip()
            if not text:
                wait = min(60, 10 * (attempt + 1))
                print(
                    f"[GDELT] Leerer HTTP-200-Body mode={mode} — warte {wait}s "
                    f"(Versuch {attempt + 1}/{max_attempts}) …",
                    flush=True,
                )
                time.sleep(wait)
                continue
            try:
                return json.loads(text)
            except json.JSONDecodeError as je:
                prev = text[:240].replace(chr(10), " ")
                print(
                    f"[GDELT] Ungültiges JSON ({mode}): {je!r} — Anfang: {prev!r} — Retry …",
                    flush=True,
                )
                time.sleep(min(90, 12 * (attempt + 1)))
                continue
        except requests.RequestException as ex:
            print(f"[GDELT] Request-Fehler ({mode}): {ex!r} — Retry …", flush=True)
            time.sleep(10 * (attempt + 1))
    print(
        f"[GDELT] Rate limit / Fehler nach {max_attempts} Versuchen (mode={mode}) — Abbruch dieses Chunks.",
        flush=True,
    )
    return None


def _parse_timeline_points(data):
    # Parse (timestamp, float) from GDELT JSON variants
    out = []
    if data is None:
        return out
    if not isinstance(data, dict):
        return out
    tl = data.get("timeline") or data.get("timelineData") or data.get("data")
    if tl is None:
        return out
    if isinstance(tl, dict):
        tl = tl.get("data") or tl.get("series") or list(tl.values())
    if not isinstance(tl, list):
        return out
    for item in tl:
        if not isinstance(item, dict):
            continue
        ds = item.get("date") or item.get("datetime") or item.get("time")
        if ds is None:
            continue
        val = item.get("value")
        if val is None:
            for k, v in item.items():
                if k in ("date", "datetime", "time"):
                    continue
                if isinstance(v, (int, float)):
                    val = float(v)
                    break
        if val is None:
            continue
        try:
            d = pd.to_datetime(str(ds)[:8], format="%Y%m%d", errors="coerce")
        except Exception:
            continue
        if pd.isna(d):
            continue
        out.append((d.normalize(), float(val)))
    return out


def _merge_points(points):
    # Mean per calendar day
    if not points:
        return pd.Series(dtype=float)
    df = pd.DataFrame(points, columns=["d", "v"])
    return df.groupby("d")["v"].mean()


def _tv_series_from_points(tone_pts, vol_pts, start_ts, end_last):
    tone_s = _merge_points(tone_pts)
    vol_s = _merge_points(vol_pts)
    bdays = pd.bdate_range(start_ts, end_last)
    tone_s = tone_s.reindex(bdays).fillna(0.0)
    vol_s = vol_s.reindex(bdays).fillna(0.0)
    if len(tone_s) and tone_s.abs().median() > 1.5:
        tone_s = tone_s / 100.0
    return tone_s, vol_s


def _df_channel_from_tv(tone_s, vol_s, channel, impact_s=None):
    if impact_s is None:
        impact_s = pd.Series(0.0, index=tone_s.index)
    else:
        impact_s = impact_s.reindex(tone_s.index).fillna(0.0)
    return pd.DataFrame(
        {
            "Date": tone_s.index,
            "channel": channel,
            "tone": tone_s.values.astype(float),
            "vol": vol_s.values.astype(float),
            "impact": impact_s.values.astype(float),
        }
    )


def _iter_90d_chunks(start_inclusive, end_inclusive):
    """API-Zeit-Chunks (Länge GDELT_CHUNK_DAYS); end = letzter Kalendertag inkl."""
    start_ts = pd.Timestamp(start_inclusive).normalize()
    end_last = pd.Timestamp(end_inclusive).normalize()
    end_excl = end_last + pd.Timedelta(days=1)
    dt = start_ts
    _span = pd.Timedelta(days=GDELT_CHUNK_DAYS)
    while dt < end_excl:
        chunk_end = min(dt + _span, end_excl)
        yield (dt, chunk_end)
        dt = chunk_end + pd.Timedelta(days=1)


def _count_http_calls_for_span(start_inclusive, end_inclusive):
    return len(list(_iter_90d_chunks(start_inclusive, end_inclusive))) * 2


def _n_90d_chunks_in_span(start_inclusive, end_inclusive):
    return len(list(_iter_90d_chunks(start_inclusive, end_inclusive)))


def _normalize_news_cache_df(df):
    if df is None or df.empty:
        return pd.DataFrame(columns=["Date", "channel", "tone", "vol", "impact"])
    out = df.copy()
    out["Date"] = pd.to_datetime(out["Date"]).dt.normalize()
    out["channel"] = out["channel"].astype(str)
    if "impact" not in out.columns:
        out["impact"] = 0.0
    return out


def _load_news_cache(path):
    import pickle
    empty = pd.DataFrame(columns=["Date", "channel", "tone", "vol", "impact"])
    if not os.path.isfile(path):
        return empty, {}
    try:
        with open(path, "rb") as f:
            obj = pickle.load(f)
        if isinstance(obj, dict) and "df" in obj:
            return _normalize_news_cache_df(obj["df"]), dict(obj.get("meta") or {})
        if isinstance(obj, pd.DataFrame):
            return _normalize_news_cache_df(obj), {}
    except Exception as e:
        print(f"News cache read failed ({e}), starting empty.")
    return empty, {}


def _save_news_cache(df, path, meta=None):
    import pickle
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = path + ".tmp"
    payload = {"df": df, "meta": meta or {}}
    with open(tmp, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)


def _merge_news_cache_rows(old_df, new_df):
    if new_df is None or new_df.empty:
        return old_df if old_df is not None else pd.DataFrame()
    if old_df is None or old_df.empty:
        return new_df
    combined = pd.concat([old_df, new_df], ignore_index=True)
    combined["Date"] = pd.to_datetime(combined["Date"]).dt.normalize()
    combined = combined.drop_duplicates(subset=["Date", "channel"], keep="last")
    return combined.sort_values(["channel", "Date"]).reset_index(drop=True)


def _cluster_sorted_dates(sorted_dates):
    if not sorted_dates:
        return []
    ranges = []
    a = b = sorted_dates[0]
    for d in sorted_dates[1:]:
        if (d - b).days <= 3:
            b = d
        else:
            ranges.append((a, b))
            a = b = d
    ranges.append((a, b))
    return ranges


def _missing_bday_ranges_for_channel(cache_df, channel, bdays_index):
    need = set(pd.to_datetime(bdays_index).normalize())
    have = set()
    if cache_df is not None and not cache_df.empty:
        sub = cache_df[cache_df["channel"] == channel]
        if len(sub):
            have = set(pd.to_datetime(sub["Date"]).dt.normalize())
    missing = sorted(need - have)
    return _cluster_sorted_dates(missing)


def _fetch_channel_tone_vol(query, start, end, log_ctx=None, persist_chunk=None):
    # Returns (tone Series, vol Series), index = business days in [start, end] inclusive
    # persist_chunk: optional callback(tone_pts, vol_pts) — z. B. Pickle nach jedem Sub-Chunk
    start_ts = pd.Timestamp(start).normalize()
    end_last = pd.Timestamp(end).normalize()
    tone_pts, vol_pts = [], []
    chunks = list(_iter_90d_chunks(start_ts, end_last))
    n_sub = len(chunks)
    for ci, (cf, ct) in enumerate(chunks, start=1):
        g = tot = ch = gap_s = None
        if log_ctx is not None:
            log_ctx["counter"][0] += 1
            g = log_ctx["counter"][0]
            tot = log_ctx["total"]
            ch = log_ctx.get("channel", "?")
            gi, gn = log_ctx.get("gap_i", 0), log_ctx.get("gap_n", 0)
            gap_s = f" Lücke {gi}/{gn}" if gn else ""
            print(
                f"[GDELT] ({g}/{tot}) Kanal={ch}{gap_s} | Sub-Chunk {ci}/{n_sub} | "
                f"{cf.strftime('%Y-%m-%d')} … {ct.strftime('%Y-%m-%d')} | timelinetone + vol …",
                flush=True,
            )
        jt = _fetch_doc_json(query, cf, ct, "timelinetone")
        if jt is None:
            if tone_pts:
                print(
                    "[GDELT] Rate limit / Abbruch bei timelinetone — Zwischenstand wurde nach dem "
                    "letzten erfolgreichen Sub-Chunk gespeichert. Nächster Run füllt die Lücke.",
                    flush=True,
                )
            else:
                print(
                    "[GDELT] Rate limit / Abbruch bei timelinetone — noch keine Daten für diese "
                    "Lücke (Pickle bleibt für diesen Kanal/Bereich leer). Später erneut versuchen.",
                    flush=True,
                )
            break
        jv = _fetch_doc_json(query, cf, ct, "timelinevolraw")
        if jv is None:
            if tone_pts:
                print(
                    "[GDELT] Rate limit / Abbruch bei timelinevolraw — Zwischenstand nach letztem OK-Chunk gespeichert.",
                    flush=True,
                )
            else:
                print(
                    "[GDELT] Rate limit / Abbruch bei timelinevolraw — noch keine Daten gespeichert.",
                    flush=True,
                )
            break
        if not jv.get("timeline") and not jv.get("timelineData"):
            jv = _fetch_doc_json(query, cf, ct, "timelinevol")
            if jv is None:
                print(
                    "[GDELT] Abbruch bei timelinevol-Fallback (Rate limit). "
                    + (
                        "Zwischenstand gespeichert."
                        if tone_pts
                        else "Noch keine Daten gespeichert."
                    ),
                    flush=True,
                )
                break
            if log_ctx is not None:
                print(
                    f"[GDELT] ({g}/{tot}) Kanal={ch}{gap_s} | Sub-Chunk {ci}/{n_sub} | Fallback: timelinevol",
                    flush=True,
                )
        pt, pv = _parse_timeline_points(jt), _parse_timeline_points(jv)
        nt, nv = len(pt), len(pv)
        tone_pts.extend(pt)
        vol_pts.extend(pv)
        if log_ctx is not None:
            print(
                f"[GDELT] ({g}/{tot}) Kanal={ch}{gap_s} | Sub-Chunk {ci}/{n_sub} | OK "
                f"(Punkte tone={nt}, vol={nv}).",
                flush=True,
            )
        if persist_chunk is not None:
            persist_chunk(tone_pts, vol_pts)
    tone_s, vol_s = _tv_series_from_points(tone_pts, vol_pts, start_ts, end_last)
    return tone_s, vol_s


def _bq_ymd_int(ts):
    return int(pd.Timestamp(ts).strftime("%Y%m%d"))


def _bq_ts_utc_midnight(ts):
    from datetime import datetime, timezone
    t = pd.Timestamp(ts).normalize()
    return datetime(t.year, t.month, t.day, tzinfo=timezone.utc)


def _bq_resolve_project():
    """GCP-Projekt für BigQuery-Client (Billing). Reihenfolge: Config → Env → ADC → gcloud."""
    p = globals().get("BQ_PROJECT_ID")
    if p:
        return str(p).strip() or None
    import os
    p = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT") or os.environ.get("GCP_PROJECT_ID")
    if p:
        return p.strip()
    try:
        import google.auth
        _, project = google.auth.default()
        if project:
            return project
    except Exception:
        pass
    try:
        import subprocess
        r = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True,
            text=True,
            timeout=15,
        )
        if r.returncode == 0 and r.stdout.strip():
            return r.stdout.strip()
    except Exception:
        pass
    return None


def _bq_geo_sql(is_macro):
    """Länderfilter: Makro oft weltweit; Sektoren z. B. EU+US (Attention-Fokus)."""
    if is_macro:
        mc = globals().get("BQ_MACRO_GEO_COUNTRIES")
    else:
        if not globals().get("BQ_SECTOR_USE_GEO_FILTER", True):
            return ""
        mc = globals().get("BQ_SECTOR_GEO_COUNTRIES")
    if not mc:
        return ""
    if isinstance(mc, str):
        return f" AND ActionGeo_CountryCode = '{mc}'"
    return " AND (" + " OR ".join(f"ActionGeo_CountryCode = '{c}'" for c in mc) + ")"


def _gap_union_span(fetch_plan, channel_list):
    """Min/Max über alle Kanal-Lücken — ein Zeitraum für Single-Scan."""
    intervals = []
    for ch in channel_list:
        for a, b in fetch_plan.get(ch, []):
            intervals.append((pd.Timestamp(a).normalize(), pd.Timestamp(b).normalize()))
    if not intervals:
        return None, None
    return min(t[0] for t in intervals), max(t[1] for t in intervals)


def _bq_query_all_channels_gkg(a, b):
    """Eine Query, ein Scan: OR über alle Theme-Filter, IF/COUNTIF pro Kanal."""
    from google.cloud import bigquery
    if not globals().get("BQ_USE_PARTITION_FILTER", True):
        print(
            "[BigQuery] WARNUNG: BQ_USE_PARTITION_FILTER=False kann Full-Table-Scan bedeuten.",
            flush=True,
        )
    table = globals().get("GDELT_BQ_EVENTS_TABLE", "`gdelt-bq.gdeltv2.gkg_partitioned`")
    channels = ["macro"] + list(TICKERS_BY_SECTOR.keys())
    tone_part = "SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(0)] AS FLOAT64)"
    or_parts = [f"({MACRO_BQ_THEME_WHERE})"]
    for s in TICKERS_BY_SECTOR:
        or_parts.append(f"({SECTOR_BQ_THEME_WHERE[s]})")
    combined_or = "(" + " OR ".join(or_parts) + ")"
    sel_parts = ["DATE(_PARTITIONTIME) AS d"]
    for ch in channels:
        cond = MACRO_BQ_THEME_WHERE if ch == "macro" else SECTOR_BQ_THEME_WHERE[ch]
        sel_parts.append(f"AVG(IF( ({cond}), {tone_part}, NULL)) AS {ch}_tone")
        sel_parts.append(f"COUNTIF( ({cond}) ) AS {ch}_vol")
    select_sql = ",\n  ".join(sel_parts)
    params = [
        bigquery.ScalarQueryParameter("ts_start", "TIMESTAMP", _bq_ts_utc_midnight(a)),
        bigquery.ScalarQueryParameter("ts_end", "TIMESTAMP", _bq_ts_utc_midnight(b)),
    ]
    partition_where = (
        "_PARTITIONTIME >= @ts_start "
        "AND _PARTITIONTIME < TIMESTAMP_ADD(@ts_end, INTERVAL 1 DAY)"
    )
    sql = f"""
SELECT
  {select_sql}
FROM {table}
WHERE {partition_where}
  AND {combined_or}
  AND V2Themes IS NOT NULL
  AND V2Tone IS NOT NULL
GROUP BY d
ORDER BY d
"""
    proj = _bq_resolve_project()
    if not proj:
        raise OSError("BigQuery: Kein GCP-Projekt (BQ_PROJECT_ID / GOOGLE_CLOUD_PROJECT).")
    client = bigquery.Client(project=proj)
    job_config = bigquery.QueryJobConfig(query_parameters=params)
    job = client.query(sql, job_config=job_config)
    df = job.result().to_dataframe(create_bqstorage_client=False)
    nbytes = job.total_bytes_processed or 0
    return df, nbytes


def _bq_query_channel_date_range(channel, a, b):
    """GKG: V2Themes + V2Tone (kommasepariert). events-Tabellen haben keine Themes."""
    from google.cloud import bigquery
    use_gkg = globals().get("BQ_USE_GKG_TABLE", True)
    is_macro = channel == "macro"
    if is_macro:
        theme_sql = globals().get("MACRO_BQ_THEME_WHERE", "(1=1)")
    else:
        theme_sql = SECTOR_BQ_THEME_WHERE[channel]
    table = globals().get("GDELT_BQ_EVENTS_TABLE", "`gdelt-bq.gdeltv2.gkg_partitioned`")
    geo_sql = "" if use_gkg else _bq_geo_sql(is_macro)
    d0, d1 = _bq_ymd_int(a), _bq_ymd_int(b)
    params = []
    partition_sql = ""
    if globals().get("BQ_USE_PARTITION_FILTER", True):
        partition_sql = (
            " AND _PARTITIONTIME >= @ts_start "
            "AND _PARTITIONTIME < TIMESTAMP_ADD(@ts_end, INTERVAL 1 DAY)"
        )
        params.extend(
            [
                bigquery.ScalarQueryParameter("ts_start", "TIMESTAMP", _bq_ts_utc_midnight(a)),
                bigquery.ScalarQueryParameter("ts_end", "TIMESTAMP", _bq_ts_utc_midnight(b)),
            ]
        )
    if use_gkg:
        sql = f"""
SELECT
  DATE(_PARTITIONTIME) AS d,
  AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(0)] AS FLOAT64)) AS avg_daily_tone,
  COUNT(*) AS article_count,
  CAST(NULL AS FLOAT64) AS avg_goldstein
FROM {table}
WHERE 1=1
  {partition_sql}
  AND ({theme_sql})
  AND V2Themes IS NOT NULL
  AND V2Tone IS NOT NULL
  {geo_sql}
GROUP BY d
ORDER BY d
"""
    else:
        tc = globals().get("BQ_THEMES_COLUMN", "V2Themes")
        theme_sql = theme_sql.replace("V2Themes", tc)
        dname = str(globals().get("BQ_EVENT_DATE_FIELD", "SQLDATE"))
        dref = "`DATE`" if dname.upper() == "DATE" else dname
        params = [
            bigquery.ScalarQueryParameter("d0", "INT64", d0),
            bigquery.ScalarQueryParameter("d1", "INT64", d1),
        ]
        if globals().get("BQ_USE_PARTITION_FILTER", True):
            partition_sql = (
                " AND _PARTITIONTIME >= @ts_start "
                "AND _PARTITIONTIME < TIMESTAMP_ADD(@ts_end, INTERVAL 1 DAY)"
            )
            params.extend(
                [
                    bigquery.ScalarQueryParameter("ts_start", "TIMESTAMP", _bq_ts_utc_midnight(a)),
                    bigquery.ScalarQueryParameter("ts_end", "TIMESTAMP", _bq_ts_utc_midnight(b)),
                ]
            )
        sql = f"""
SELECT
  PARSE_DATE('%Y%m%d', CAST({dref} AS STRING)) AS d,
  AVG(AvgTone) AS avg_daily_tone,
  COUNT(*) AS article_count,
  AVG(GoldsteinScale) AS avg_goldstein
FROM {table}
WHERE {dref} >= @d0 AND {dref} <= @d1
  AND ({theme_sql})
  AND {tc} IS NOT NULL
  {geo_sql}
  {partition_sql}
GROUP BY d
ORDER BY d
"""
    proj = _bq_resolve_project()
    if not proj:
        raise OSError(
            "BigQuery: Kein GCP-Projekt. Setze in Cell 2 z. B. BQ_PROJECT_ID = 'dein-projekt-id' "
            "oder im Terminal: gcloud config set project dein-projekt-id  "
            "und $env:GOOGLE_CLOUD_PROJECT = 'dein-projekt-id' (PowerShell)."
        )
    client = bigquery.Client(project=proj)
    job_config = bigquery.QueryJobConfig(query_parameters=params)
    job = client.query(sql, job_config=job_config)
    df = job.result().to_dataframe(create_bqstorage_client=False)
    nbytes = job.total_bytes_processed or 0
    return df, nbytes


def _fetch_news_sentiment_bigquery(df, start, end):
    import time as _time
    # Gleicher Pickle-Cache wie GDELT-API. GKG+BQ_SINGLE_SCAN: 1 Query (IF/COUNTIF), sonst 1 Query pro (Kanal, Lücke).
    cache_path = globals().get("NEWS_CACHE_FILE", os.path.join(os.getcwd(), "data", "news_gdelt_cache.pkl"))
    channels = ["macro"] + list(TICKERS_BY_SECTOR.keys())
    start_t, end_t = pd.Timestamp(start), pd.Timestamp(end)
    bdays = pd.bdate_range(start_t.normalize(), end_t.normalize())
    cache_df, meta = _load_news_cache(cache_path)
    meta = dict(meta or {})
    n_rows_before = len(cache_df)
    cur_ch = _current_channels()
    if not cache_df.empty:
        d0, d1 = cache_df["Date"].min(), cache_df["Date"].max()
        print(f"News cache: {cache_path}  rows={len(cache_df)}  Date span: {d0} … {d1}")
    else:
        print(f"News cache: (empty) {cache_path}")
    fetch_plan = {ch: _missing_bday_ranges_for_channel(cache_df, ch, bdays) for ch in channels}
    n_ranges = sum(len(fetch_plan[ch]) for ch in channels)
    use_gkg = globals().get("BQ_USE_GKG_TABLE", True)
    single_scan = bool(globals().get("BQ_SINGLE_SCAN", True)) and use_gkg
    if n_ranges == 0:
        pass
    elif single_scan:
        print(
            f"[BigQuery] {n_ranges} Lücken-Bereiche über {len(channels)} Kanäle "
            f"→ 1 Query (Single-Scan, _PARTITIONTIME). Tabelle: {globals().get('GDELT_BQ_EVENTS_TABLE', '')}",
            flush=True,
        )
    else:
        print(
            f"[BigQuery] {n_ranges} Lücken-Bereiche über {len(channels)} Kanäle "
            f"(≈ {n_ranges} Queries; kein HTTP-429). Tabelle: {globals().get('GDELT_BQ_EVENTS_TABLE', '')}",
            flush=True,
        )
    print(
        "[BigQuery] Hinweis: „gescannt GB“ = von BigQuery gelesenes Volumen (Abrechnung), "
        "nicht Download auf deinen PC. Ergebnis sind nur kleine aggregierte Tabellen.",
        flush=True,
    )
    t0 = _time.perf_counter()
    n_ch = len(channels)
    total_bytes = 0
    if n_ranges == 0:
        print("[BigQuery] Cache vollständig — keine Abfragen.", flush=True)
    elif single_scan:
        a_u, b_u = _gap_union_span(fetch_plan, channels)
        if a_u is None or b_u is None:
            print("[BigQuery] Single-Scan: kein gültiger Zeitraum — übersprungen.", flush=True)
        else:
            print(
                f"[BigQuery] Single-Scan Zeitraum: {a_u.date()} … {b_u.date()} "
                "(Vereinigung aller Kanal-Lücken).",
                flush=True,
            )
            try:
                df_wide, nbytes = _bq_query_all_channels_gkg(a_u, b_u)
            except Exception as ex:
                print(
                    f"[BigQuery] Fehler Single-Scan: {ex!r} — "
                    "Bei accessDenied: öffentliches Projekt `gdelt-bq` nutzen.",
                    flush=True,
                )
                df_wide, nbytes = None, 0
            total_bytes += nbytes
            if nbytes:
                print(
                    f"  … in BigQuery gescannt (fakturierbar): {nbytes / 1e9:.3f} GB — "
                    "ein Scan für alle Kanäle.",
                    flush=True,
                )
            if df_wide is not None and len(df_wide):
                st = pd.Timestamp(a_u).normalize()
                el = pd.Timestamp(b_u).normalize()
                for ch in channels:
                    tc, vc = f"{ch}_tone", f"{ch}_vol"
                    if tc not in df_wide.columns:
                        continue
                    tone_pts, vol_pts = [], []
                    for _, row in df_wide.iterrows():
                        dd = pd.Timestamp(row["d"]).normalize()
                        tv = row[tc]
                        vv = row[vc]
                        tone_pts.append((dd, float(tv) if pd.notna(tv) else 0.0))
                        vol_pts.append((dd, float(vv) if pd.notna(vv) else 0.0))
                    tone_s, vol_s = _tv_series_from_points(tone_pts, vol_pts, st, el)
                    imp_s = pd.Series(0.0, index=tone_s.index)
                    mini = _df_channel_from_tv(tone_s, vol_s, ch, imp_s)
                    cache_df = _merge_news_cache_rows(cache_df, mini)
                meta["channels"] = cur_ch
                meta["tickers"] = sorted(ALL_TICKERS)
                meta["last_run_end_date"] = str(end_t.date())
                meta["saved_at"] = pd.Timestamp.now().isoformat()
                meta["source"] = "bigquery"
                _save_news_cache(cache_df, cache_path, meta)
                print(
                    f"[BigQuery] Zwischenspeicher: {len(cache_df)} Zeilen (Kanal=alle, Single-Scan).",
                    flush=True,
                )
    else:
        for ch_i, ch in enumerate(channels, start=1):
            ranges = fetch_plan[ch]
            if not ranges:
                print(f"[BigQuery] Kanal {ch_i}/{n_ch} {ch!r}: übersprungen (Cache).", flush=True)
                continue
            gn = len(ranges)
            print(f"[BigQuery] Kanal {ch_i}/{n_ch} {ch!r}: {gn} Lücken → Abfragen …", flush=True)
            for gi, (a, b) in enumerate(ranges, start=1):
                print(
                    f"[BigQuery]   Lücke {gi}/{gn}: {a.date()} … {b.date()}", flush=True
                )
                try:
                    df_bq, nbytes = _bq_query_channel_date_range(ch, a, b)
                except Exception as ex:
                    print(
                        f"[BigQuery] Fehler Kanal={ch!r}: {ex!r} — "
                        "Bei accessDenied: öffentliche Tabelle `gdelt-bq.gdeltv2.events` nutzen (nicht gdelt-vq). "
                        "Sonst Partition/BQ_EVENT_DATE_FIELD prüfen oder BQ_USE_PARTITION_FILTER=False.",
                        flush=True,
                    )
                    continue
                total_bytes += nbytes
                if nbytes:
                    print(
                        f"  … in BigQuery gescannt (fakturierbar): {nbytes / 1e9:.3f} GB — "
                        "kein Datei-Download; nur aggregierte Tageszeilen nach Python.",
                        flush=True,
                    )
                st = pd.Timestamp(a).normalize()
                el = pd.Timestamp(b).normalize()
                tone_pts, vol_pts, impact_pts = [], [], []
                if df_bq is not None and len(df_bq):
                    for _, row in df_bq.iterrows():
                        dd = pd.Timestamp(row["d"]).normalize()
                        tone_pts.append((dd, float(row["avg_daily_tone"])))
                        vol_pts.append((dd, float(row["article_count"])))
                        gs = row["avg_goldstein"]
                        impact_pts.append((dd, float(gs)) if pd.notna(gs) else (dd, 0.0))
                tone_s, vol_s = _tv_series_from_points(tone_pts, vol_pts, st, el)
                imp_s = _merge_points(impact_pts).reindex(tone_s.index).fillna(0.0)
                mini = _df_channel_from_tv(tone_s, vol_s, ch, imp_s)
                cache_df = _merge_news_cache_rows(cache_df, mini)
                meta["channels"] = cur_ch
                meta["tickers"] = sorted(ALL_TICKERS)
                meta["last_run_end_date"] = str(end_t.date())
                meta["saved_at"] = pd.Timestamp.now().isoformat()
                meta["source"] = "bigquery"
                _save_news_cache(cache_df, cache_path, meta)
                print(
                    f"[BigQuery] Zwischenspeicher: {len(cache_df)} Zeilen (Kanal={ch!r}).",
                    flush=True,
                )
    meta["channels"] = cur_ch
    meta["tickers"] = sorted(ALL_TICKERS)
    meta["last_run_end_date"] = str(end_t.date())
    meta["saved_at"] = pd.Timestamp.now().isoformat()
    meta["source"] = "bigquery"
    _save_news_cache(cache_df, cache_path, meta)
    if total_bytes:
        print(f"[BigQuery] Summe gescannt (letzter Lauf): {total_bytes / 1e9:.3f} GB", flush=True)
    if len(cache_df) > n_rows_before:
        print(
            f"Cache: +{len(cache_df) - n_rows_before} Zeilen → {cache_path} ({len(cache_df)} rows)."
        )
    print(f"[BigQuery] Laufzeit: {(_time.perf_counter() - t0) / 60:.2f} min")
    out = cache_df[
        (cache_df["Date"] >= start_t.normalize()) & (cache_df["Date"] <= end_t.normalize())
    ]
    out = out[out["channel"].isin(channels)].copy()
    need_keys = {(d, ch) for d in bdays for ch in channels}
    have_keys = set(zip(pd.to_datetime(out["Date"]).dt.normalize(), out["channel"]))
    still_missing = len(need_keys - have_keys)
    if still_missing:
        print(f"Warnung: {still_missing} (Datum,Kanal)-Paare fehlen noch (z. B. Query-Fehler).")
    print(f"News timelines für Pipeline: {len(out)} rows, channels={len(channels)}")
    return out


def _fetch_news_sentiment_gdelt_api(df, start=START_DATE, end=END_DATE, max_threads=8):
    # GDELT timelines + Pickle-Cache: fehlende (Kanal, Datum)-Kombinationen nachladen
    import time as _time
    cache_path = globals().get("NEWS_CACHE_FILE", os.path.join(os.getcwd(), "data", "news_gdelt_cache.pkl"))
    sec_per_call = float(globals().get("NEWS_GDELT_SECONDS_PER_CALL", 6.0))

    channels = ["macro"] + list(TICKERS_BY_SECTOR.keys())
    start_t, end_t = pd.Timestamp(start), pd.Timestamp(end)
    bdays = pd.bdate_range(start_t.normalize(), end_t.normalize())

    cache_df, meta = _load_news_cache(cache_path)
    meta = dict(meta or {})
    n_rows_before = len(cache_df)
    cur_ch = _current_channels()
    if meta.get("channels"):
        new_ch = set(cur_ch) - set(meta["channels"])
        if new_ch:
            print(
                f"[GDELT] Neue News-Kanäle (z. B. neuer Sektor): {sorted(new_ch)} — "
                "fehlende Historie wird wie üblich nachgeladen.",
                flush=True,
            )
    if meta.get("tickers"):
        new_t = set(ALL_TICKERS) - set(meta["tickers"])
        if new_t:
            secs = sorted({TICKER_TO_SECTOR[t] for t in new_t if t in TICKER_TO_SECTOR})
            sample = sorted(new_t)[:18]
            suf = "…" if len(new_t) > 18 else ""
            print(
                f"[GDELT] Neue Wertpapiere ({len(new_t)}): {sample}{suf}", flush=True
            )
            print(
                f"        → Sektoren: {secs} (News pro Sektor; Lücken werden im Pickle ergänzt.)",
                flush=True,
            )
    if not cache_df.empty:
        d0, d1 = cache_df["Date"].min(), cache_df["Date"].max()
        print(f"News cache: {cache_path}  rows={len(cache_df)}  Date span: {d0} … {d1}")
    else:
        print(f"News cache: (empty) {cache_path}")

    fetch_plan = {}
    total_http = 0
    for ch in channels:
        ranges = _missing_bday_ranges_for_channel(cache_df, ch, bdays)
        fetch_plan[ch] = ranges
        for a, b in ranges:
            total_http += _count_http_calls_for_span(a, b)

    n_ranges = sum(len(fetch_plan[ch]) for ch in channels)
    total_chunks = 0
    for ch in channels:
        for a, b in fetch_plan[ch]:
            total_chunks += _n_90d_chunks_in_span(a, b)
    est_sec = total_http * sec_per_call
    print(
        f"GDELT fetch plan: {n_ranges} Lücken-Bereiche über {len(channels)} Kanäle, "
        f"ca. {total_http} HTTP-Calls (timelinetone + vol pro {GDELT_CHUNK_DAYS}d-Chunk)."
    )
    print(
        f"[GDELT] Fortschritt: {total_chunks} Sub-Chunks gesamt — Logs zeigen (k/{total_chunks}) pro {GDELT_CHUNK_DAYS}d-Chunk.",
        flush=True,
    )
    print(f"Grobe Zeit-Schätzung: ~{est_sec/60:.1f} min (bei ~{sec_per_call:.2f}s/Call, ohne 429/Retries).")

    new_rows = []
    t0 = _time.perf_counter()
    log_ctx = {"counter": [0], "total": total_chunks}
    n_ch = len(channels)

    def _make_persist(aa, bb, chh):
        def _p(tone_pts, vol_pts):
            nonlocal cache_df
            st = pd.Timestamp(aa).normalize()
            el = pd.Timestamp(bb).normalize()
            tone_s, vol_s = _tv_series_from_points(tone_pts, vol_pts, st, el)
            mini = _df_channel_from_tv(tone_s, vol_s, chh)
            cache_df = _merge_news_cache_rows(cache_df, mini)
            meta["channels"] = cur_ch
            meta["tickers"] = sorted(ALL_TICKERS)
            meta["last_run_end_date"] = str(end_t.date())
            meta["saved_at"] = pd.Timestamp.now().isoformat()
            _save_news_cache(cache_df, cache_path, meta)
            print(
                f"[GDELT] Zwischenspeicher: Pickle {len(cache_df)} Zeilen (Kanal={chh!r}, nach Sub-Chunk).",
                flush=True,
            )

        return _p

    if total_chunks == 0:
        print("[GDELT] Keine Sub-Chunks zu laden — gesamter Zeitraum bereits im Cache.", flush=True)
    else:
        for ch_i, ch in enumerate(channels, start=1):
            ranges = fetch_plan[ch]
            if not ranges:
                print(
                    f"[GDELT] Kanal {ch_i}/{n_ch} {ch!r}: Cache vollständig — übersprungen.",
                    flush=True,
                )
                continue
            q = MACRO_NEWS_QUERY if ch == "macro" else _sector_query(ch)
            gn = len(ranges)
            print(
                f"[GDELT] Kanal {ch_i}/{n_ch} {ch!r}: {gn} Lücken-Bereiche → Download startet.",
                flush=True,
            )
            for gi, (a, b) in enumerate(ranges, start=1):
                n_sub = _n_90d_chunks_in_span(a, b)
                print(
                    f"[GDELT]   → Lücke {gi}/{gn}: {a.strftime('%Y-%m-%d')} … {b.strftime('%Y-%m-%d')} "
                    f"({n_sub} Sub-Chunks)",
                    flush=True,
                )
                log_ctx["channel"] = ch
                log_ctx["gap_i"] = gi
                log_ctx["gap_n"] = gn
                persist = _make_persist(a, b, ch)
                tone_s, vol_s = _fetch_channel_tone_vol(
                    q, a, b, log_ctx=log_ctx, persist_chunk=persist
                )
                for d in tone_s.index:
                    new_rows.append(
                        {
                            "Date": d,
                            "channel": ch,
                            "tone": float(tone_s.get(d, 0.0)),
                            "vol": float(vol_s.get(d, 0.0)),
                        }
                    )
    meta["channels"] = cur_ch
    meta["tickers"] = sorted(ALL_TICKERS)
    meta["last_run_end_date"] = str(end_t.date())
    meta["saved_at"] = pd.Timestamp.now().isoformat()
    _save_news_cache(cache_df, cache_path, meta)
    n_added = len(cache_df) - n_rows_before
    if n_added > 0:
        print(
            f"Cache: +{n_added} Zeilen seit Start dieses Laufs → {cache_path} ({len(cache_df)} rows). "
            "(Zwischenspeicher nach jedem Sub-Chunk bei Rate-Limits.)"
        )
    elif total_chunks == 0:
        print("Keine fehlenden News-Daten — Cache vollständig für den angefragten Zeitraum.")
    else:
        print("Download beendet (Cache-Zeilen siehe oben).")

    elapsed = _time.perf_counter() - t0
    if total_chunks > 0:
        print(f"Netzwerk-Zeit (Lücken-Downloads): {elapsed/60:.2f} min")

    out = cache_df[
        (cache_df["Date"] >= start_t.normalize()) & (cache_df["Date"] <= end_t.normalize())
    ]
    out = out[out["channel"].isin(channels)].copy()
    need_keys = {(d, ch) for d in bdays for ch in channels}
    have_keys = set(zip(pd.to_datetime(out["Date"]).dt.normalize(), out["channel"]))
    still_missing = len(need_keys - have_keys)
    if still_missing:
        print(f"Warnung: {still_missing} (Datum,Kanal)-Paare fehlen noch (z. B. API-Fehler).")

    print(f"News timelines für Pipeline: {len(out)} rows, channels={len(channels)}")
    return out


def fetch_news_sentiment(df, start=START_DATE, end=END_DATE, max_threads=8):
    if not USE_NEWS_SENTIMENT:
        print("News sentiment disabled (USE_NEWS_SENTIMENT=False). Returning empty.")
        return pd.DataFrame()
    price_start, price_end = start, end
    ns = globals().get("NEWS_BQ_START_DATE")
    ne = globals().get("NEWS_BQ_END_DATE")
    if ns:
        start = ns
    else:
        start = price_start
        yb = int(globals().get("NEWS_EXTRA_HISTORY_YEARS_BEFORE", 0) or 0)
        if yb > 0:
            start = (pd.Timestamp(start) - pd.DateOffset(years=yb)).strftime("%Y-%m-%d")
    if ne:
        end = ne
    else:
        end = price_end
        ya = int(globals().get("NEWS_EXTRA_HISTORY_YEARS_AFTER", 0) or 0)
        if ya > 0:
            end = (pd.Timestamp(end) + pd.DateOffset(years=ya)).strftime("%Y-%m-%d")
    if (start, end) != (price_start, price_end):
        print(f"[News] Zeitraum {start} … {end}  (Kurs {price_start} … {price_end})", flush=True)
    if globals().get("NEWS_SOURCE", "bigquery") == "bigquery":
        return _fetch_news_sentiment_bigquery(df, start, end)
    return _fetch_news_sentiment_gdelt_api(df, start, end, max_threads)


In [58]:
# Cell 8 — 5: Assemble feature matrix

def _compute_news_block(ch_df, mom_w, vol_ma_w, tone_roll, col_prefix):
    # ch_df: Date, tone, vol (one row per day)
    if ch_df is None or ch_df.empty:
        return pd.DataFrame()
    g = ch_df.sort_values("Date").copy()
    dates = g["Date"].values
    ts = pd.Series(g["tone"].values, dtype=float)
    vs = pd.Series(g["vol"].values, dtype=float)
    if tone_roll > 0:
        ts = ts.rolling(window=tone_roll, min_periods=1).mean()
    tone_mom = ts - ts.shift(int(mom_w))
    vol_ma_roll = vs.rolling(window=int(vol_ma_w), min_periods=max(1, int(vol_ma_w) // 2)).mean()
    vol_spike = vs / (vol_ma_roll + 1e-10)
    out = pd.DataFrame({"Date": dates})
    out[f"{col_prefix}tone"] = ts.values
    out[f"{col_prefix}vol"] = vs.values
    out[f"{col_prefix}tone_mom"] = tone_mom.fillna(0.0).values
    out[f"{col_prefix}vol_spike"] = vol_spike.fillna(0.0).values
    for lag in (1, 3, 5):
        out[f"{col_prefix}tone_l{lag}"] = ts.shift(lag).fillna(0.0).values
        out[f"{col_prefix}vol_l{lag}"] = vs.shift(lag).fillna(0.0).values
    for zw in [w for w in NEWS_EXTRA_ZSCORE_WINDOWS if int(w) > 0]:
        min_p = max(3, int(zw) // 2)
        rt = ts.rolling(window=int(zw), min_periods=min_p)
        rv = vs.rolling(window=int(zw), min_periods=min_p)
        tone_z = (ts - rt.mean()) / (rt.std() + 1e-10)
        vol_z = (vs - rv.mean()) / (rv.std() + 1e-10)
        sw = f'_w{int(zw)}'
        out[f"{col_prefix}tone_z{sw}"] = tone_z.fillna(0.0).values
        out[f"{col_prefix}vol_z{sw}"] = vol_z.fillna(0.0).values
    if True in NEWS_EXTRA_TONE_ACCEL_OPTIONS:
        accel = ts.diff().diff()
        out[f"{col_prefix}tone_accel"] = accel.fillna(0.0).values
    return out


def assemble_features(df, sentiment_df=None, meta_only=False):
    # Assemble feature matrix: sector/month, btc, breadth, optional news sentiment, dropna on technical worst-case
    # meta_only=True: nur Fenster rsi_w/bb_w/sma_w + ein News-Tripel aus best_params (wie FEAT_COLS) — für OOS.
    df = df.copy()

    df["sector"] = df["ticker"].map(TICKER_TO_SECTOR)
    df["sector_id"] = df["sector"].map(SECTOR_LABELS).astype(float)
    df["month"] = df["Date"].dt.month.astype(float)

    btc = df[df["ticker"] == "BTC-USD"][["Date", "momentum_20d"]].rename(
        columns={"momentum_20d": "btc_momentum"}
    )
    df = df.merge(btc, on="Date", how="left")
    df["btc_momentum"] = df["btc_momentum"].fillna(0.0)

    if meta_only:
        _swm = globals()["sma_w"]
        _rwm = globals()["rsi_w"]
        sma_loop = [_swm]
        rsi_loop = [_rwm]
    else:
        sma_loop = list(SMA_WINDOWS)
        rsi_loop = list(RSI_WINDOWS)

    for sw in sma_loop:
        breadth = (
            df.groupby("Date")[f"sma_ratio_{sw}"]
            .apply(lambda x: (x > 1.0).mean())
            .rename(f"market_breadth_{sw}")
        )
        df = df.merge(breadth, on="Date", how="left")
        df[f"market_breadth_{sw}"] = df[f"market_breadth_{sw}"].fillna(0.5)

    for rw in rsi_loop:
        df[f"sector_avg_rsi_{rw}"] = df.groupby(["Date", "sector"])[f"rsi_{rw}"].transform("mean")

    if USE_NEWS_SENTIMENT:
        if sentiment_df is None or sentiment_df.empty:
            print(
                "Warning: USE_NEWS_SENTIMENT but no news sentiment data — news features set to 0."
            )
            if meta_only:
                for c in FEAT_COLS:
                    if str(c).startswith("news_"):
                        df[c] = 0.0
            else:
                for c in all_news_model_cols():
                    df[c] = 0.0
        else:
            s = sentiment_df.copy()
            s["Date"] = pd.to_datetime(s["Date"]).dt.normalize()
            if meta_only:
                bp = best_params
                mom_w = bp["news_mom_w"]
                vol_ma = bp["news_vol_ma"]
                tone_roll = bp["news_tone_roll"]
                tag = news_feat_tag(mom_w, vol_ma, tone_roll)
                macro = s[s["channel"] == "macro"][["Date", "tone", "vol"]]
                blk = _compute_news_block(macro, mom_w, vol_ma, tone_roll, f"news_macro_{tag}_")
                if not blk.empty:
                    df = df.merge(blk, on="Date", how="left")
                sec_parts = []
                for sector in TICKERS_BY_SECTOR.keys():
                    sec = s[s["channel"] == sector][["Date", "tone", "vol"]]
                    blk_s = _compute_news_block(sec, mom_w, vol_ma, tone_roll, f"news_sec_{tag}_")
                    if blk_s.empty:
                        continue
                    blk_s = blk_s.copy()
                    blk_s["sector"] = sector
                    sec_parts.append(blk_s)
                if sec_parts:
                    sec_wide = pd.concat(sec_parts, ignore_index=True)
                    df = df.merge(sec_wide, on=["Date", "sector"], how="left")
                use_cross = bool(
                    bp.get(
                        "news_extra_macro_sec_diff",
                        SEED_PARAMS.get("news_extra_macro_sec_diff"),
                    )
                )
                if use_cross and True in NEWS_EXTRA_MACRO_SEC_DIFF_OPTIONS:
                    mc = f"news_macro_{tag}_tone"
                    sc = f"news_sec_{tag}_tone"
                    if mc in df.columns and sc in df.columns:
                        df[f"news_cross_{tag}_macro_minus_sec_tone"] = df[mc] - df[sc]
                for c in FEAT_COLS:
                    if not str(c).startswith("news_"):
                        continue
                    if c not in df.columns:
                        df[c] = 0.0
                    else:
                        df[c] = df[c].fillna(0.0)
            else:
                for mom_w in NEWS_MOM_WINDOWS:
                    for vol_ma in NEWS_VOL_MA_WINDOWS:
                        for tone_roll in NEWS_TONE_ROLL_WINDOWS:
                            tag = news_feat_tag(mom_w, vol_ma, tone_roll)
                            macro = s[s["channel"] == "macro"][["Date", "tone", "vol"]]
                            blk = _compute_news_block(
                                macro, mom_w, vol_ma, tone_roll, f"news_macro_{tag}_"
                            )
                            if not blk.empty:
                                df = df.merge(blk, on="Date", how="left")
                            sec_parts = []
                            for sector in TICKERS_BY_SECTOR.keys():
                                sec = s[s["channel"] == sector][["Date", "tone", "vol"]]
                                blk_s = _compute_news_block(
                                    sec, mom_w, vol_ma, tone_roll, f"news_sec_{tag}_"
                                )
                                if blk_s.empty:
                                    continue
                                blk_s = blk_s.copy()
                                blk_s["sector"] = sector
                                sec_parts.append(blk_s)
                            if sec_parts:
                                sec_wide = pd.concat(sec_parts, ignore_index=True)
                                df = df.merge(sec_wide, on=["Date", "sector"], how="left")
                if True in NEWS_EXTRA_MACRO_SEC_DIFF_OPTIONS:
                    for mom_w in NEWS_MOM_WINDOWS:
                        for vol_ma in NEWS_VOL_MA_WINDOWS:
                            for tone_roll in NEWS_TONE_ROLL_WINDOWS:
                                tag = news_feat_tag(mom_w, vol_ma, tone_roll)
                                mc = f"news_macro_{tag}_tone"
                                sc = f"news_sec_{tag}_tone"
                                if mc in df.columns and sc in df.columns:
                                    df[f"news_cross_{tag}_macro_minus_sec_tone"] = df[mc] - df[sc]
                for c in all_news_model_cols():
                    if c not in df.columns:
                        df[c] = 0.0
                    else:
                        df[c] = df[c].fillna(0.0)

    if meta_only:
        _rw = globals()["rsi_w"]
        _bw = globals()["bb_w"]
        _sw = globals()["sma_w"]
        worst_case_cols = build_technical_cols(_rw, _bw, _sw)
    else:
        worst_case_cols = build_technical_cols(21, 25, 70)
    all_indicator_cols = [c for c in worst_case_cols if c in df.columns]
    df = df.dropna(subset=all_indicator_cols)

    _suffix = " (meta_only)" if meta_only else ""
    print(
        f"Feature matrix assembled{_suffix}. Shape: {df.shape}, "
        f"positive rate: {df['target'].mean():.1%}"
    )
    return df.reset_index(drop=True)



In [59]:
# Cell 9 — 6: Optuna optimisation (base XGBoost)

# Gates: OPT_MIN_PRECISION_BASE (Cell 2); Phase 5 nutzt OPT_MIN_PRECISION (= THRESHOLD-Ziel)
_OPT_MIN_PRECISION = OPT_MIN_PRECISION_BASE
# Maximum consecutive FP signals per ticker before a trial is penalised
_OPT_MAX_CONSEC_FP = 3


def _rsi_from_close_1d(close_arr, window):
    """RSI für Anti-Peak-Filter aus Schlusskursen (ta) — unabhängig von FEAT_COLS."""
    if close_arr is None or window is None:
        return None
    w = int(window)
    if w < 2 or len(close_arr) < w + 1:
        return None
    s = pd.Series(np.asarray(close_arr, dtype=np.float64))
    r = ta.momentum.rsi(s, window=w)
    return np.asarray(r.values, dtype=np.float64)


def _peak_rsi_mask_1d(close, rsi, skip_peak, N, min_dist, max_rsi):
    """True = Tag besteht Anti-Peak-/RSI-Check (gleiche Logik wie apply_signal_filters)."""
    close = np.asarray(close, dtype=np.float64)
    n = len(close)
    if skip_peak:
        ser = pd.Series(close)
        rh = ser.rolling(int(N), min_periods=min(5, int(N))).max()
        dist_hi = (rh - ser) / (rh + 1e-12)
        not_at_peak = (dist_hi.values >= float(min_dist))
    else:
        not_at_peak = np.ones(n, dtype=bool)
    if max_rsi is None or rsi is None:
        rsi_ok = np.ones(n, dtype=bool)
    else:
        rsi = np.asarray(rsi, dtype=np.float64)
        rsi_ok = np.isfinite(rsi) & (rsi <= float(max_rsi))
    return not_at_peak & rsi_ok


def _apply_filters_cv(probs_arr, dates_arr, tickers_arr, targets_arr,
                      threshold, consecutive_days, signal_cooldown_days,
                      close_arr=None, rsi_window=None,
                      signal_skip_near_peak=True,
                      peak_lookback_days=20,
                      peak_min_dist_from_high_pct=0.012,
                      signal_max_rsi=78.0):
    """
    Apply consecutive + cooldown filter per ticker on a fold's val set.
    Anti-Peak/RSI: RSI aus close via rsi_window (ta); rsi_window=None → globals()['rsi_w'].
    Returns (n_tp, n_signals, max_consec_fp).
    """
    n_tp = 0
    n_signals = 0
    max_consec_fp = 0
    df_v = pd.DataFrame({
        'ticker': tickers_arr,
        'Date':   dates_arr,
        'prob':   probs_arr,
        'target': targets_arr,
    })
    if close_arr is not None:
        df_v['close'] = close_arr

    _rw = rsi_window if rsi_window is not None else globals().get('rsi_w')

    for ticker, sub in df_v.groupby('ticker'):
        sub = sub.sort_values('Date').reset_index(drop=True)
        raw = (sub['prob'].values >= threshold).astype(np.int8)
        n   = len(raw)

        # Consecutive filter: at least consecutive_days of 3 must be positive
        consec = np.zeros(n, dtype=np.int8)
        for i in range(2, n):
            if raw[i-2] + raw[i-1] + raw[i] >= consecutive_days:
                consec[i] = 1

        # Cooldown filter
        final = np.zeros(n, dtype=np.int8)
        last_sig = -999
        for i in range(n):
            if consec[i] == 1 and (i - last_sig) >= signal_cooldown_days:
                final[i] = 1
                last_sig = i

        if close_arr is not None and 'close' in sub.columns:
            rsi_sub = _rsi_from_close_1d(sub['close'].values, _rw)
            mask_ok = _peak_rsi_mask_1d(
                sub['close'].values, rsi_sub,
                signal_skip_near_peak,
                peak_lookback_days,
                peak_min_dist_from_high_pct,
                signal_max_rsi,
            )
            for i in range(n):
                if final[i] == 1 and not mask_ok[i]:
                    final[i] = 0

        sig_targets = sub.loc[final == 1, 'target'].values
        n_tp      += int(sig_targets.sum())
        n_signals += int(sig_targets.size)

        # Max consecutive FP run for this ticker
        run = 0
        for is_tp in sig_targets:
            if is_tp == 0:
                run += 1
                if run > max_consec_fp:
                    max_consec_fp = run
            else:
                run = 0

    return n_tp, n_signals, max_consec_fp


def optimize_xgb(df_train, n_trials=None, seed_params=SEED_PARAMS):
    """
    Hyperparameter optimisation via Optuna with Walk-Forward temporal CV.

    Jointly optimises (depending on OPT_MODEL_HYPERPARAMS in Cell 2):
      always:               rally definition, target-shaping (inkl. min_rally_tail_days), consecutive/cooldown
      OPT_MODEL_HYPERPARAMS=True:  also XGBoost model hyperparameters (Option A)
      OPT_MODEL_HYPERPARAMS=False: model hyperparameters fixed to SEED_PARAMS (Option B)

    Nicht optimiert hier (wird später überschrieben — siehe Cell 12/13):
      - Schwelle ``threshold`` für CV: fest aus ``seed_params`` (Phase 5 setzt ``best_threshold``).
      - Anti-Peak/RSI: fest aus ``seed_params`` (Phase 4 Meta-Optuna optimiert diese Werte).

    Objective (compound, precision-first):
      - Hard gate: per-ticker max consecutive FP run <= _OPT_MAX_CONSEC_FP
      - Soft gate: filter-Precision TP/Signals >= OPT_MIN_PRECISION_BASE (pro Fold)
      - Reward:    n_TP wenn Gate erfüllt; sonst weicher Penalty (TP/Signals) - 1

    Returns best_params dict.
    """
    if n_trials is None:
        n_trials = int(N_OPTUNA_TRIALS)
    else:
        n_trials = int(n_trials)
    wf = globals().get('OPTUNA_WF_SPLITS', None)
    wf = int(wf) if wf is not None else N_WF_SPLITS
    if wf < 1:
        wf = N_WF_SPLITS
    all_dates = np.sort(df_train['Date'].unique())
    n_dates   = len(all_dates)
    min_train = int(n_dates * 0.40)
    fold_size = max(1, (n_dates - min_train) // wf)
    print(
        f'Optuna Phase 1: TRAIN {len(df_train):,} Zeilen, {df_train["ticker"].nunique()} Ticker, '
        f'{n_dates} Kalendertage → {n_trials} Trials × {wf} WF-Folds '
        f'(pro Trial: rebuild_target + {wf}× XGB; das dominiert oft die Laufzeit).'
    )

    date_to_idx = {d: i for i, d in enumerate(all_dates)}
    df_base = df_train.copy()
    df_base['_date_idx'] = df_base['Date'].map(date_to_idx)
    _opt_y = globals().get('OPT_OPTIMIZE_Y_TARGETS', True)

    def objective(trial):
        # ── Rally-/Label-Params (nur wenn OPT_OPTIMIZE_Y_TARGETS True) ────────
        if _opt_y:
            return_window   = trial.suggest_int(  'return_window',   3,    15)
            rally_threshold = trial.suggest_float('rally_threshold', 0.07, 0.30)
            lead_days            = trial.suggest_int('lead_days',            1, 7)
            entry_days           = trial.suggest_int('entry_days',           1, 3)
            min_rally_tail_days  = trial.suggest_int('min_rally_tail_days',  3, 15)
        else:
            return_window = FIXED_Y_WINDOW_MAX
            rally_threshold = FIXED_Y_RALLY_THRESHOLD
            lead_days = 3
            entry_days = 2
            min_rally_tail_days = 5
        # ── Post-processing params ─────────────────────────────────────────
        consecutive_days     = trial.suggest_int('consecutive_days',     1, 3)
        signal_cooldown_days = trial.suggest_int('signal_cooldown_days', 1, 10)
        # Schwelle + Anti-Peak: fest aus seed_params (Phase 5 / Meta überschreiben später)
        threshold = float(seed_params['threshold'])
        signal_skip_near_peak = seed_params.get('signal_skip_near_peak', True)
        peak_lookback_days = int(seed_params.get('peak_lookback_days', 20))
        peak_min_dist_from_high_pct = float(seed_params.get('peak_min_dist_from_high_pct', 0.012))
        _sr = seed_params.get('signal_max_rsi', 78.0)
        signal_max_rsi = float(_sr) if _sr is not None else None

        # Rebuild targets for this trial's label params
        df_trial = rebuild_target_for_train(
            df_base, lead_days, entry_days,
            return_window=return_window, rally_threshold=rally_threshold,
            min_rally_tail_days=min_rally_tail_days,
        )

        # ── Model params ───────────────────────────────────────────────────
        # ── Feature-window params: always optimised (affect all base models equally) ──
        # News: Tag-Tripel + news_extra_* (Z-Score-Fenster, Accel, macro−sec) wenn USE_NEWS_SENTIMENT
        rsi_w = trial.suggest_categorical('rsi_window', RSI_WINDOWS)
        bb_w  = trial.suggest_categorical('bb_window',  BB_WINDOWS)
        sma_w = trial.suggest_categorical('sma_window', SMA_WINDOWS)
        if USE_NEWS_SENTIMENT:
            news_mom_w = trial.suggest_categorical('news_mom_w', NEWS_MOM_WINDOWS)
            news_vol_ma = trial.suggest_categorical('news_vol_ma', NEWS_VOL_MA_WINDOWS)
            news_tone_roll = trial.suggest_categorical('news_tone_roll', NEWS_TONE_ROLL_WINDOWS)
            news_extra_zscore_w = trial.suggest_categorical('news_extra_zscore_w', NEWS_EXTRA_ZSCORE_WINDOWS)
            news_extra_tone_accel = trial.suggest_categorical('news_extra_tone_accel', NEWS_EXTRA_TONE_ACCEL_OPTIONS)
            news_extra_macro_sec_diff = trial.suggest_categorical(
                'news_extra_macro_sec_diff', NEWS_EXTRA_MACRO_SEC_DIFF_OPTIONS
            )
        else:
            news_mom_w = seed_params.get('news_mom_w', NEWS_MOM_WINDOWS[len(NEWS_MOM_WINDOWS) // 2])
            news_vol_ma = seed_params.get('news_vol_ma', NEWS_VOL_MA_WINDOWS[len(NEWS_VOL_MA_WINDOWS) // 2])
            news_tone_roll = seed_params.get('news_tone_roll', NEWS_TONE_ROLL_WINDOWS[0])
            news_extra_zscore_w = None
            news_extra_tone_accel = None
            news_extra_macro_sec_diff = None


        # ── Model params: optimised (Option A) or fixed from SEED_PARAMS (Option B) ─
        if OPT_MODEL_HYPERPARAMS:
            grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide'])
            max_leaves  = trial.suggest_int('max_leaves', 31, 1024)
            if grow_policy == 'depthwise':
                max_leaves = 0
            params = dict(
                grow_policy      = grow_policy,
                max_leaves       = max_leaves,
                max_bin          = trial.suggest_categorical('max_bin', [64, 128, 256]),
                max_depth        = trial.suggest_int('max_depth', 3, 15),
                min_child_weight = trial.suggest_int('min_child_weight', 5, 100),
                gamma            = trial.suggest_float('gamma', 0.0, 10.0),
                reg_alpha        = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                reg_lambda       = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
                learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                n_estimators     = trial.suggest_int('n_estimators', 100, 1500),
                subsample        = trial.suggest_float('subsample', 0.5, 0.9),
                colsample_bytree = trial.suggest_float('colsample_bytree', 0.2, 0.7),
            )
            focal_gamma = trial.suggest_float('focal_gamma', 0.5, 5.0)
            focal_alpha = trial.suggest_float('focal_alpha', 0.05, 0.5)
        else:
            # Fixed hyperparameters from SEED_PARAMS — all base models treated equally
            params = {k: seed_params[k] for k in (
                'grow_policy', 'max_leaves', 'max_bin', 'max_depth',
                'min_child_weight', 'gamma', 'reg_alpha', 'reg_lambda',
                'learning_rate', 'n_estimators', 'subsample', 'colsample_bytree',
            )}
            focal_gamma = seed_params['focal_gamma']
            focal_alpha = seed_params['focal_alpha']

        feat_cols = build_feature_cols(
            rsi_w, bb_w, sma_w,
            news_mom_w, news_vol_ma, news_tone_roll,
            news_extra_zscore_w, news_extra_tone_accel, news_extra_macro_sec_diff,
        )
        focal_obj = make_focal_objective(focal_gamma, focal_alpha)

        fold_scores = []
        for fold_i in range(wf):
            train_end = min_train + fold_i * fold_size
            val_end   = min_train + (fold_i + 1) * fold_size
            if val_end > n_dates:
                break

            train_mask = df_trial['_date_idx'] < train_end
            val_mask   = (df_trial['_date_idx'] >= train_end) & \
                         (df_trial['_date_idx'] < val_end)

            X_tr  = df_trial.loc[train_mask, feat_cols].values.astype(np.float32)
            y_tr  = df_trial.loc[train_mask, 'target'].values.astype(np.int8)
            X_val = df_trial.loc[val_mask,   feat_cols].values.astype(np.float32)
            y_val = df_trial.loc[val_mask,   'target'].values.astype(np.int8)

            if X_tr.shape[0] < 50 or X_val.shape[0] < 10:
                continue
            if y_val.sum() < 2:
                continue

            # Inner 90/10 split for early stopping
            rng_es  = np.random.RandomState(RANDOM_STATE + fold_i)
            perm    = rng_es.permutation(len(X_tr))
            n_fit   = int(len(perm) * 0.9)
            fit_idx, es_idx = perm[:n_fit], perm[n_fit:]

            dtrain = xgb.DMatrix(X_tr[fit_idx], label=y_tr[fit_idx])
            des    = xgb.DMatrix(X_tr[es_idx],  label=y_tr[es_idx])
            dval   = xgb.DMatrix(X_val, label=y_val)

            # n_estimators = sklearn-Name; xgb.train nutzt num_boost_round (XGBoost 2.x warnt sonst)
            n_trees = int(params['n_estimators'])
            xgb_params = {k: v for k, v in params.items() if k != 'n_estimators'}
            xgb_params.update({'tree_method': 'hist', 'seed': RANDOM_STATE,
                 'disable_default_eval_metric': 1})
            bst = xgb.train(
                xgb_params,
                dtrain,
                num_boost_round=n_trees,
                obj=focal_obj,
                evals=[(des, 'es')],
                custom_metric=lambda p, d: ('logloss',
                    float(np.mean(
                        -d.get_label() * np.log(np.clip(1/(1+np.exp(-p)), 1e-7, 1-1e-7))
                        -(1-d.get_label()) * np.log(np.clip(1/(1+np.exp(p)), 1e-7, 1-1e-7))
                    ))),
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose_eval=False,
            )

            raw_preds = bst.predict(dval)
            probs_val = 1.0 / (1.0 + np.exp(-raw_preds))

            dates_val   = df_trial.loc[val_mask, 'Date'].values
            tickers_val = df_trial.loc[val_mask, 'ticker'].values
            close_val = df_trial.loc[val_mask, 'close'].values

            n_tp, n_sig, max_cfp = _apply_filters_cv(
                probs_val, dates_val, tickers_val, y_val,
                threshold, consecutive_days, signal_cooldown_days,
                close_arr=close_val,
                rsi_window=rsi_w,
                signal_skip_near_peak=signal_skip_near_peak,
                peak_lookback_days=peak_lookback_days,
                peak_min_dist_from_high_pct=peak_min_dist_from_high_pct,
                signal_max_rsi=signal_max_rsi,
            )

            # ── Compound objective ─────────────────────────────────────────
            if max_cfp > _OPT_MAX_CONSEC_FP:
                # Ticker producing consecutive FPs — hardest penalise
                fold_scores.append(-2.0 - (max_cfp - _OPT_MAX_CONSEC_FP) * 0.1)
            elif n_sig == 0:
                # Kein Signal nach Konsekutiv-/Cooldown-Filter: Proxy aus Roh-p
                # (mittlere Trennung pos/neg) — sonst konstanter Score, wenig Trial-Signal für den Sampler.
                p = np.clip(probs_val.astype(np.float64), 1e-7, 1.0 - 1e-7)
                pos_m = y_val == 1
                neg_m = y_val == 0
                if pos_m.any() and neg_m.any():
                    fold_scores.append(float(np.mean(p[pos_m]) - np.mean(p[neg_m])))
                elif pos_m.any():
                    fold_scores.append(float(np.mean(p[pos_m]) - 0.5))
                elif neg_m.any():
                    fold_scores.append(float(0.5 - np.mean(p[neg_m])))
                else:
                    fold_scores.append(0.0)
            elif (n_tp / n_sig) >= _OPT_MIN_PRECISION:
                fold_scores.append(float(n_tp))  # Precision-Gate: mehr Treffer bevorzugen
            else:
                fold_scores.append((n_tp / n_sig) - 1.0)  # unter Precision-Floor

            trial.report(np.mean(fold_scores), fold_i)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return np.mean(fold_scores) if fold_scores else -1.0

    sampler = optuna.samplers.TPESampler(multivariate=True, constant_liar=True, seed=42)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
    study   = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)

    _seed_enq = dict(seed_params)
    if not globals().get('OPT_OPTIMIZE_Y_TARGETS', True):
        for _k in ('return_window', 'rally_threshold', 'lead_days', 'entry_days', 'min_rally_tail_days'):
            _seed_enq.pop(_k, None)
    study.enqueue_trial(_seed_enq)
    # Nur tqdm-Fortschritt (eine Zeile); Optuna-INFO würde jeden Trial doppelt loggen
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best = study.best_params
    # Ensure all model hyperparameters are present — if OPT_MODEL_HYPERPARAMS=False
    # they were never suggested by Optuna, so fill them from seed_params.
    for k, v in seed_params.items():
        if k not in best:
            best[k] = v

    if not globals().get('OPT_OPTIMIZE_Y_TARGETS', True):
        best['return_window'] = FIXED_Y_WINDOW_MAX
        best['rally_threshold'] = FIXED_Y_RALLY_THRESHOLD
        best['lead_days'] = 3
        best['entry_days'] = 2
        best['min_rally_tail_days'] = 5

    mode = 'Option A (model HPs optimised)' if OPT_MODEL_HYPERPARAMS \
           else 'Option B (model HPs fixed from SEED_PARAMS)'
    print(f'\nBest trial score={study.best_value:.4f}  '
          f'(= mean TP/fold bei Filter-Prec>={_OPT_MIN_PRECISION:.0%}, '
          f'max consec FP <= {_OPT_MAX_CONSEC_FP})  [{mode}]')
    print(f'Best params: {best}')
    return best


In [60]:
# Cell 10 — 7: Holdout plot function

def plot_holdout_results(df_final, final_tickers, filtered_signals, title='FINAL Holdout'):
    """
    For each ticker in final_tickers: plot close price, rally periods,
    target days, and model buy signals.
    """
    n = len(final_tickers)
    fig, axes = plt.subplots(n, 1, figsize=(18, 5 * n))
    if n == 1:
        axes = [axes]

    for ax, ticker in zip(axes, final_tickers):
        sub = df_final[df_final['ticker'] == ticker].sort_values('Date')
        if sub.empty:
            ax.set_title(f'{ticker} — no data')
            continue

        dates = sub['Date'].values
        close = sub['close'].values

        # Close price
        ax.plot(dates, close, color='black', linewidth=0.8, label='Close')

        # Rally shading
        rally_mask = sub['rally'].values == 1
        in_rally = False
        rally_start = None
        for i, (d, r) in enumerate(zip(dates, rally_mask)):
            if r and not in_rally:
                rally_start = d
                in_rally = True
            elif not r and in_rally:
                ax.axvspan(rally_start, dates[i - 1], alpha=0.15, color='green')
                in_rally = False
        if in_rally:
            ax.axvspan(rally_start, dates[-1], alpha=0.15, color='green')

        # Target days (orange dots)
        target_mask = sub['target'].values == 1
        ax.scatter(dates[target_mask], close[target_mask],
                   color='orange', s=20, zorder=3, label='Target day')

        # Model signals (red triangles)
        sig_dates = filtered_signals.get(ticker, np.array([], dtype='datetime64[ns]'))
        if len(sig_dates) > 0:
            sig_days = {pd.Timestamp(x).normalize().date() for x in sig_dates}
            row_dates = sub['Date'].map(lambda x: pd.Timestamp(x).normalize().date())
            sig_sub = sub[row_dates.isin(sig_days)]
            ax.scatter(sig_sub['Date'].values, sig_sub['close'].values,
                       marker='^', color='red', s=60, zorder=4, label='Buy signal')

        ax.set_title(f"{ticker} — {COMPANY_NAMES.get(ticker, ticker)}", fontsize=10)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.legend(loc='upper left', fontsize=7)
        ax.set_ylabel('Price', fontsize=8)

    fig.suptitle(title, fontsize=14, y=1.001)
    plt.tight_layout()
    plt.show()


def _rows_for_signal_calendar_day(sub, d):
    """Zeilen zu Signaldatum d (Kalendertag; robust datetime64 vs Timestamp)."""
    sig_day = pd.Timestamp(d).normalize().date()
    m = sub['Date'].map(lambda x: pd.Timestamp(x).normalize().date()) == sig_day
    return sub.loc[m]


def _offset_from_rally_start(rally):
    """Handelstage seit Segmentbeginn (0 = erster Rally-Tag). NaN außerhalb Rally."""
    rally = np.asarray(rally, dtype=np.int8)
    n = len(rally)
    off = np.full(n, np.nan)
    i = 0
    while i < n:
        if rally[i] != 1:
            i += 1
            continue
        s = i
        while i < n and rally[i] == 1:
            off[i] = i - s
            i += 1
    return off


def _next_rally_start_idx(rally):
    """Index des nächsten Rally-Starts (erster grüner Tag ab inkl. i), sonst -1."""
    rally = np.asarray(rally, dtype=np.int8)
    n = len(rally)
    nxt = np.full(n, -1, dtype=np.int32)
    next_s = -1
    for i in range(n - 1, -1, -1):
        if rally[i] == 1 and (i == 0 or rally[i - 1] == 0):
            next_s = i
        nxt[i] = next_s
    return nxt


def summarize_filtered_signals_vs_target(df, filtered_signals, tickers=None):
    """
    Gefilterte Modell-Signale vs. target (positives Label) und Lage zur Rally.
    target==1: Vorlauf + Einstiegszone nur mit genug Rally-Rest (MIN_RALLY_TAIL_DAYS); sonst target=0.
    """
    if tickers is None:
        tickers = list(filtered_signals.keys())
    rows = []
    for ticker in tickers:
        sig_dates = filtered_signals.get(ticker)
        if sig_dates is None or len(sig_dates) == 0:
            continue
        sub = df[df['ticker'] == ticker].sort_values('Date').reset_index(drop=True)
        if sub.empty:
            continue
        rally = sub['rally'].values.astype(np.int8)
        tgt = sub['target'].values.astype(np.int8)
        off = _offset_from_rally_start(rally)
        nxt = _next_rally_start_idx(rally)
        for d in sig_dates:
            hit = _rows_for_signal_calendar_day(sub, d)
            if hit.empty:
                continue
            j = int(hit.index[0])
            r0, t0 = int(rally[j]), int(tgt[j])
            row = dict(
                ticker=ticker,
                date=str(pd.Timestamp(sub['Date'].iloc[j]).date()),
                target=t0,
                rally=r0,
            )
            if r0 == 1 and not np.isnan(off[j]):
                row['days_from_rally_start'] = int(off[j])
            else:
                row['days_from_rally_start'] = None
            if r0 == 0 and t0 == 1 and nxt[j] >= 0:
                row['days_to_rally_start'] = int(nxt[j] - j)
            else:
                row['days_to_rally_start'] = None
            rows.append(row)
    out = pd.DataFrame(rows)
    if out.empty:
        print('summarize_filtered_signals_vs_target: keine Signalzeilen.')
        return out
    n_one = int((out['target'] == 1).sum())
    n_zero = int((out['target'] == 0).sum())
    print(f'\nGefilterte Signale vs. target:  target=1 → {n_one}  target=0 → {n_zero}  (n={len(out)})')
    if len(out):
        print(f'  Anteil auf Label-Positiv (target=1): {n_one / len(out):.1%}')
    g = out.groupby('ticker', as_index=False).agg(
        n=('target', 'size'),
        on_target=('target', lambda s: int((s == 1).sum())),
    )
    g['on_target_pct'] = (g['on_target'] / g['n'] * 100).round(1)
    print('\nPro Ticker (gefilterte Signale):')
    print(g.to_string(index=False))
    late = out[(out['target'] == 0) & (out['rally'] == 1)]
    if len(late) > 0 and late['days_from_rally_start'].notna().any():
        print('\nSignale mit target=0, rally=1 (spät in grüner Phase) — Tage seit Rally-Start:')
        print(late[['ticker', 'date', 'days_from_rally_start']].to_string(index=False))
    return out


def apply_signal_filters(df_ticker_prob, threshold,
                         consecutive_days=None, signal_cooldown_days=None):
    """
    Apply consecutive filter then cooldown filter for a single ticker.
    consecutive_days / signal_cooldown_days default to the global constants
    (which are updated after Optuna with the optimised values).
    Optional: skip signals at local price peaks (near N-day high) and/or very high RSI.
    Returns array of Date values where filtered signals fire.
    """
    cd  = CONSECUTIVE_DAYS     if consecutive_days     is None else consecutive_days
    scd = SIGNAL_COOLDOWN_DAYS if signal_cooldown_days is None else signal_cooldown_days

    df_s = df_ticker_prob.sort_values('Date').reset_index(drop=True)
    raw  = (df_s['prob'].values >= threshold).astype(np.int8)
    n    = len(raw)

    consec = np.zeros(n, dtype=np.int8)
    for i in range(2, n):
        if raw[i-2] + raw[i-1] + raw[i] >= cd:
            consec[i] = 1

    final = np.zeros(n, dtype=np.int8)
    last_signal = -999
    for i in range(n):
        if consec[i] == 1 and (i - last_signal) >= scd:
            final[i] = 1
            last_signal = i

    if 'close' in df_s.columns:
        rw = globals().get('rsi_w')
        rsi_series = _rsi_from_close_1d(df_s['close'].values, rw)
        mask_ok = _peak_rsi_mask_1d(
            df_s['close'].values,
            rsi_series,
            bool(globals().get('SIGNAL_SKIP_NEAR_PEAK', False)),
            int(globals().get('PEAK_LOOKBACK_DAYS', 20)),
            float(globals().get('PEAK_MIN_DIST_FROM_HIGH_PCT', 0.012)),
            globals().get('SIGNAL_MAX_RSI', None),
        )
        for i in range(n):
            if final[i] == 1 and not mask_ok[i]:
                final[i] = 0

    return df_s.loc[final == 1, 'Date'].values

In [61]:
# Cell 11 — 8: Run pipeline
# Download → Target → Indicators → Features → Split (Zeit: gleiche Ticker / Legacy: Ticker-Disjunkt)

# ── 1. Load data ────────────────────────────────────────────────────────────
if globals().get('SCORING_ONLY', False):
    load_scoring_artifacts()

# UNIVERSE_FRACTION: hier anwenden (wird bei SCORING_ONLY übersprungen, wenn Artefakt Ticker enthält)
if globals().get('SCORING_ONLY', False) and globals().get('_tickers_for_run'):
    _tickers_for_run = list(globals()['_tickers_for_run'])
    print(
        f"SCORING_ONLY: {len(_tickers_for_run)} Ticker aus Artefakt (UNIVERSE_FRACTION nicht erneut angewandt)."
    )
else:
    _tickers_for_run = list(ALL_TICKERS)
    _uf = float(globals().get("UNIVERSE_FRACTION", 1.0))
    if _uf < 1.0:
        _rng_u = np.random.RandomState(int(globals().get("UNIVERSE_SAMPLE_SEED", RANDOM_STATE)))
        _rng_u.shuffle(_tickers_for_run)
        _n_keep = max(1, int(round(len(_tickers_for_run) * _uf)))
        _tickers_for_run = _tickers_for_run[:_n_keep]
        print(
            f"UNIVERSE_FRACTION={_uf}: nutze {len(_tickers_for_run)}/{len(ALL_TICKERS)} Ticker "
            "(vor Download, Target, Indikatoren, News, Features)."
        )
df_raw = load_stock_data(tickers=_tickers_for_run)

# ── 2. Target vector ────────────────────────────────────────────────────────
df_with_target = create_target(df_raw)

# ── 3. Technical indicators ─────────────────────────────────────────────────
df_with_indicators = add_technical_indicators(df_with_target)

# ── 4. News (GDELT timelines, optional) ─────────────────────────────────────
sentiment_df = fetch_news_sentiment(df_with_indicators)

# ── 5. Assemble feature matrix ──────────────────────────────────────────────
df_features = assemble_features(df_with_indicators, sentiment_df)

# ── 6. Train/Test split: Zeit (gleiche Ticker) oder Legacy (Ticker-Disjunktheit) ─
# TRAIN       → BASE+META verschmolzen (früherer Kalender bis f_train)
# THRESHOLD   → Threshold-Tuning
# FINAL       → Final-Test (TRAIN_END_DATE)

_train_cutoff = pd.Timestamp(TRAIN_END_DATE)  # uniq: Handelstage bis hier, dann zeitliche Partition (walk-forward)
available_tickers = set(df_features["ticker"].unique())


def _df_on_trading_days(df, tickers, date_array):
    """Zeilen zu exakt diesen Kalendertagen (normalisiert), gleiche Ticker."""
    ds = {pd.Timestamp(d).normalize() for d in date_array}
    dn = pd.to_datetime(df["Date"]).dt.normalize()
    return df.loc[df["ticker"].isin(tickers) & dn.isin(ds)].copy()


_split_mode = globals().get("SPLIT_MODE", "time")

if _split_mode == "time":
    fb = float(TIME_SPLIT_FRAC_BASE)
    fm = float(globals().get("TIME_SPLIT_FRAC_META", 0.0))
    ft = float(TIME_SPLIT_FRAC_THRESHOLD)
    f_train = fb + max(0.0, fm)
    ff = 1.0 - f_train - ft
    if ff < 0.0:
        raise ValueError("TIME_SPLIT_FRAC_BASE+META+THRESHOLD darf 1.0 nicht überschreiten.")
    p = int(globals().get("TIME_PURGE_TRADING_DAYS", 0))
    dc = pd.to_datetime(df_features["Date"])
    uniq = np.sort(dc[dc <= _train_cutoff].unique())
    n = len(uniq)
    if n < 30:
        raise ValueError(f"Zu wenig Handelstage für Zeit-Split (n={n}).")
    c_train = max(1, int(round(n * f_train)))
    thr_start = c_train + p
    if thr_start >= n - 2:
        raise ValueError(
            "Zeit-Split: Purge zu groß oder TRAIN-Anteil zu groß — TIME_PURGE_TRADING_DAYS "
            "oder TIME_SPLIT_FRAC_BASE+META verkleinern."
        )
    c_thr_end = max(thr_start + 1, int(round(n * (f_train + ft))))
    c_thr_end = min(c_thr_end, n - 1)
    if c_thr_end <= thr_start:
        raise ValueError("Zeit-Split: ungültige Grenzen — Anteile anpassen.")
    train_dates = uniq[:c_train]
    thr_dates = uniq[thr_start:c_thr_end]
    final_dates = uniq[c_thr_end:]
    if not len(thr_dates) or not len(final_dates):
        raise ValueError("Zeit-Split: eine Partition ist leer — Fraktionen anpassen.")
    universe_tickers = sorted(available_tickers)
    train_base_tickers = universe_tickers
    train_meta_tickers = train_base_tickers
    threshold_tickers = universe_tickers
    final_tickers = universe_tickers
    df_train_base = _df_on_trading_days(df_features, universe_tickers, train_dates)
    df_train_meta = df_train_base
    df_threshold = _df_on_trading_days(df_features, universe_tickers, thr_dates)
    df_final = _df_on_trading_days(df_features, universe_tickers, final_dates)
    print(
        f"SPLIT_MODE=time — {len(universe_tickers)} Ticker; TRAIN = BASE+META ({f_train:.0%} Kalender), Purge vor THRESHOLD: {p} Tage"
    )
    print(f"  TRAIN:     {pd.Timestamp(train_dates[0]).date()} … {pd.Timestamp(train_dates[-1]).date()}  ({len(train_dates)} Tage)")
    print(f"  THRESHOLD: {pd.Timestamp(thr_dates[0]).date()} … {pd.Timestamp(thr_dates[-1]).date()}  ({len(thr_dates)} Tage)")
    print(f"  FINAL:     {pd.Timestamp(final_dates[0]).date()} … {pd.Timestamp(final_dates[-1]).date()}  ({len(final_dates)} Tage)")
else:
    TARGET_META = max(15, round(MAX_TRAIN_TICKERS * 0.6))
    TARGET_THRESHOLD = max(10, round(MAX_TRAIN_TICKERS * 0.4))
    TARGET_FINAL = max(10, round(MAX_TRAIN_TICKERS * 0.4))
    print(
        f"SPLIT_MODE=ticker (Legacy) — Ziel: TRAIN_BASE≤{MAX_TRAIN_TICKERS}  "
        f"META≥{TARGET_META}  THRESHOLD≥{TARGET_THRESHOLD}  FINAL≥{TARGET_FINAL}"
    )
    rng_split = np.random.RandomState(42)
    train_base_tickers = []
    train_meta_tickers = []
    threshold_tickers = []
    final_tickers = []
    assigned_so_far = set()
    for sector, tickers in TICKERS_BY_SECTOR.items():
        avail = [t for t in tickers if t in available_tickers and t not in assigned_so_far]
        rng_split.shuffle(avail)
        n = len(avail)
        if n >= 4:
            train_meta_tickers.append(avail[0])
            threshold_tickers.append(avail[1])
            final_tickers.append(avail[2])
            train_base_tickers.extend(avail[3:])
        elif n == 3:
            train_meta_tickers.append(avail[0])
            threshold_tickers.append(avail[1])
            train_base_tickers.append(avail[2])
        elif n == 2:
            train_meta_tickers.append(avail[0])
            train_base_tickers.append(avail[1])
        elif n == 1:
            train_base_tickers.extend(avail)
        assigned_so_far.update(avail)

    def _fill_partition(target_list, source_list, target_size, exclude_sets):
        pool = [t for t in source_list if not any(t in s for s in exclude_sets)]
        rng_split.shuffle(pool)
        for t in pool:
            if len(target_list) >= target_size:
                break
            if t in source_list:
                target_list.append(t)
                source_list.remove(t)

    _fill_partition(train_meta_tickers, train_base_tickers, TARGET_META, [set(threshold_tickers), set(final_tickers)])
    _fill_partition(threshold_tickers, train_base_tickers, TARGET_THRESHOLD, [set(train_meta_tickers), set(final_tickers)])
    _fill_partition(final_tickers, train_base_tickers, TARGET_FINAL, [set(train_meta_tickers), set(threshold_tickers)])
    if len(train_base_tickers) > MAX_TRAIN_TICKERS:
        rng_split.shuffle(train_base_tickers)
        overflow = train_base_tickers[MAX_TRAIN_TICKERS:]
        train_base_tickers = train_base_tickers[:MAX_TRAIN_TICKERS]
        already_assigned = set(train_meta_tickers) | set(threshold_tickers) | set(final_tickers)
        overflow_clean = [t for t in overflow if t not in already_assigned]
        train_meta_tickers.extend(overflow_clean[0::3])
        threshold_tickers.extend(overflow_clean[1::3])
        final_tickers.extend(overflow_clean[2::3])
    all_assigned = train_base_tickers + train_meta_tickers + threshold_tickers + final_tickers
    assert len(all_assigned) == len(set(all_assigned)), "Ticker-Überlappung zwischen Partitionen!"
    _seen_m = set()
    _merged_tb = []
    for t in train_base_tickers + train_meta_tickers:
        if t not in _seen_m:
            _merged_tb.append(t)
            _seen_m.add(t)
    train_base_tickers = _merged_tb
    train_meta_tickers = train_base_tickers
    print(f"TRAIN:       {len(train_base_tickers):3d} tickers (BASE+META zusammengelegt)")
    print(f"THRESHOLD:   {len(threshold_tickers):3d} tickers — {threshold_tickers}")
    print(f"FINAL:       {len(final_tickers):3d} tickers — {final_tickers}")
    df_train_base = df_features[
        (df_features["ticker"].isin(train_base_tickers)) & (df_features["Date"] <= _train_cutoff)
    ].copy()
    df_train_meta = df_train_base
    df_threshold = df_features[
        (df_features["ticker"].isin(threshold_tickers)) & (df_features["Date"] <= _train_cutoff)
    ].copy()
    df_final = df_features[
        (df_features["ticker"].isin(final_tickers)) & (df_features["Date"] <= _train_cutoff)
    ].copy()

# Rückwärtskompatibilität für bestehenden Code der df_train / df_test erwartet
# Unverzerrte Auswertung (OOS): nur FINAL / df_final. df_test ist Alias von TRAIN (Meta-CV intern).
df_train = df_train_base
df_test  = df_train_meta

print(f'\nZeilenanzahl — TRAIN: {len(df_train_base):,}  '
      f'THRESHOLD: {len(df_threshold):,}  FINAL: {len(df_final):,}')


Geladen: models\scoring_artifacts.joblib  threshold=0.7100
SCORING_ONLY: 203 Ticker aus Artefakt (UNIVERSE_FRACTION nicht erneut angewandt).


$SHA.DE: possibly delisted; no timezone found
$ORAN: possibly delisted; no timezone found
$SANT.DE: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-04-10) (Yahoo error = "No data found, symbol may be delisted")
$O2BC.DE: possibly delisted; no timezone found
$HOC.DE: possibly delisted; no timezone found
$O2D.DE: possibly delisted; no timezone found
$SDO.DE: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-04-10)
$DBAG.DE: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-04-10)
$LVMH.PA: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-04-10)

9 Failed downloads:
['SHA.DE', 'ORAN', 'O2BC.DE', 'HOC.DE', 'O2D.DE']: possibly delisted; no timezone found
['SANT.DE']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-04-10) (Yahoo error = "No data found, symbol may be delisted")
['SDO.DE', 'DBAG.DE', 'LVMH.PA']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-04-10)


  Skipping SANT.DE: only 0 rows
  Skipping DBAG.DE: only 0 rows
  Skipping O2BC.DE: only 0 rows
  Skipping LVMH.PA: only 0 rows
  Skipping HOC.DE: only 0 rows
  Skipping TGR.DE: only 1 rows
  Skipping SHA.DE: only 0 rows
  Skipping SDO.DE: only 0 rows
  Skipping ORAN: only 0 rows
  Skipping O2D.DE: only 0 rows
Loaded 193 tickers, 396,764 rows.
Target created. Positive rate: 46.1%
Indicators computed. Shape: (396764, 58)
[News] Zeitraum 2017-01-01 … 2026-04-10  (Kurs 2018-01-01 … 2026-04-10)
News cache: c:\Python projects\stock_rally\data\news_gdelt_cache.pkl  rows=33866  Date span: 2017-01-02 00:00:00 … 2026-04-09 00:00:00
[BigQuery] 14 Lücken-Bereiche über 14 Kanäle → 1 Query (Single-Scan, _PARTITIONTIME). Tabelle: `gdelt-bq.gdeltv2.gkg_partitioned`
[BigQuery] Hinweis: „gescannt GB“ = von BigQuery gelesenes Volumen (Abrechnung), nicht Download auf deinen PC. Ergebnis sind nur kleine aggregierte Tabellen.
[BigQuery] Single-Scan Zeitraum: 2026-04-10 … 2026-04-10 (Vereinigung aller Kanal

In [62]:
if globals().get('SCORING_ONLY', False):
    print('[SCORING_ONLY] Training-Zelle übersprungen.')
else:
    globals()['_SCORING_ARTIFACT_SAVED_THIS_SESSION'] = False  # bis Cell 14 (Phase 5) kein aktuelles Bundle
    # Cell 13 — 8b: Optuna + Base-Modelle (Phases 1–2); SHAP → eigene Zelle 13a
    # Phase 1: Optuna base; Phase 2: Base models. Anschließend Cell 13a (SHAP), dann Cell 14 (Meta …)
    # Phase 4: Meta-Learner Optuna, Phase 5: Calibration + Threshold

    try:
        import mlflow
        mlflow.autolog(disable=True)
    except Exception:
        pass

    # ── Phase 1: Optuna HP optimisation on TRAIN ────────────────────────────────
    print('=' * 60)
    print('Phase 1: Optuna base-model HP optimisation')
    print('=' * 60)
    best_params = optimize_xgb(df_train)

    # ── Extract & apply all optimised label/filter params ────────────────────────
    if globals().get('OPT_OPTIMIZE_Y_TARGETS', True):
        RETURN_WINDOW        = best_params['return_window']
        RALLY_THRESHOLD      = best_params['rally_threshold']
        LEAD_DAYS            = best_params['lead_days']
        ENTRY_DAYS           = best_params['entry_days']
        MIN_RALLY_TAIL_DAYS  = best_params.get('min_rally_tail_days', SEED_PARAMS['min_rally_tail_days'])
    else:
        RETURN_WINDOW        = FIXED_Y_WINDOW_MAX
        RALLY_THRESHOLD      = FIXED_Y_RALLY_THRESHOLD
        LEAD_DAYS            = 3
        ENTRY_DAYS           = 2
        MIN_RALLY_TAIL_DAYS  = 5
    CONSECUTIVE_DAYS     = best_params['consecutive_days']
    SIGNAL_COOLDOWN_DAYS = best_params['signal_cooldown_days']
    # Schwelle: Phase 1 optimiert sie nicht mehr; Seed bis Phase 5 (find_precision_threshold)
    BEST_THRESHOLD       = best_params.get('threshold', SEED_PARAMS['threshold'])
    SIGNAL_SKIP_NEAR_PEAK = SEED_PARAMS['signal_skip_near_peak']
    PEAK_LOOKBACK_DAYS = int(SEED_PARAMS['peak_lookback_days'])
    PEAK_MIN_DIST_FROM_HIGH_PCT = float(SEED_PARAMS['peak_min_dist_from_high_pct'])
    _v_rsi_se = SEED_PARAMS.get('signal_max_rsi', SIGNAL_MAX_RSI)
    SIGNAL_MAX_RSI = float(_v_rsi_se) if _v_rsi_se is not None else None

    if globals().get('OPT_OPTIMIZE_Y_TARGETS', True):
        print(f'\nOptimised rally def:      return_window={RETURN_WINDOW}, '
              f'rally_threshold={RALLY_THRESHOLD:.2%}')
        print(f'Optimised target params:  lead_days={LEAD_DAYS}, entry_days={ENTRY_DAYS}, '
              f'min_rally_tail_days={MIN_RALLY_TAIL_DAYS}')
    else:
        print(f'\nFixed rally band (no y-opt): windows {FIXED_Y_WINDOW_MIN}-{FIXED_Y_WINDOW_MAX}d, '
              f'threshold={RALLY_THRESHOLD:.2%}; target rule by segment vs {FIXED_Y_SEGMENT_SPLIT}d')
        print(f'Fixed target (nur Info):  lead_days={LEAD_DAYS}, entry_days={ENTRY_DAYS} '
              f'(Rebuild nutzt _create_target_one_ticker_fixed_bands)')
    print(f'Optimised filter params:  consecutive_days={CONSECUTIVE_DAYS}, '
          f'signal_cooldown_days={SIGNAL_COOLDOWN_DAYS}')
    print(f'Seed threshold (bis Phase 5): {BEST_THRESHOLD:.3f}')
    _rsi_s = f'{SIGNAL_MAX_RSI:.1f}' if SIGNAL_MAX_RSI is not None else 'off'
    print(f'Anti-peak (SEED_PARAMS bis Meta): skip={SIGNAL_SKIP_NEAR_PEAK}, lookback={PEAK_LOOKBACK_DAYS}d, '
          f'minDist={PEAK_MIN_DIST_FROM_HIGH_PCT:.4f}, maxRSI={_rsi_s}')

    # Rebuild targets for ALL splits so base models, meta-learner, calibration
    # and threshold evaluation all use the same optimised label definition.
    _rebuild_kw = dict(
        return_window=RETURN_WINDOW,
        rally_threshold=RALLY_THRESHOLD,
        min_rally_tail_days=MIN_RALLY_TAIL_DAYS,
    )
    df_train     = rebuild_target_for_train(df_train,     LEAD_DAYS, ENTRY_DAYS, **_rebuild_kw)
    df_test      = rebuild_target_for_train(df_test,      LEAD_DAYS, ENTRY_DAYS, **_rebuild_kw)
    df_threshold = rebuild_target_for_train(df_threshold, LEAD_DAYS, ENTRY_DAYS, **_rebuild_kw)
    df_final     = rebuild_target_for_train(df_final,     LEAD_DAYS, ENTRY_DAYS, **_rebuild_kw)
    for _name, _df in [('df_train', df_train), ('df_test', df_test),
                       ('df_threshold', df_threshold), ('df_final', df_final)]:
        print(f'{_name:15s} target rebuilt  (positive rate: {_df["target"].mean():.1%})')

    rsi_w = best_params['rsi_window']
    bb_w  = best_params['bb_window']
    sma_w = best_params['sma_window']
    if USE_NEWS_SENTIMENT:
        FEAT_COLS = build_feature_cols(
            rsi_w, bb_w, sma_w,
            best_params['news_mom_w'], best_params['news_vol_ma'], best_params['news_tone_roll'],
            best_params.get('news_extra_zscore_w', SEED_PARAMS.get('news_extra_zscore_w')),
            best_params.get('news_extra_tone_accel', SEED_PARAMS.get('news_extra_tone_accel')),
            best_params.get('news_extra_macro_sec_diff', SEED_PARAMS.get('news_extra_macro_sec_diff')),
        )
    else:
        FEAT_COLS = build_feature_cols(rsi_w, bb_w, sma_w)
    print(f'\nUsing features: RSI={rsi_w}, BB={bb_w}, SMA={sma_w}  ({len(FEAT_COLS)} features)')

    focal_gamma = best_params['focal_gamma']
    focal_alpha = best_params['focal_alpha']
    focal_obj   = make_focal_objective(focal_gamma, focal_alpha)
    focal_obj_lgb = make_focal_objective_lgb(focal_gamma, focal_alpha)

    xgb_base_params = {
        k: v for k, v in best_params.items()
        if k not in (
            'rsi_window', 'bb_window', 'sma_window',
            'news_mom_w', 'news_vol_ma', 'news_tone_roll',
            'news_extra_zscore_w', 'news_extra_tone_accel', 'news_extra_macro_sec_diff',
            'focal_gamma', 'focal_alpha',
            'return_window', 'rally_threshold', 'lead_days', 'entry_days', 'min_rally_tail_days',
            'consecutive_days', 'signal_cooldown_days', 'threshold',
            'signal_skip_near_peak', 'peak_lookback_days',
            'peak_min_dist_from_high_pct', 'signal_max_rsi',
        )
    }
    xgb_base_params['tree_method'] = 'hist'

    lgb_params = dict(
        max_depth        = best_params['max_depth'],
        num_leaves       = min(best_params.get('max_leaves', 127), 255),
        learning_rate    = best_params['learning_rate'],
        subsample        = best_params['subsample'],
        colsample_bytree = best_params['colsample_bytree'],
        min_child_weight = best_params['min_child_weight'],
        reg_alpha        = best_params['reg_alpha'],
        reg_lambda       = best_params['reg_lambda'],
        n_estimators     = best_params['n_estimators'],
        verbose          = -1,
    )

    X_train_all = df_train[FEAT_COLS].values.astype(np.float32)
    y_train_all = df_train['target'].values.astype(np.int8)

    # ── Phase 2: Train 6 base models ────────────────────────────────────────────
    print('\n' + '=' * 60)
    print('Phase 2: Training 10 base models')
    print('=' * 60)
    base_models = []  # list of (name, model_or_booster, model_type)

    def _bootstrap_split(seed_i, X, y):
        """Bootstrap sample + Out-of-Bag early-stopping set.
        fit  = bootstrap sample (with replacement, same size as X)
        es   = OOB indices — never seen during training, truly independent.
        Fallback to last 10% of boot_idx if OOB set is too small (rare).
        """
        rng      = np.random.RandomState(seed_i)
        boot_idx = rng.choice(len(X), size=len(X), replace=True)
        oob_idx  = np.setdiff1d(np.arange(len(X)), boot_idx)
        if len(oob_idx) < 10:
            oob_idx = boot_idx[int(len(boot_idx) * 0.9):]
        return X[boot_idx], y[boot_idx], X[oob_idx], y[oob_idx]

    # Models 1–4: XGBoost (4 bootstrap variants)
    for m_idx, seed_i in enumerate([RANDOM_STATE, RANDOM_STATE + 1,
                                     RANDOM_STATE + 2, RANDOM_STATE + 3]):
        name = f'XGB-{m_idx+1}'
        print(f'  Training {name}...')
        X_fit, y_fit, X_es, y_es = _bootstrap_split(seed_i, X_train_all, y_train_all)
        dtrain_m = xgb.DMatrix(X_fit, label=y_fit)
        des_m    = xgb.DMatrix(X_es,  label=y_es)
        p = {**xgb_base_params, 'seed': seed_i, 'disable_default_eval_metric': 1}
        bst = xgb.train(
            p, dtrain_m,
            num_boost_round=xgb_base_params['n_estimators'],
            obj=focal_obj,
            evals=[(des_m, 'es')],
            custom_metric=lambda pr, d: ('logloss',
                float(np.mean(-d.get_label() * np.log(np.clip(1/(1+np.exp(-pr)), 1e-7, 1-1e-7))
                          - (1-d.get_label()) * np.log(np.clip(1/(1+np.exp(pr)), 1e-7, 1-1e-7))))),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )
        base_models.append((name, bst, 'xgb'))
        print(f'  {name} done — best iteration: {bst.best_iteration}')

    # Models 5–7: LightGBM (3 bootstrap variants)
    for m_idx, seed_i in enumerate([RANDOM_STATE + 10, RANDOM_STATE + 11,
                                     RANDOM_STATE + 12]):
        name = f'LGB-{m_idx+1}'
        print(f'  Training {name}...')
        X_fit, y_fit, X_es, y_es = _bootstrap_split(seed_i, X_train_all, y_train_all)
        dtrain_lgb = lgb.Dataset(X_fit, label=y_fit)
        des_lgb    = lgb.Dataset(X_es,  label=y_es, reference=dtrain_lgb)
        callbacks  = [lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(-1)]
        p_lgb = {**lgb_params, 'seed': seed_i, 'verbosity': -1}
        p_lgb['objective'] = focal_obj_lgb
        p_lgb['metric'] = 'binary_logloss'
        n_est = p_lgb.pop('n_estimators', 300)
        bst_lgb = lgb.train(
            p_lgb, dtrain_lgb,
            num_boost_round=n_est,
            valid_sets=[des_lgb],
            callbacks=callbacks,
        )
        base_models.append((name, bst_lgb, 'lgb'))
        print(f'  {name} done — best iteration: {bst_lgb.best_iteration}')

    # Model 8: Random Forest
    print('  Training RF...')
    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=best_params['max_depth'],
        min_samples_leaf=20,
        max_features='sqrt',
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_train_all, y_train_all)
    base_models.append(('RF', rf, 'rf'))
    print('  RF done.')

    # Model 9: ExtraTreesClassifier — extreme randomisation, orthogonal errors to RF/XGB
    print('  Training ET...')
    et = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=best_params['max_depth'],
        min_samples_leaf=20,
        max_features='sqrt',
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE + 20,
        n_jobs=-1,
    )
    et.fit(X_train_all, y_train_all)
    base_models.append(('ET', et, 'et'))
    print('  ET done.')

    # Model 10: Logistic Regression — linear perspective, low variance, balanced classes
    print('  Training LR...')
    lr_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(
            C=0.1,                   # strong L2 regularisation — avoids overfitting on noisy labels
            class_weight='balanced',
            max_iter=1000,
            random_state=RANDOM_STATE,
            solver='lbfgs',
            n_jobs=-1,
        )),
    ])
    lr_pipe.fit(X_train_all, y_train_all)
    base_models.append(('LR', lr_pipe, 'lr'))
    print('  LR done.')

    print(f'\nBase models ready: {[m[0] for m in base_models]}')

    # ── Predict helper ───────────────────────────────────────────────────────────
    def _predict_base(model_tuple, X):
        name, model, mtype = model_tuple
        if mtype == 'xgb':
            raw = model.predict(xgb.DMatrix(X))
            return 1.0 / (1.0 + np.exp(-raw))
        elif mtype == 'lgb':
            raw = model.predict(X)
            return 1.0 / (1.0 + np.exp(-raw))
        elif mtype in ('rf', 'et'):
            return model.predict_proba(X)[:, 1]
        elif mtype == 'lr':
            return model.predict_proba(X)[:, 1]

    # ── Phase 3: SHAP feature selection + meta feature matrix ────────────────────
    print('\n' + '=' * 60)
    print('Phase 3: SHAP feature selection')
    print('=' * 60)

    X_test_feat  = df_test[FEAT_COLS].values.astype(np.float32)
    y_test       = df_test['target'].values.astype(np.int8)
    X_final_feat = df_final[FEAT_COLS].values.astype(np.float32)
    y_final      = df_final['target'].values.astype(np.int8)

    rename_map = build_rename_map(
        rsi_w, bb_w, sma_w,
        best_params.get('news_mom_w'), best_params.get('news_vol_ma'), best_params.get('news_tone_roll'),
        best_params.get('news_extra_zscore_w'), best_params.get('news_extra_tone_accel'),
        best_params.get('news_extra_macro_sec_diff'),
    )
    feat_display = [rename_map.get(c, c) for c in FEAT_COLS]

    # SHAP on XGB-1 (optimierte Optuna-Hyperparameter aus Phase 1), Stichprobe aus TEST
    _shap_hp = {k: best_params[k] for k in (
        'max_depth', 'learning_rate', 'n_estimators', 'subsample', 'colsample_bytree',
        'min_child_weight', 'reg_alpha', 'reg_lambda', 'gamma', 'max_leaves',
    ) if k in best_params}
    print('SHAP-Modell: XGB-1; verwendete optimierte XGB-Hyperparameter:', _shap_hp)

    rng_shap    = np.random.RandomState(RANDOM_STATE)
    shap_idx    = rng_shap.choice(len(X_test_feat), size=min(2000, len(X_test_feat)), replace=False)
    xgb_model_1 = base_models[0][1]  # first XGBoost booster (trainiert mit xgb_base_params / best_params)
    explainer   = shap.TreeExplainer(xgb_model_1)
    shap_vals   = explainer.shap_values(xgb.DMatrix(X_test_feat[shap_idx]))
    mean_shap   = np.abs(shap_vals).mean(axis=0)

    # ── SHAP-Analyse: globale Bedeutung aller Features (mittlere |SHAP|) ───────────
    import matplotlib.pyplot as plt

    shap_df = (
        pd.DataFrame({'feature': feat_display, 'mean_abs_shap': mean_shap})
        .sort_values('mean_abs_shap', ascending=False)
        .reset_index(drop=True)
    )
    print('\nMittlere |SHAP| (alle Features, absteigend):')
    print(shap_df.to_string(index=False))

    _shap_top = min(25, len(shap_df))
    fig, ax = plt.subplots(figsize=(8, max(6, _shap_top * 0.28)))
    top = shap_df.head(_shap_top).iloc[::-1]
    ax.barh(top['feature'], top['mean_abs_shap'], color='steelblue')
    ax.set_xlabel('Mittlere |SHAP|')
    ax.set_title('SHAP: Base-XGB-1 (Optuna-Parameter), Test-Stichprobe')
    plt.tight_layout()
    plt.show()

    shap.summary_plot(
        shap_vals, X_test_feat[shap_idx], feature_names=feat_display,
        max_display=min(20, len(FEAT_COLS)), show=True,
    )

    topk_idx   = np.argsort(mean_shap)[::-1][:META_SHAP_TOP_K]
    topk_names = [FEAT_COLS[i] for i in topk_idx]
    print(f'\nTop {META_SHAP_TOP_K} SHAP features (Meta-Stacking): {[rename_map.get(n, n) for n in topk_names]}')
    print('\nCell 13 abgeschlossen. Bitte Cell 14 ausführen → Meta-Features + Meta-Learner + Threshold.')


[SCORING_ONLY] Training-Zelle übersprungen.


In [63]:
if globals().get('SCORING_ONLY', False):
    print('[SCORING_ONLY] Training-Zelle übersprungen.')
else:
    # Cell 14 — 8c: Meta-Feature-Aufbau + Meta-Learner + Meta-SHAP + Threshold
    # Hinweis: Tracebacks stehen ggf. am Ende der Zell-Ausgabe (nach langem Optuna-Log) — nach unten scrollen.
    # Voraussetzung: Cell 13 (Phase 1–2) + Cell 13a (SHAP → topk_names/topk_idx) ausgeführt

    import time

    # ── Phase 3b: Meta-Feature-Matrix aufbauen ────────────────────────────────────
    print('=' * 60) 
    print('Phase 3b: Meta-Feature-Matrix aufbauen')
    print('=' * 60)

    def _predict_base_logged(model_tuple, X, dataset_label=''):
        """Wie _predict_base, aber mit Fortschritts-Logs."""
        name, model, mtype = model_tuple
        t0 = time.time()
        n = len(X)
        print(f'  [{name}] scoring {n:,} Zeilen ({dataset_label})...', end='', flush=True)
        if mtype == 'xgb':
            result = 1.0 / (1.0 + np.exp(-model.predict(xgb.DMatrix(X))))
        elif mtype == 'lgb':
            result = 1.0 / (1.0 + np.exp(-model.predict(X)))
        else:  # rf, et, lr
            result = model.predict_proba(X)[:, 1]
        print(f' {time.time()-t0:.1f}s', flush=True)
        return result

    def build_meta_features(X_feat, dataset_label=''):
        """Stapelt Base-Model-Predictions + Top-K-SHAP-Roh-Features zur Meta-Feature-Matrix."""
        if dataset_label:
            print(f'\n--- {dataset_label}: {len(X_feat):,} Samples ---')
        base_preds = np.column_stack([
            _predict_base_logged(m, X_feat, dataset_label) for m in base_models
        ])
        topk_feats = X_feat[:, topk_idx]
        result = np.hstack([base_preds, topk_feats]).astype(np.float32)
        if dataset_label:
            print(f'  Meta-Matrix Shape: {result.shape}')
        return result

    # Feature-Namen für das Meta-Modell (Basis-Predictions + Top-K-SHAP-Roh-Features)
    meta_feature_names = [m[0] + '_prob' for m in base_models] + \
                         [rename_map.get(n, n) for n in topk_names]
    print(f'Meta-Feature-Namen: {meta_feature_names}')

    # Rohdaten für alle Partitionen vorbereiten
    X_test_feat      = df_test[FEAT_COLS].values.astype(np.float32)
    y_test           = df_test['target'].values.astype(np.int8)
    X_final_feat     = df_final[FEAT_COLS].values.astype(np.float32)
    y_final          = df_final['target'].values.astype(np.int8)
    X_threshold_feat = df_threshold[FEAT_COLS].values.astype(np.float32)
    y_threshold      = df_threshold['target'].values.astype(np.int8)

    t_total = time.time()
    X_meta_test      = build_meta_features(X_test_feat,      'TRAIN_META')
    X_meta_final     = build_meta_features(X_final_feat,     'FINAL')
    X_meta_threshold = build_meta_features(X_threshold_feat, 'THRESHOLD')
    print(f'\nAlle Meta-Matrizen fertig in {time.time()-t_total:.0f}s')
    print(f'  TRAIN_META:  {X_meta_test.shape}')
    print(f'  THRESHOLD:   {X_meta_threshold.shape}')
    print(f'  FINAL:       {X_meta_final.shape}')

    # ── Phase 4: Meta-Learner Optuna ─────────────────────────────────────────────
    print('\n' + '=' * 60)
    print('Phase 4: Meta-Learner Optuna')
    print('=' * 60)

    _OPT_MIN_PRECISION = OPT_MIN_PRECISION_META

    N_META_FOLDS = 3
    all_dates_test = np.sort(df_test['Date'].unique())
    n_meta_dates   = len(all_dates_test)
    meta_min_train = int(n_meta_dates * 0.40)
    meta_fold_size = (n_meta_dates - meta_min_train) // N_META_FOLDS
    date_to_idx_test = {d: i for i, d in enumerate(all_dates_test)}
    df_test_idx  = df_test['Date'].map(date_to_idx_test).values

    def meta_objective(trial):
        """
        Compound objective — wie Phase 1 (Cell 9): Filter-Precision >= OPT_MIN_PRECISION_META pro Fold.
          - Hard gate: max consecutive FP run per ticker <= _OPT_MAX_CONSEC_FP
          - Soft gate: TP/Signals (gefiltert) >= Gate
          - Reward:    n_TP wenn Gate erfüllt; sonst weicher Penalty (TP/Signals) - 1
        Konsekutiv / Cooldown wie Phase 1; Anti-Peak/RSI wie Cell 9.
        WICHTIG: Nicht BEST_THRESHOLD (Phase-1, Base-Roh-p) — Meta-predict_proba hat andere Skala.
        """
        signal_skip_near_peak = trial.suggest_categorical('signal_skip_near_peak', [True, False])
        peak_lookback_days = trial.suggest_int('peak_lookback_days', 10, 40)
        peak_min_dist_from_high_pct = trial.suggest_float('peak_min_dist_from_high_pct', 0.004, 0.035)
        signal_max_rsi = trial.suggest_float('signal_max_rsi', 68.0, 88.0)
        meta_eval_threshold = trial.suggest_float('meta_eval_threshold', 0.05, 0.95)

        params = dict(
            max_depth        = trial.suggest_int('max_depth', 2, 6),
            min_child_weight = trial.suggest_int('min_child_weight', 10, 200),
            gamma            = trial.suggest_float('gamma', 0.0, 5.0),
            reg_alpha        = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            reg_lambda       = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            n_estimators     = trial.suggest_int('n_estimators', 50, 500),
            subsample        = trial.suggest_float('subsample', 0.5, 0.9),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.3, 1.0),
            tree_method           = 'hist',
            eval_metric           = 'aucpr',
            early_stopping_rounds = 20,
            seed                  = RANDOM_STATE,
        )

        _dates_test   = df_test['Date'].values
        _tickers_test = df_test['ticker'].values
        _close_test   = df_test['close'].values

        fold_scores = []
        for fold_i in range(N_META_FOLDS):
            train_end = meta_min_train + fold_i * meta_fold_size
            val_end   = meta_min_train + (fold_i + 1) * meta_fold_size
            if val_end > n_meta_dates:
                break

            tr_mask  = df_test_idx < train_end
            val_mask = (df_test_idx >= train_end) & (df_test_idx < val_end)

            X_tr  = X_meta_test[tr_mask]
            y_tr  = y_test[tr_mask]
            X_val = X_meta_test[val_mask]
            y_val = y_test[val_mask]

            if X_tr.shape[0] < 20 or y_val.sum() < 2:
                continue

            rng_m = np.random.RandomState(RANDOM_STATE)
            perm  = rng_m.permutation(len(X_tr))
            n_fit = int(len(perm) * 0.85)

            clf = xgb.XGBClassifier(**params)
            clf.fit(
                X_tr[perm[:n_fit]], y_tr[perm[:n_fit]],
                eval_set=[(X_tr[perm[n_fit:]], y_tr[perm[n_fit:]])],
                verbose=False,
            )
            probs = clf.predict_proba(X_val)[:, 1]

            n_tp, n_sig, max_cfp = _apply_filters_cv(
                probs,
                _dates_test[val_mask],
                _tickers_test[val_mask],
                y_val,
                meta_eval_threshold, CONSECUTIVE_DAYS, SIGNAL_COOLDOWN_DAYS,
                close_arr=_close_test[val_mask],
                rsi_window=rsi_w,
                signal_skip_near_peak=signal_skip_near_peak,
                peak_lookback_days=peak_lookback_days,
                peak_min_dist_from_high_pct=peak_min_dist_from_high_pct,
                signal_max_rsi=signal_max_rsi,
            )

            # ── Compound objective (same as Phase 1) ──────────────────────────
            if max_cfp > _OPT_MAX_CONSEC_FP:
                fs = -2.0 - (max_cfp - _OPT_MAX_CONSEC_FP) * 0.1
            elif n_sig == 0:
                p = np.clip(probs.astype(np.float64), 1e-7, 1.0 - 1e-7)
                pos_m = y_val == 1
                neg_m = y_val == 0
                if pos_m.any() and neg_m.any():
                    fs = float(np.mean(p[pos_m]) - np.mean(p[neg_m]))
                elif pos_m.any():
                    fs = float(np.mean(p[pos_m]) - 0.5)
                elif neg_m.any():
                    fs = float(0.5 - np.mean(p[neg_m]))
                else:
                    fs = 0.0
            elif (n_tp / n_sig) >= _OPT_MIN_PRECISION:
                fs = float(n_tp)
            else:
                fs = (n_tp / n_sig) - 1.0
            fold_scores.append(fs)
            trial.report(fs, fold_i)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return np.mean(fold_scores) if fold_scores else -1.0

    meta_sampler = optuna.samplers.TPESampler(multivariate=True, constant_liar=True, seed=42)
    meta_pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
    meta_study   = optuna.create_study(direction='maximize', sampler=meta_sampler, pruner=meta_pruner)
    # Nur tqdm-Fortschritt (eine Zeile); Optuna-INFO würde jeden Trial doppelt loggen
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    meta_study.optimize(meta_objective, n_trials=N_META_TRIALS, show_progress_bar=True)

    _META_NONMODEL = {
        'signal_skip_near_peak', 'peak_lookback_days',
        'peak_min_dist_from_high_pct', 'signal_max_rsi',
        'meta_eval_threshold',
    }
    meta_best = {k: v for k, v in meta_study.best_params.items() if k not in _META_NONMODEL}
    meta_best.update(
        tree_method='hist',
        eval_metric='aucpr',
        early_stopping_rounds=20,
        seed=RANDOM_STATE,
    )
    SIGNAL_SKIP_NEAR_PEAK = meta_study.best_params.get('signal_skip_near_peak', SIGNAL_SKIP_NEAR_PEAK)
    PEAK_LOOKBACK_DAYS = int(meta_study.best_params.get('peak_lookback_days', PEAK_LOOKBACK_DAYS))
    PEAK_MIN_DIST_FROM_HIGH_PCT = float(meta_study.best_params.get('peak_min_dist_from_high_pct', PEAK_MIN_DIST_FROM_HIGH_PCT))
    _mv = meta_study.best_params.get('signal_max_rsi', SIGNAL_MAX_RSI)
    SIGNAL_MAX_RSI = float(_mv) if _mv is not None else None
    _mrsi_s = f'{SIGNAL_MAX_RSI:.1f}' if SIGNAL_MAX_RSI is not None else 'off'
    print(f'Meta anti-peak: skip={SIGNAL_SKIP_NEAR_PEAK}, lookback={PEAK_LOOKBACK_DAYS}d, '
          f'minDist={PEAK_MIN_DIST_FROM_HIGH_PCT:.4f}, maxRSI={_mrsi_s}')

    print(f'\nMeta-Learner best score={meta_study.best_value:.4f}  '
          f'(= mean TP/fold bei Filter-Prec>={_OPT_MIN_PRECISION:.0%}, '
          f'max consec FP <= {_OPT_MAX_CONSEC_FP})')
    _met = meta_study.best_params.get('meta_eval_threshold')
    if _met is not None:
        print(f'  (CV nutzt meta_eval_threshold={_met:.3f} — nicht Phase-1 BEST_THRESHOLD; '
              'Phase 5 setzt den produktiven Schwellenwert auf THRESHOLD.)')

    # Finales Meta-Modell auf allen TRAIN_META-Daten trainieren
    rng_final_meta = np.random.RandomState(RANDOM_STATE)
    perm_final     = rng_final_meta.permutation(len(X_meta_test))
    n_fit_final    = int(len(perm_final) * 0.9)
    meta_clf = xgb.XGBClassifier(**meta_best)
    meta_clf.fit(
        X_meta_test[perm_final[:n_fit_final]],  y_test[perm_final[:n_fit_final]],
        eval_set=[(X_meta_test[perm_final[n_fit_final:]], y_test[perm_final[n_fit_final:]])],
        verbose=False,
    )
    print('Finales Meta-Modell trainiert.')

    # ── Meta-Learner SHAP ─────────────────────────────────────────────────────────
    print('\n' + '=' * 60)
    print('Meta-Learner SHAP: Welche Features sind dem Meta-Classifier wichtig?')
    print('=' * 60)

    meta_explainer  = shap.TreeExplainer(meta_clf)
    meta_shap_vals  = meta_explainer.shap_values(X_meta_test)
    meta_mean_shap  = np.abs(meta_shap_vals).mean(axis=0)

    print('\nMeta-Feature Wichtigkeit (absteigend):')
    sorted_meta = sorted(zip(meta_feature_names, meta_mean_shap), key=lambda x: -x[1])
    for fname, imp in sorted_meta:
        bar = '█' * max(1, int(imp / meta_mean_shap.max() * 25))
        print(f'  {fname:30s}  {imp:.4f}  {bar}')

    fig, ax = plt.subplots(figsize=(10, max(4, len(meta_feature_names) * 0.5)))
    names_s = [p[0] for p in sorted_meta[::-1]]
    vals_s  = [p[1] for p in sorted_meta[::-1]]
    ax.barh(names_s, vals_s, color='steelblue')
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Meta-Learner Feature Importance (SHAP)')
    plt.tight_layout()
    plt.show()

    # ── Phase 5: Threshold-Optimierung auf THRESHOLD-Set ──────────────────────
    print('\n' + '=' * 60)
    print('Phase 5: Threshold-Optimierung (saubere Partition)')
    print('=' * 60)

    print(f'Scoring THRESHOLD-Set ({len(y_threshold):,} Zeilen)...', flush=True)
    y_prob_threshold = meta_clf.predict_proba(X_meta_threshold)[:, 1]
    print(f'Scoring FINAL-Set ({len(y_final):,} Zeilen)...', flush=True)
    y_prob_final = meta_clf.predict_proba(X_meta_final)[:, 1]

    print(f'THRESHOLD-Set: Positive Rate = {y_threshold.mean():.1%}')

    def find_precision_threshold(y_true, y_prob, target_prec=None, min_signals=None):
        """Niedrigstes thr mit sklearn-Precision >= target_prec (roh: jede Zeile = Vorhersage)."""
        if target_prec is None:
            target_prec = OPT_MIN_PRECISION
        if min_signals is None:
            min_signals = int(globals().get("PHASE5_MIN_SIGNALS", 5))
        for thr in np.arange(0.01, 0.991, 0.005):
            preds = (y_prob >= thr).astype(int)
            if preds.sum() < min_signals:
                continue
            if precision_score(y_true, preds, zero_division=0) >= target_prec:
                return float(thr)
        return None

    prec_arr, rec_arr, thr_arr = precision_recall_curve(y_threshold, y_prob_threshold)
    f1_arr    = 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-10)
    f1_thresh = float(thr_arr[np.argmax(f1_arr[:-1])])
    prec_thresh = find_precision_threshold(y_threshold, y_prob_threshold)
    _min_s = int(globals().get("PHASE5_MIN_SIGNALS", 5))

    if prec_thresh is not None:
        best_threshold = prec_thresh
        _p_ok = precision_score(
            y_threshold, (y_prob_threshold >= best_threshold).astype(int), zero_division=0
        )
        print(
            f'Precision-Ziel OPT_MIN_PRECISION={OPT_MIN_PRECISION:.0%} erreicht: '
            f'threshold={best_threshold:.3f}, Roh-Precision={_p_ok:.2%} (THRESHOLD-Set) ✓'
        )
    else:
        best_prec, best_thresh_fb = 0.0, 0.50
        for thr in np.arange(0.01, 0.991, 0.005):
            preds = (y_prob_threshold >= thr).astype(int)
            if preds.sum() < _min_s:
                continue
            p = precision_score(y_threshold, preds, zero_division=0)
            if p > best_prec:
                best_prec, best_thresh_fb = p, float(thr)
        best_threshold = float(best_thresh_fb)
        print(
            f'OPT_MIN_PRECISION={OPT_MIN_PRECISION:.0%} mit ≥{_min_s} Roh-Signalen nicht erreichbar. '
            f'Fallback: threshold={best_threshold:.3f} (beste Roh-Precision={best_prec:.2%}). '
            f'OPT_MIN_PRECISION senken oder PHASE5_MIN_SIGNALS reduzieren.'
        )

    print(f'F1-optimaler Threshold:  {f1_thresh:.3f}')
    print(f'Gewählter Threshold:     {best_threshold:.3f}')

    df_threshold['prob'] = y_prob_threshold
    df_final['prob']     = y_prob_final
    df_test['prob']      = meta_clf.predict_proba(X_meta_test)[:, 1]
    print('\nPhase 5 complete.')
    save_scoring_artifacts()
    print(
        '\n[Artefakt] Automatisch gespeichert (models/scoring_artifacts.joblib). '
        'Zelle 18 nur nötig, wenn du ohne diese Zelle erneut speichern willst.',
        flush=True,
    )


[SCORING_ONLY] Training-Zelle übersprungen.


In [64]:
if globals().get('SCORING_ONLY', False):
    print('[SCORING_ONLY] Training-Zelle übersprungen.')
else:
    # Cell 13b — Markt-Regime nur für Evaluation (kein Training)
    # Benchmark-Kurs: Trend (Close vs. SMA200) und Volatilität (RV20 vs. RV60), alles nur aus Vergangenheit.
    # Optional: VIX vs. rollierendem Median (Vergangenheit). Danach: AP/Precision/Recall pro Jahr & Regime.

    REGIME_BENCHMARK = '^GSPC'   # z. B. '^GSPC', '^STOXX', '^GDAXI', 'SPY'
    REGIME_VIX = '^VIX'          # None = VIX weglassen
    REGIME_VIX_MIN_PERIODS = 60


    def _yf_close_series(ticker):
        """Einzelner Close als Series (DatetimeIndex)."""
        raw = yf.download(
            ticker, start=START_DATE, end=END_DATE,
            auto_adjust=True, threads=False, progress=False,
        )
        if raw is None or len(raw) == 0:
            return None
        if isinstance(raw.columns, pd.MultiIndex):
            if ticker in raw['Close'].columns:
                c = raw['Close'][ticker]
            else:
                c = raw['Close'].iloc[:, 0]
        else:
            c = raw['Close']
        c = c.dropna()
        if hasattr(c.index, 'tz') and c.index.tz is not None:
            c.index = c.index.tz_localize(None)
        return c


    def _regime_table_from_close(close: pd.Series, name='bench'):
        """Pro Handelstag des Benchmarks: Trend- und Vol-Label (kausal)."""
        s = close.sort_index().astype(float)
        log_ret = np.log(s / s.shift(1))
        rv20 = log_ret.rolling(20, min_periods=10).std()
        rv60 = log_ret.rolling(60, min_periods=20).std()
        sma200 = s.rolling(200, min_periods=50).mean()
        trend = np.where(s > sma200, 'bull', 'bear')
        vol_c = np.where(rv20 > rv60, 'high_vol', 'low_vol')
        return pd.DataFrame({
            'Date': pd.to_datetime(s.index).normalize(),
            f'{name}_trend': trend,
            f'{name}_vol': vol_c,
        })


    def _merge_regime_on_threshold(df_thr, regime_df, date_col='Date'):
        """Ordnet jeder Zeile den letzten bekannten Regime-Stand (merge_asof backward)."""
        left = df_thr.copy()
        left['_d'] = pd.to_datetime(left[date_col]).dt.normalize()
        right = regime_df.sort_values('Date').rename(columns={'Date': '_regime_date'})
        left = left.sort_values('_d')
        out = pd.merge_asof(
            left, right,
            left_on='_d', right_on='_regime_date',
            direction='backward',
        )
        return out.drop(columns=['_d', '_regime_date'], errors='ignore')


    def _report_regime_metrics(df_ev, prob_col='prob', pred_col='pred', true_col='target', group_cols=None):
        """AP aus Wahrscheinlichkeiten; Precision/Recall aus binärer Vorhersage (≥ best_threshold)."""
        from sklearn.metrics import average_precision_score, precision_score, recall_score
        if group_cols is None:
            group_cols = []
        rows = []
        if not group_cols:
            y = df_ev[true_col].values
            pr = df_ev[prob_col].values
            pb = df_ev[pred_col].values.astype(int)
            rows.append({
                'n': len(df_ev),
                'pos': int(y.sum()),
                'AP': average_precision_score(y, pr) if y.sum() > 0 else float('nan'),
                'Prec': precision_score(y, pb, zero_division=0),
                'Rec': recall_score(y, pb, zero_division=0),
            })
            return pd.DataFrame(rows)
        for keys, sub in df_ev.groupby(group_cols, dropna=False):
            if len(sub) < 50:
                continue
            yt = sub[true_col].values
            pr = sub[prob_col].values
            pb = sub[pred_col].values.astype(int)
            ap = average_precision_score(yt, pr) if yt.sum() > 0 else float('nan')
            row = {
                'n': len(sub),
                'pos': int(yt.sum()),
                'AP': ap,
                'Prec': precision_score(yt, pb, zero_division=0),
                'Rec': recall_score(yt, pb, zero_division=0),
            }
            keys = keys if isinstance(keys, tuple) else (keys,)
            for c, v in zip(group_cols, keys):
                row[c] = v
            rows.append(row)
        return pd.DataFrame(rows)


    # --- Ausführung (nach Phase 5: df_threshold['prob'], best_threshold gesetzt) ---
    try:
        _close_b = _yf_close_series(REGIME_BENCHMARK)
        if _close_b is None:
            print('Regime-Evaluation: Benchmark-Download leer — übersprungen.')
        else:
            _reg_b = _regime_table_from_close(_close_b, name='mkt')
            _df_r = df_threshold.copy()
            _df_r['year'] = pd.to_datetime(_df_r['Date']).dt.year
            _df_r = _merge_regime_on_threshold(_df_r, _reg_b)
            if REGIME_VIX:
                _vx = _yf_close_series(REGIME_VIX)
                if _vx is not None and len(_vx) > REGIME_VIX_MIN_PERIODS:
                    _v = _vx.sort_index().astype(float)
                    _med = _v.rolling(252, min_periods=REGIME_VIX_MIN_PERIODS).median().shift(1)
                    _vb = np.where(
                        (_v.values > _med.values) & np.isfinite(_med.values),
                        'vix_high',
                        'vix_low',
                    )
                    _vix_df = pd.DataFrame({
                        'Date': pd.to_datetime(_v.index).normalize(),
                        'vix_bucket': _vb,
                    }).sort_values('Date')
                    _df_r = _merge_regime_on_threshold(_df_r, _vix_df)
            thr = float(best_threshold)
            _df_r['pred'] = (_df_r['prob'].values >= thr).astype(np.int8)
            print('\n' + '=' * 60)
            print('Regime-Evaluation (nur Bericht, THRESHOLD-Set)')
            print(f'Benchmark: {REGIME_BENCHMARK}  |  thr={thr:.3f}  |  AP aus prob; Prec/Rec aus pred ≥ thr')
            print('=' * 60)
            _by_y = _report_regime_metrics(_df_r, 'prob', 'pred', 'target', ['year'])
            if len(_by_y):
                print('\nPro Kalenderjahr:')
                print(_by_y.sort_values('year').to_string(index=False))
            _cols = [c for c in ['mkt_trend', 'mkt_vol'] if c in _df_r.columns]
            if _cols:
                print('\nPro Markt-Regime (Benchmark), Trend × Vol (alle Kombinationen):')
                print(_report_regime_metrics(_df_r, 'prob', 'pred', 'target', _cols).to_string(index=False))
            if 'vix_bucket' in _df_r.columns:
                print('\nPro VIX-Bucket (vs. 252d-Median, kausal shift(1)):')
                print(_report_regime_metrics(_df_r, 'prob', 'pred', 'target', ['vix_bucket']).to_string(index=False))
            print('\nHinweis: pred = (prob >= best_threshold); Konsekutiv-/Cooldown-Filter sind hier nicht enthalten.')
    except NameError as _e:
        print(f'Regime-Evaluation übersprungen (fehlt: {_e}). Zuerst Cell 13 (Phase 5) ausführen.')
    except Exception as _e:
        print(f'Regime-Evaluation Fehler: {_e}')


[SCORING_ONLY] Training-Zelle übersprungen.


In [65]:
if globals().get('SCORING_ONLY', False):
    print('[SCORING_ONLY] Training-Zelle übersprungen.')
else:
    # Cell 13 — 8d: Threshold analysis

    # Variablen aus df_* rekonstruieren (robust gegen Kernel-Neustart nach Cell 13)
    y_prob_threshold = df_threshold['prob'].values
    y_threshold      = df_threshold['target'].values.astype(np.int8)
    y_prob_final     = df_final['prob'].values
    y_final          = df_final['target'].values.astype(np.int8)

    # ── PR curves ────────────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    prec_thr, rec_thr, _ = precision_recall_curve(y_threshold, y_prob_threshold)
    ap_thr = average_precision_score(y_threshold, y_prob_threshold)
    ax1.plot(rec_thr, prec_thr, color='steelblue', label=f'THRESHOLD (AP={ap_thr:.3f})')

    prec_fin, rec_fin, _ = precision_recall_curve(y_final, y_prob_final)
    ap_fin = average_precision_score(y_final, y_prob_final)
    ax1.plot(rec_fin, prec_fin, color='tomato', linestyle='--', label=f'FINAL (AP={ap_fin:.3f})')
    ax1.axhline(OPT_MIN_PRECISION, color='gray', linestyle=':', linewidth=0.8,
                label=f'{OPT_MIN_PRECISION:.0%} precision target')
    ax1.set_xlabel('Recall'); ax1.set_ylabel('Precision')
    ax1.set_title('Precision-Recall Curve'); ax1.legend(); ax1.set_xlim([0, 1]); ax1.set_ylim([0, 1])

    # Threshold vs Precision / Recall / F1 auf THRESHOLD-Set
    thr_sweep = np.arange(0.01, 1.0, 0.01)
    prec_sw = []; rec_sw = []; f1_sw = []
    for t in thr_sweep:
        p_ = (y_prob_threshold >= t).astype(int)
        prec_sw.append(precision_score(y_threshold, p_, zero_division=0))
        rec_sw.append(recall_score(y_threshold, p_, zero_division=0))
        f1_sw.append(f1_score(y_threshold, p_, zero_division=0))

    ax2.plot(thr_sweep, prec_sw, color='tomato', label='Precision')
    ax2.plot(thr_sweep, rec_sw,  color='steelblue', label='Recall')
    ax2.plot(thr_sweep, f1_sw,   color='green', linestyle='--', label='F1')
    ax2.axvline(best_threshold, color='black', linewidth=1.2,
                label=f'Best thr={best_threshold:.2f}')
    ax2.axvline(f1_thresh, color='purple', linestyle=':', linewidth=1.0,
                label=f'F1-opt={f1_thresh:.2f}')
    ax2.set_xlabel('Threshold'); ax2.set_ylabel('Score')
    ax2.set_title('THRESHOLD-Set — Threshold vs Metrics'); ax2.legend()
    ax2.set_xlim([0, 1]); ax2.set_ylim([0, 1])
    plt.tight_layout()
    plt.show()

    # ── THRESHOLD-Set raw sweep table ───────────────────────────────────────────
    print('\n── THRESHOLD-Set (raw) Sweep ──')
    rows_test = []
    for t in np.arange(0.05, 0.96, 0.05):
        preds = (y_prob_threshold >= t).astype(int)
        n_sig = preds.sum()
        tp    = int((preds * y_threshold).sum())
        fp    = int(preds.sum() - tp)
        prec  = precision_score(y_threshold, preds, zero_division=0)
        rec   = recall_score(y_threshold, preds, zero_division=0)
        f1    = f1_score(y_threshold, preds, zero_division=0)
        rows_test.append({'Thr': f'{t:.2f}', 'Prec': f'{prec:.2%}', 'Rec': f'{rec:.2%}',
                          'F1': f'{f1:.3f}', 'Signals': n_sig, 'TP': tp, 'FP': fp})
    print(pd.DataFrame(rows_test).to_string(index=False))

    # ── Signal filters ───────────────────────────────────────────────────────────
    def apply_signal_filters(df_ticker_prob, threshold,
                             consecutive_days=None, signal_cooldown_days=None):
        """
        Apply consecutive filter then cooldown filter for a single ticker.
        consecutive_days / signal_cooldown_days default to the global constants
        (which are updated after Optuna with the optimised values).
        Optional: skip signals at local price peaks (near N-day high) and/or very high RSI.
        Returns array of Date values where filtered signals fire.
        """
        cd  = CONSECUTIVE_DAYS     if consecutive_days     is None else consecutive_days
        scd = SIGNAL_COOLDOWN_DAYS if signal_cooldown_days is None else signal_cooldown_days

        df_s = df_ticker_prob.sort_values('Date').reset_index(drop=True)
        raw  = (df_s['prob'].values >= threshold).astype(np.int8)
        n    = len(raw)

        # Consecutive filter: at least cd of 3 consecutive days must be positive
        consec = np.zeros(n, dtype=np.int8)
        for i in range(2, n):
            if raw[i-2] + raw[i-1] + raw[i] >= cd:
                consec[i] = 1

        # Cooldown filter
        final = np.zeros(n, dtype=np.int8)
        last_signal = -999
        for i in range(n):
            if consec[i] == 1 and (i - last_signal) >= scd:
                final[i] = 1
                last_signal = i

        # Anti-Peak / RSI: RSI aus close (Cell 8: _rsi_from_close_1d), Fenster = rsi_w
        if 'close' in df_s.columns:
            rw = globals().get('rsi_w')
            rsi_series = _rsi_from_close_1d(df_s['close'].values, rw)
            mask_ok = _peak_rsi_mask_1d(
                df_s['close'].values,
                rsi_series,
                bool(globals().get('SIGNAL_SKIP_NEAR_PEAK', False)),
                int(globals().get('PEAK_LOOKBACK_DAYS', 20)),
                float(globals().get('PEAK_MIN_DIST_FROM_HIGH_PCT', 0.012)),
                globals().get('SIGNAL_MAX_RSI', None),
            )
            for i in range(n):
                if final[i] == 1 and not mask_ok[i]:
                    final[i] = 0

        return df_s.loc[final == 1, 'Date'].values


    # ── FINAL filtered threshold sweep table ─────────────────────────────────────
    print('\n── FINAL (filtered) Threshold Sweep ──')
    rows_final = []
    for t in np.arange(0.05, 0.96, 0.05):
        sig_dates_by_ticker = {}
        for ticker in final_tickers:
            sub = df_final[df_final['ticker'] == ticker]
            sig_dates_by_ticker[ticker] = apply_signal_filters(sub, t)

        # Compute precision over all FINAL tickers
        n_sig = 0; tp = 0; fp = 0
        for ticker, sig_dates in sig_dates_by_ticker.items():
            sub = df_final[df_final['ticker'] == ticker]
            for d in sig_dates:
                row = _rows_for_signal_calendar_day(sub, d)
                if row.empty:
                    continue
                n_sig += 1
                if row['target'].values[0] == 1:
                    tp += 1
                else:
                    fp += 1
        if n_sig == 0:
            continue
        prec = tp / n_sig
        check = '✓' if prec >= OPT_MIN_PRECISION else ''
        rows_final.append({'Thr': f'{t:.2f}', 'Prec': f'{prec:.2%}',
                           'Signals': n_sig, 'TP': tp, 'FP': fp,
                           f'≥{OPT_MIN_PRECISION:.0%}': check})
    print(pd.DataFrame(rows_final).to_string(index=False))

    # ── Summary ──────────────────────────────────────────────────────────────────
    print('\n── Result Summary ──')
    print(
        f'  Hinweis: Die Sweep-Tabellen oben zeigen Precision für *jedes* t (was-wäre-wenn). '
        f'Die Zeilen unten beziehen sich nur auf best_threshold={best_threshold:.3f} aus Phase 5 '
        f'(niedrigstes t mit roher Precision ≥ OPT_MIN_PRECISION={OPT_MIN_PRECISION:.0%}, sonst Fallback). '
        f'THRESHOLD (raw) = ohne Konsekutiv-/Cooldown-Filter; FINAL (filtered) = mit Filter.'
    )
    for label, y_true, y_prob_arr, tickers_list, df_part in [
            ('THRESHOLD (raw)',  y_threshold, y_prob_threshold, threshold_tickers,  df_threshold),
            ('FINAL (filtered)', y_final,     y_prob_final,     final_tickers,       df_final),
    ]:
        if label.startswith('THRESHOLD'):
            preds = (y_prob_arr >= best_threshold).astype(int)
            n_sig = preds.sum(); tp = int((preds * y_true).sum())
            fp = n_sig - tp
            prec = precision_score(y_true, preds, zero_division=0)
        else:
            n_sig = tp = fp = 0
            for t in tickers_list:
                sub = df_part[df_part['ticker'] == t]
                for d in apply_signal_filters(sub, best_threshold):
                    row = _rows_for_signal_calendar_day(sub, d)
                    if row.empty: continue
                    n_sig += 1
                    if row['target'].values[0] == 1: tp += 1
                    else: fp += 1
            prec = tp / n_sig if n_sig > 0 else 0.0

        status = 'PASS ✓' if prec >= OPT_MIN_PRECISION else 'MISS ✗'
        print(f'  {label:20s} | Signals={n_sig:4d} | TP={tp:4d} | FP={fp:4d} | '
              f'Precision={prec:.1%} | {status}  (gate: ≥{OPT_MIN_PRECISION:.0%})')


[SCORING_ONLY] Training-Zelle übersprungen.


In [66]:
if globals().get('SCORING_ONLY', False):
    print('[SCORING_ONLY] Training-Zelle übersprungen.')
else:
    # Cell 15 — 8d: Holdout plots (FINAL tickers)
    # Unbiased OOS: nur FINAL — df_test ist TRAIN_META (Meta-Training, kein echter Holdout für Performance).

    # Build filtered signals dict for FINAL tickers
    filtered_signals_final = {}
    for ticker in final_tickers:
        sub = df_final[df_final['ticker'] == ticker]
        filtered_signals_final[ticker] = apply_signal_filters(sub, best_threshold)

    signal_target_diag = summarize_filtered_signals_vs_target(
        df_final, filtered_signals_final, tickers=final_tickers)

    # Visualise
    plot_holdout_results(
        df_final,
        final_tickers,
        filtered_signals_final,
        title=f'FINAL Holdout — Threshold={best_threshold:.2f}'
    )
    print('Forward-Return-/Qualitätsanalyse: signals_holdout_final in signals.json (nicht die volle Historie "signals").')


[SCORING_ONLY] Training-Zelle übersprungen.


In [67]:
# Cell 17 — Daily Scoring & HTML Export  (full history edition)
# Prerequisite: Cells 1–10, 11 (Daten), ggf. 12–16 Training — oder SCORING_ONLY: 1–10, 11, 17.
# Scores the FULL history for all tickers, collects every past signal,
# and writes docs/index.html (+ docs/signals.json) for GitHub Pages.
# signals.json: "signals" = alle (inkl. In-Sample); "signals_holdout_final" = nur FINAL (unbiased OOS).
# data/master_complete.csv = volle Historie; master_daily_update.csv = letzter Signaltag, schlanke LLM-Spalten (ohne Forward/Labels).

import csv, json as _json, base64, io, warnings, os
from pathlib import Path
from datetime import datetime, timedelta

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')

docs_dir  = Path('docs')
docs_dir.mkdir(exist_ok=True)
today_str = datetime.today().strftime('%Y-%m-%d %H:%M')

# ─────────────────────────────────────────────────────────────────────────────
# Gemini: Schnelltest vor Scoring/HTML (nur wenn GEMINI_API_KEY gesetzt)
# ─────────────────────────────────────────────────────────────────────────────
import sys as _sys_smoke
_root_smoke = Path.cwd()
if str(_root_smoke) not in _sys_smoke.path:
    _sys_smoke.path.insert(0, str(_root_smoke))
from config.load_env import load_project_env
load_project_env(_root_smoke)
if os.environ.get('GEMINI_API_KEY', '').strip():
    try:
        from google import genai as _genai_smoke
        from google.genai import types as _gtypes_smoke
        _client_smoke = _genai_smoke.Client(
            api_key=os.environ['GEMINI_API_KEY'].strip(),
            http_options=_gtypes_smoke.HttpOptions(timeout=600_000),
        )
        _mraw = os.environ.get('GEMINI_MODEL', 'models/gemini-2.5-pro').strip()
        _model_smoke = _mraw[len('models/') :] if _mraw.startswith('models/') else _mraw
        _r_smoke = _client_smoke.models.generate_content(
            model=_model_smoke,
            contents='Antworte ausschließlich mit einem einzigen Wort: OK',
        )
        try:
            _txt_smoke = (_r_smoke.text or '').strip()[:500]
        except ValueError:
            _parts = []
            for c in _r_smoke.candidates or []:
                if not c.content or not c.content.parts:
                    continue
                for p in c.content.parts:
                    if hasattr(p, 'text') and p.text:
                        _parts.append(p.text)
            _txt_smoke = ''.join(_parts).strip()[:500]
        if not _txt_smoke:
            print('[Gemini] Smoke-Test: keine Text-Antwort — Zelle 17 abgebrochen.', flush=True)
            raise SystemExit(1)
        print(f'[Gemini] Smoke-Test OK ({_model_smoke}): {_txt_smoke!r}', flush=True)
    except SystemExit:
        raise
    except Exception as _e_smoke:
        print(
            f'[Gemini] Smoke-Test fehlgeschlagen ({type(_e_smoke).__name__}: {_e_smoke}) — Zelle 17 abgebrochen.',
            flush=True,
        )
        raise SystemExit(1)
else:
    print('[Gemini] Kein GEMINI_API_KEY — Smoke-Test übersprungen.', flush=True)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  Score the FULL history of df_features (all tickers, all dates)
# ─────────────────────────────────────────────────────────────────────────────
print('Preparing full-history feature matrix …')
df_s = df_features.copy()
feat_arr   = df_s[FEAT_COLS].values.astype(np.float32)
valid_mask = ~np.isnan(feat_arr).any(axis=1)
df_s       = df_s[valid_mask].reset_index(drop=True)
feat_arr   = feat_arr[valid_mask]
print(f'  {len(df_s):,} valid rows across {df_s["ticker"].nunique()} tickers — building meta features …')

X_meta_all = build_meta_features(feat_arr, dataset_label='FULL HISTORY')
probs_all  = meta_clf.predict_proba(X_meta_all)[:, 1]
df_s['prob'] = probs_all
print('  Scoring done.')

# ─────────────────────────────────────────────────────────────────────────────
# 2.  Apply signal filters per ticker → collect all historical signals
# ─────────────────────────────────────────────────────────────────────────────
all_hist_signals = []
for ticker in sorted(df_s['ticker'].unique()):
    sub       = df_s[df_s['ticker'] == ticker].sort_values('Date').reset_index(drop=True)
    sig_dates = apply_signal_filters(sub, best_threshold)
    for d in sig_dates:
        match = _rows_for_signal_calendar_day(sub, d)
        if match.empty:
            continue
        all_hist_signals.append({
            'ticker':  ticker,
            'company': COMPANY_NAMES.get(ticker, ticker),
            'sector':  TICKER_TO_SECTOR.get(ticker, '—'),
            'date':    str(d)[:10],
            'prob':    float(match['prob'].values[0]),
        })

# FINAL-Zeitfenster (df_final): unverzerrte Teststichprobe — nicht TRAIN_META/df_test
holdout_keys = set(zip(df_final['ticker'], pd.to_datetime(df_final['Date']).dt.strftime('%Y-%m-%d')))
signals_holdout_final = [s for s in all_hist_signals if (s['ticker'], s['date']) in holdout_keys]
signals_holdout_final.sort(key=lambda x: x['date'], reverse=True)
all_hist_signals.sort(key=lambda x: x['date'], reverse=True)
print(f'\n{len(all_hist_signals)} historical signals across all tickers.')
print(f'{len(signals_holdout_final)} signals in FINAL holdout (unbiased / OOS analysis).')

# Unbiased FINAL holdout → eine CSV (Meta + Forward-Renditen + Zusatzfilter)
Path('data').mkdir(parents=True, exist_ok=True)
_ho_fields = ['ticker', 'Date', 'prob', 'threshold_used', 'company', 'sector']
_ho_rows = [
    {
        'ticker': s['ticker'],
        'Date': s['date'],
        'prob': s['prob'],
        'threshold_used': float(best_threshold),
        'company': s['company'],
        'sector': s['sector'],
    }
    for s in signals_holdout_final
]
import pandas as pd
import sys as _sys
if str(Path.cwd()) not in _sys.path:
    _sys.path.insert(0, str(Path.cwd()))
from holdout.build_holdout_signals_master import main as _build_holdout_master
_build_holdout_master(holdout_df=pd.DataFrame(_ho_rows))
print(f'wrote data/master_complete.csv & master_daily_update.csv ({len(_ho_rows)} holdout rows)')

# "Recent" = last 30 days
recent_cutoff  = (pd.Timestamp(END_DATE) - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
recent_signals = [s for s in all_hist_signals if s['date'] >= recent_cutoff]
print(f'{len(recent_signals)} signals in the last 30 days.')

# ─────────────────────────────────────────────────────────────────────────────
# 3.  Price charts  (±1 Kalendermonat um Signaldatum, Tagesdaten)
#     Links: Close; rechts: %-Veränderung vs. Signalkurs (0 % = Signaltag).
#     Basis: df_features['close']; fehlende Handelstage bis Fensterende/heute werden für die Anzeige
#     immer per yfinance nachgezogen (history → download, 3 Versuche). Scoring bleibt aus df_features.
# ─────────────────────────────────────────────────────────────────────────────
_chart_yf_failures = []  # Ticker, bei denen der Nachzug trotz allem fehlschlug


def _yf_close_rows_from_series(ser):
    """Index = Zeit, Werte = Close → Liste {Date, close} mit kalendertag-normalisierten Dates."""
    rows = []
    if ser is None or len(ser) == 0:
        return rows
    s = ser.dropna()
    for _ti, _px in s.items():
        rows.append(
            {'Date': pd.Timestamp(pd.Timestamp(_ti).date()), 'close': float(_px)}
        )
    return rows


def _extend_chart_close_yfinance(ticker, win, win_lo, win_hi):
    """Fehlende Tage nach letztem df_features-Tag bis _chart_end; mutiert nicht das Scoring."""
    _last_feat = pd.to_datetime(win['Date']).max().normalize()
    _chart_end = min(pd.Timestamp(win_hi).normalize(), pd.Timestamp(datetime.today().date()).normalize())
    if _last_feat >= _chart_end:
        return win
    _d0 = (_last_feat + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    _d1 = (_chart_end + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    _rows = []
    _last_err = None
    for _attempt in range(3):
        try:
            _hist = yf.Ticker(ticker).history(
                start=_d0, end=_d1, interval='1d', auto_adjust=True, actions=False,
            )
            if _hist is not None and len(_hist) and 'Close' in _hist.columns:
                _rows = _yf_close_rows_from_series(_hist['Close'])
            if not _rows:
                _ext = yf.download(ticker, start=_d0, end=_d1, progress=False, threads=False)
                if _ext is not None and len(_ext) > 0:
                    _ser = _ext['Close']
                    if isinstance(_ser, pd.DataFrame):
                        _ser = _ser.iloc[:, 0]
                    _rows = _yf_close_rows_from_series(_ser)
            if _rows:
                break
        except Exception as _e:
            _last_err = _e
            time.sleep(0.35 * (_attempt + 1))
    if not _rows:
        _chart_yf_failures.append((ticker, _d0, _d1, repr(_last_err) if _last_err else 'keine Zeilen'))
        return win
    _w = pd.concat([win[['Date', 'close']].copy(), pd.DataFrame(_rows)], ignore_index=True)
    _w['Date'] = pd.to_datetime(_w['Date']).dt.normalize()
    _w = _w.drop_duplicates(subset=['Date'], keep='last').sort_values('Date')
    return _w[(_w['Date'] >= win_lo) & (_w['Date'] <= win_hi)].copy()


def _make_chart(ticker, sig_date_str):
    sub = df_s[df_s['ticker'] == ticker].sort_values('Date').reset_index(drop=True)
    if 'close' not in sub.columns or len(sub) < 5:
        return None
    sig_ts = pd.Timestamp(sig_date_str).normalize()
    win_lo = sig_ts - pd.DateOffset(months=1)
    win_hi = sig_ts + pd.DateOffset(months=1)
    win = sub[(sub['Date'] >= win_lo) & (sub['Date'] <= win_hi)].copy()
    if len(win) < 5:
        return None
    win = _extend_chart_close_yfinance(ticker, win, win_lo, win_hi)
    sig_row = win[pd.to_datetime(win['Date']).dt.normalize() == sig_ts]
    if not sig_row.empty:
        ref = float(sig_row['close'].iloc[0])
    else:
        j = int(pd.to_datetime(sub['Date']).searchsorted(sig_ts))
        j = min(max(j, 0), len(sub) - 1)
        ref = float(sub.iloc[j]['close'])
    if not np.isfinite(ref) or ref <= 0:
        return None
    try:
        close = win['close'].astype(float)
        dt = pd.to_datetime(win['Date'])
        pct = (close / ref - 1.0) * 100.0
        fig, ax = plt.subplots(figsize=(9, 3.2))
        ax.plot(dt, close, color='#42a5f5', lw=1.5, label='Close')
        ax.fill_between(dt, close, alpha=0.10, color='#42a5f5')
        ax2 = ax.twinx()
        ax2.plot(dt, pct, color='#ce93d8', lw=1.2, label='% vs. Datenstandstag')
        ax2.axhline(0.0, color='#66bb6a', lw=1.0, ls='-', alpha=0.9, zorder=4)
        ax.axvline(sig_ts, color='#66bb6a', lw=2, ls='--', zorder=5, label='Datenstand (0 %)')
        # Gleiches Kalenderfenster für alle Ticker: Strich liegt immer an derselben relativen Position
        # (sonst skaliert matplotlib nur auf die jeweiligen Daten → optisch verschoben).
        ax.set_xlim(win_lo, win_hi)
        ymin, ymax = float(close.min()), float(close.max())
        if ymax > ymin:
            pad = (ymax - ymin) * 0.02
            ax.set_ylim(ymin - pad, ymax + pad)
        else:
            span = max(abs(ymax), 1e-6) * 0.02
            ax.set_ylim(ymin - span, ymax + span)
        pmin, pmax = float(np.nanmin(pct)), float(np.nanmax(pct))
        ppad = max((pmax - pmin) * 0.06, 0.8)
        ax2.set_ylim(pmin - ppad, pmax + ppad)
        today_ts = pd.Timestamp(datetime.today().date()).normalize()
        if win_lo <= today_ts <= win_hi:
            ax.axvline(today_ts, color='#ffa726', lw=1.5, ls=':', zorder=5, label='Heute')
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, fontsize=7, loc='upper left', facecolor='#1a1a2e',
                  edgecolor='#2d2d4e', labelcolor='#e0e0e0')
        _ds = pd.Timestamp(sig_date_str).strftime('%d.%m.%Y')
        ax.set_title(
            f"{ticker} — {COMPANY_NAMES.get(ticker, ticker)}\n"
            f"Datenstand Modell: bis einschl. {_ds} (grüner Strich); rechts nur Kurs (Anzeige)",
            fontsize=8,
            color='#81d4fa',
        )
        ax.set_ylabel('Close', color='#90a4ae', fontsize=8)
        ax2.set_ylabel('% vs. Datenstand (Close)', color='#e1bee7', fontsize=8)
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m.%y'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right', fontsize=7)
        ax.tick_params(colors='#90a4ae', labelsize=7)
        ax2.tick_params(colors='#ce93d8', labelsize=7)
        ax.grid(True, alpha=0.18)
        ax.set_facecolor('#0d1117')
        fig.patch.set_facecolor('#1a1a2e')
        for sp in ax.spines.values():
            sp.set_edgecolor('#2d2d4e')
        ax2.spines['right'].set_edgecolor('#4a3f55')
        plt.tight_layout(pad=0.8)
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=95, bbox_inches='tight', facecolor=fig.get_facecolor())
        plt.close()
        buf.seek(0)
        return base64.b64encode(buf.read()).decode()
    except Exception:
        plt.close('all')
        return None

print('Generating charts …')
_chart_yf_failures.clear()
chart_cache = {}                            # (ticker, date) → b64 PNG
for s in all_hist_signals[:600]:           # cap at 600 to keep HTML manageable
    key = (s['ticker'], s['date'])
    if key not in chart_cache:
        b64 = _make_chart(s['ticker'], s['date'])
        if b64:
            chart_cache[key] = b64
print(f'  {len(chart_cache)} charts generated.')
if _chart_yf_failures:
    _bad = sorted({t for t, _d0, _d1, _e in _chart_yf_failures})
    print(
        f'  Warnung: Kursnachzug (yfinance): {len(_chart_yf_failures)} fehlgeschlagene Abrufe '
        f'({len(_bad)} eindeutige Ticker) — Charts enden am letzten df_features-Tag. '
        f'Ticker: {_bad[:25]}'
        f'{" …" if len(_bad) > 25 else ""}',
        flush=True,
    )

_yf_failed_tickers = {t for t, _d0, _d1, _e in _chart_yf_failures}

# ─────────────────────────────────────────────────────────────────────────────
# 4.  PR-curve plot (nur nach vollem Training; bei SCORING_ONLY entfallen)
# ─────────────────────────────────────────────────────────────────────────────
pr_b64 = ''
if not globals().get('SCORING_ONLY', False):
    try:
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
        fig.patch.set_facecolor('#1a1a2e')
        for ax, y_t, y_p, lbl, col in [
            (axes[0], y_threshold, y_prob_threshold, 'THRESHOLD', '#42a5f5'),
            (axes[1], y_final,     y_prob_final,     'FINAL',     '#66bb6a'),
        ]:
            pc, rc, _ = precision_recall_curve(y_t, y_p)
            ap = average_precision_score(y_t, y_p)
            ax.plot(rc, pc, color=col, lw=2, label=f'AP={ap:.3f}')
            ax.axhline(y=0.60, color='#ef5350', lw=1, ls='--', label='60 %')
            ax.set_title(f'PR — {lbl}', color='#81d4fa', fontsize=9)
            ax.set_xlabel('Recall', color='#90a4ae', fontsize=8)
            ax.set_ylabel('Precision', color='#90a4ae', fontsize=8)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.2)
            ax.set_facecolor('#0d1117')
            ax.tick_params(colors='#90a4ae', labelsize=7)
            for sp in ax.spines.values():
                sp.set_edgecolor('#2d2d4e')
        plt.tight_layout(pad=1.2)
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=95, bbox_inches='tight', facecolor=fig.get_facecolor())
        plt.close()
        buf.seek(0)
        pr_b64 = base64.b64encode(buf.read()).decode()
    except Exception as exc:
        print(f'PR plot failed: {exc}')
else:
    print('[SCORING_ONLY] PR-Kurven übersprungen (keine Trainings-Labels in dieser Session).')

# ─────────────────────────────────────────────────────────────────────────────
# 5.  Build HTML
# ─────────────────────────────────────────────────────────────────────────────
def _signal_card(s):
    key  = (s['ticker'], s['date'])
    b64  = chart_cache.get(key, '')
    bar  = int(s['prob'] * 100)
    _yf_note = (
        '<p class="yf-hint">Kursnachzug (yfinance) fehlgeschlagen — Chart endet am letzten Tag der '
        'Feature-Matrix; beim nächsten Lauf kann es wieder klappen.</p>'
        if b64 and s['ticker'] in _yf_failed_tickers else ''
    )
    chart_html = (
        f'<img src="data:image/png;base64,{b64}" alt="{s["ticker"]}" loading="lazy">'
        if b64 else ''
    )
    chart_note = (
        f'<p class="sig-chart-note" style="font-size:0.72em;color:#546e7a;margin-top:6px;line-height:1.35">'
        f'Merkmale und Kursbasis für dieses Signal: bis einschließlich <strong>{s["date"]}</strong> '
        f'(nicht der Zeitpunkt der Berechnung); rechts vom grünen Strich nur nachgelagerter Kurs '
        f'(Anzeige) — kein Look-ahead fürs Modell.</p>'
        if b64 else ''
    )
    return f"""
      <div class="sig-card">
        <div class="sig-head">
          <span class="sig-ticker">{s['ticker']}</span>
          <span class="sig-company">{s['company']}</span>
          <span class="sig-sector">{s['sector'].replace('_',' ').title()}</span>
          <span class="sig-date" title="Kurs- und Merkmalsdaten bis einschließlich diesem Tag — nicht der Laufzeitpunkt der Berechnung"><span class="sig-date-pre">Daten bis</span> {s['date']}</span>
          <div class="score-bar-bg"><div class="score-bar" style="width:{bar}%">{s['prob']:.3f}</div></div>
        </div>
        {_yf_note}
        {chart_html}
        {chart_note}
      </div>"""

recent_html = ''.join(_signal_card(s) for s in recent_signals) or '<p class="empty">Keine Signale in den letzten 30 Tagen.</p>'
hist_html   = ''.join(_signal_card(s) for s in all_hist_signals[:600]) or '<p class="empty">Keine historischen Signale.</p>'

pr_section = (
    f'<div class="section"><h2>Model Quality &#8212; Precision-Recall</h2>'
    f'<img src="data:image/png;base64,{pr_b64}" alt="PR" style="max-width:100%;border-radius:6px"></div>'
    if pr_b64 else ''
)
recent_badge_cls = ' zero' if not recent_signals else ''

import html as _html_std
_prompt_path = Path('docs') / 'website_analysis_prompt.txt'
if _prompt_path.is_file():
    _ANALYSIS_PROMPT_DE = _prompt_path.read_text(encoding='utf-8')
else:
    _ANALYSIS_PROMPT_DE = '[docs/website_analysis_prompt.txt fehlt]'
try:
    from lib.website_rally_prompt import load_rally_prompt_injection
    _rally_llm_block = load_rally_prompt_injection(Path.cwd())
    if _rally_llm_block.strip():
        _ANALYSIS_PROMPT_DE = _rally_llm_block.rstrip() + '\n\n---\n\n' + _ANALYSIS_PROMPT_DE
except Exception:
    pass
_ANALYSIS_PROMPT_ESC = _html_std.escape(_ANALYSIS_PROMPT_DE)

import os as _os_llm
import sys as _sys_llm
try:
    _root_llm = Path.cwd()
    if str(_root_llm) not in _sys_llm.path:
        _sys_llm.path.insert(0, str(_root_llm))
    from config.load_env import load_project_env
    load_project_env(_root_llm)
except Exception:
    pass
_analysis_llm_section = ''
_script_gemini = Path('scripts') / 'run_website_analysis_gemini.py'
_gemini_key = _os_llm.environ.get('GEMINI_API_KEY', '').strip()
if _gemini_key and _script_gemini.is_file():
    import subprocess as _sp_llm
    try:
        print('Gemini: KI-Analyse (CSV-Upload + Google Search) …', flush=True)
        _r = _sp_llm.run(
            [_sys_llm.executable, str(_script_gemini)],
            cwd=str(Path.cwd()),
            capture_output=True,
            text=True,
            timeout=600,
        )
        if _r.returncode != 0:
            print('LLM-Analyse Fehler:', (_r.stderr or _r.stdout or '')[:2500], flush=True)
        _hf = Path('docs') / 'analysis_llm_last.html'
        if _hf.is_file():
            _analysis_llm_section = _hf.read_text(encoding='utf-8')
        else:
            print('Hinweis: docs/analysis_llm_last.html fehlt nach Gemini-Lauf. Ausgabe:', ((_r.stdout or '') + (_r.stderr or ''))[:3000], flush=True)
    except Exception as _e_llm:
        print('LLM-Analyse:', _e_llm, flush=True)
else:
    if not _gemini_key:
        print('Hinweis: GEMINI_API_KEY setzen — keine automatische KI-Analyse.', flush=True)

# Spätester Kalendertag in der Feature-Matrix (Kurs/Merkmale) — für Modell-Info-Chip
_matrix_last_date = str(pd.to_datetime(df_s['Date']).max().date())
_thr_cal = globals().get('threshold_calibration_end_date')
if _thr_cal is None and 'df_threshold' in dir() and df_threshold is not None:
    try:
        _thr_cal = str(pd.to_datetime(df_threshold['Date']).max().date())
    except Exception:
        _thr_cal = None
if isinstance(_thr_cal, str) and len(_thr_cal) >= 10:
    _thr_cal = _thr_cal[:10]
_thr_chip = (
    f'<div class="chip" title="Letzter Handelstag der THRESHOLD-Stichprobe bei der Kalibrierung von best_threshold">'
    f'Threshold-Kalibrierung: Stichprobe bis <strong>{_thr_cal}</strong></div>'
    if _thr_cal
    else '<div class="chip" title="Artefakt mit Zelle 18 neu speichern (nach Training) — dann steht das Datum der THRESHOLD-Stichprobe im Bundle.">'
    'Threshold-Kalibrierung: Stichprobe bis <strong>—</strong> (nicht im Artefakt)</div>'
)

html = f"""<!DOCTYPE html>
<html lang="de">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width,initial-scale=1.0">
  <meta name="theme-color" content="#16213e">
  <link rel="manifest" href="manifest.json">
  <title>Stock Signals</title>
  <style>
    *{{box-sizing:border-box;margin:0;padding:0}}
    body{{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;background:#0f0f1a;color:#e0e0e0;min-height:100vh}}
    header{{background:#16213e;padding:12px 18px;display:flex;align-items:center;justify-content:space-between;position:sticky;top:0;z-index:100;box-shadow:0 2px 8px rgba(0,0,0,.6)}}
    header h1{{font-size:1.1em;color:#81d4fa}}
    header .ts{{font-size:.72em;color:#546e7a}}
    .page-wrap{{display:flex;align-items:flex-start;gap:18px;max-width:1320px;margin:0 auto;padding:12px 16px}}
    .main-col{{flex:1;min-width:0;max-width:900px}}
    .llm-sidebar{{width:min(340px,100%);flex-shrink:0;position:sticky;top:52px;align-self:flex-start;max-height:calc(100vh - 64px);overflow-y:auto;background:#141428;border:1px solid #2d2d4e;border-radius:10px;padding:10px 12px}}
    .llm-details summary{{cursor:pointer;color:#81d4fa;font-size:.92em;font-weight:600;padding:6px 4px;user-select:none;list-style:none}}
    .llm-details summary::-webkit-details-marker{{display:none}}
    .llm-details summary::before{{content:'▸ ';color:#64b5f6}}
    .llm-details[open] summary::before{{content:'▾ '}}
    .llm-sum-hint{{font-weight:400;font-size:.78em;color:#78909c;margin-left:6px}}
    .llm-slot{{margin-top:8px;padding-top:8px;border-top:1px solid #2a2a44}}
    .llm-sidebar .analysis-llm h2{{display:none}}
    .llm-sidebar .section.analysis-llm{{background:transparent;padding:0;margin:0;border:none}}
    .llm-sidebar .analysis-llm-body{{max-width:none;font-size:.88em}}
    .yf-hint{{font-size:.68em;color:#78909c;margin:0 0 6px;line-height:1.35}}
    .section{{background:#1a1a2e;border-radius:10px;padding:16px;margin-bottom:14px}}
    .section h2{{font-size:.95em;color:#81d4fa;margin-bottom:12px;border-bottom:1px solid #2d2d4e;padding-bottom:7px;display:flex;align-items:center;gap:8px}}
    .badge{{background:#4caf50;color:#fff;border-radius:10px;padding:1px 8px;font-size:.72em}}
    .badge.zero{{background:#607d8b}}
    .sig-card{{background:#0d1117;border-radius:8px;padding:12px;margin-bottom:10px}}
    .sig-head{{display:flex;flex-wrap:wrap;align-items:center;gap:8px;margin-bottom:8px}}
    .sig-ticker{{font-weight:700;color:#81d4fa;font-size:.95em;min-width:70px}}
    .sig-company{{color:#90a4ae;font-size:.82em;flex:1;min-width:100px}}
    .sig-sector{{background:#1e2a3a;color:#64b5f6;font-size:.72em;padding:2px 7px;border-radius:10px;white-space:nowrap;align-self:center}}
    .sig-date{{color:#546e7a;font-size:.78em;white-space:nowrap}}
    .sig-date-pre{{font-size:.68em;color:#78909c;margin-right:5px}}
    .score-bar-bg{{background:#1a1a2e;border-radius:4px;overflow:hidden;min-width:70px;max-width:110px}}
    .score-bar{{background:#4caf50;padding:2px 6px;color:#fff;font-size:.75em;white-space:nowrap;border-radius:4px}}
    img{{max-width:100%;border-radius:5px;display:block}}
    .model-chips{{display:flex;flex-wrap:wrap;gap:8px}}
    .chip{{background:#0d1117;border-radius:6px;padding:5px 10px;font-size:.8em;color:#90a4ae}}
    .chip strong{{color:#e0e0e0}}
    .empty{{color:#546e7a;font-size:.85em;padding:10px 0}}
    details summary{{cursor:pointer;color:#81d4fa;font-size:.9em;padding:4px 0;user-select:none;list-style:none}}
    details summary::before{{content:'&#9654; '}}
    details[open] summary::before{{content:'&#9660; '}}
    details[open] summary{{margin-bottom:10px}}
    .prompt-lead{{font-size:.85em;color:#90a4ae;margin-bottom:10px;line-height:1.4}}
    .prompt-pre{{font-size:.78em;line-height:1.45;color:#b0bec5;white-space:pre-wrap;word-break:break-word;background:#0d1117;border-radius:8px;padding:12px;margin-top:8px;border:1px solid #2d2d4e}}
    .analysis-llm-body{{font-size:.92em;line-height:1.62;color:#eceff1;max-width:52em}}
    .analysis-llm-body.prose-analysis p{{margin:0 0 12px}}
    .analysis-llm-body h3{{font-size:1.05em;font-weight:600;color:#81d4fa;margin:18px 0 8px;border-bottom:1px solid #2d3f50;padding-bottom:4px}}
    .analysis-llm-body h4{{font-size:.98em;font-weight:600;color:#b3e5fc;margin:14px 0 6px}}
    .analysis-llm-body strong{{color:#fff;font-weight:600}}
    .analysis-llm-body .analysis-ul,.analysis-llm-body .analysis-ol{{margin:6px 0 14px 1.1em;padding:0}}
    .analysis-llm-body li{{margin:5px 0;padding-left:2px}}
    .analysis-llm-body .analysis-code{{font-family:ui-monospace,Consolas,monospace;font-size:.88em;background:#1a1a2e;color:#b0bec5;padding:2px 7px;border-radius:4px;border:1px solid #2d2d4e}}
    .analysis-llm-body .analysis-bq{{margin:10px 0;padding:10px 14px;border-left:3px solid #4fc3f7;background:#12121f;border-radius:0 6px 6px 0;color:#b0bec5}}
    .analysis-llm-body .analysis-hr{{border:none;border-top:1px solid #37474f;margin:16px 0}}
    .analysis-llm-missing h2{{font-size:.95em;color:#ffb74d;margin-bottom:8px}}
    @media(max-width:960px){{.page-wrap{{flex-direction:column}}.llm-sidebar{{position:relative;top:0;max-height:none;width:100%;order:-1}}}}
    @media(max-width:600px){{header h1{{font-size:.95em}}}}
  </style>
</head>
<body>
<header>
  <h1>&#128200; Stock Signals</h1>
  <span class="ts" title="Zeitpunkt dieses Export-Laufs (nicht der Kursdatenstand)">Lauf: {today_str}</span>
</header>
<div class="page-wrap">
<main class="main-col">

  <div class="section">
    <h2>Letzte 30 Tage <span class="badge{recent_badge_cls}">{len(recent_signals)}</span></h2>
    {recent_html}
  </div>

  <div class="section">
    <h2>Model Info</h2>
    <div class="model-chips">
      <div class="chip">Threshold <strong>{best_threshold:.3f}</strong></div>
      <div class="chip">Tickers <strong>{df_s['ticker'].nunique()}</strong></div>
      <div class="chip">Signale gesamt <strong>{len(all_hist_signals)}</strong></div>
      <div class="chip">FINAL Holdout OOS <strong>{len(signals_holdout_final)}</strong></div>
      {_thr_chip}
      <div class="chip" title="Letzter Handelstag der Merkmals-/Kursmatrix in diesem Scoring-Lauf (Volllauf über df_features)">Scoring &amp; Features bis <strong>{_matrix_last_date}</strong></div>
      <div class="chip">Run <strong>{today_str}</strong></div>
    </div>
  </div>

  <div class="section">
    <details>
      <summary>KI-Analyse-Prompt</summary>
      <p class="prompt-lead">Vorgabe für ergänzende Einordnung der Meta-Hits (aktuellster Signaltag in den Daten; Schluss: welche Aktie — falls eine — am ehesten infrage kommt). Keine Anlageberatung.</p>
      <pre class="prompt-pre">{_ANALYSIS_PROMPT_ESC}</pre>
    </details>
  </div>

  {pr_section}

  <div class="section">
    <details>
      <summary>Alle historischen Signale &mdash; {len(all_hist_signals)} gesamt (max. 600 angezeigt)</summary>
      {hist_html}
    </details>
  </div>

</main>

<aside class="llm-sidebar" aria-label="KI-Analyse">
  <details class="llm-details" open>
    <summary>KI-Analyse <span class="llm-sum-hint">(ein-/ausklappen)</span></summary>
    <div class="llm-slot">
      __ANALYSIS_LLM_SECTION__
    </div>
  </details>
</aside>

</div>
</body>
</html>"""

_anal_fallback = (
    '<div class="section analysis-llm-missing"><h2>KI-Antwort</h2>'
    '<p class="empty"><strong>Lokal:</strong> <code>.env</code> mit <code>GEMINI_API_KEY</code> (Projektroot). '
    'Dann <code>python scripts/run_website_analysis_gemini.py</code> — es muss '
    '<code>docs/analysis_llm_last.html</code> erscheinen. Anschließend <strong>diese Zelle (17)</strong> erneut ausführen.</p>'
    '<p class="empty"><strong>Daten:</strong> Vorher <code>data/master_daily_update.csv</code> erzeugen (holdout/build_holdout_signals_master, z. B. nach Holdout-Export).</p>'
    '<p class="empty"><strong>GitHub Pages:</strong> Ohne Commit von <code>docs/analysis_llm_last.html</code> und neuem '
    '<code>docs/index.html</code> bleibt die Analyse auf der Website leer — API-Keys liegen nicht im Repo.</p>'
    '</div>'
)
html = html.replace('__ANALYSIS_LLM_SECTION__', _analysis_llm_section.strip() if _analysis_llm_section.strip() else _anal_fallback)

(docs_dir / 'index.html').write_text(html, encoding='utf-8')
(docs_dir / 'signals.json').write_text(
    _json.dumps(
        {'generated': today_str, 'threshold': float(best_threshold),
         'signals': all_hist_signals,
         'signals_holdout_final': signals_holdout_final,
         'note': 'signals_holdout_final = FINAL OOS. master_complete = Historie+Forward; master_daily_update = letzter Tag, LLM-Spalten.'},
        indent=2, default=str,
    ),
    encoding='utf-8',
)

print(f'\ndocs/index.html   {len(html):,} bytes')
print(f'docs/signals.json {len(all_hist_signals)} signals total ({len(signals_holdout_final)} FINAL OOS)')
print(f'\nOpen docs/index.html in a browser to preview.')

# ── Auto git commit + push ───────────────────────────────────────────────────
import subprocess as _sp
_msg = (
    f'Daily signal update {today_str} '
    f'({len(all_hist_signals)} signals, '
    f'{len(recent_signals)} in last 30 days)'
)
_git_docs = [
    'docs/index.html', 'docs/signals.json', 'docs/website_analysis_prompt.txt',
    'docs/analysis_llm_last.html', 'docs/analysis_llm_last.txt',
]
_git_to_add = [p for p in _git_docs if Path(p).is_file()]
try:
    if not _git_to_add:
        print('\nGit: keine der erwarteten docs-Dateien gefunden — Commit/Push übersprungen.')
    else:
        _sp.run(['git', 'add', '--'] + _git_to_add, check=True)
        result = _sp.run(['git', 'diff', '--cached', '--quiet'])
        if result.returncode != 0:  # there are staged changes
            _sp.run(['git', 'commit', '-m', _msg], check=True)
            _sp.run(['git', 'push'], check=True)
            print(f'\nGit: committed and pushed — "{_msg}"')
        else:
            print('\nGit: no changes to commit (docs already up to date).')
except Exception as _e:
    print(f'\nGit push failed: {_e}')


[Gemini] Smoke-Test OK (gemini-2.5-pro): 'OK'
Preparing full-history feature matrix …
  366,816 valid rows across 193 tickers — building meta features …

--- FULL HISTORY: 366,816 Samples ---
  [XGB-1] scoring 366,816 Zeilen (FULL HISTORY)... 0.9s
  [XGB-2] scoring 366,816 Zeilen (FULL HISTORY)... 0.8s
  [XGB-3] scoring 366,816 Zeilen (FULL HISTORY)... 0.8s
  [XGB-4] scoring 366,816 Zeilen (FULL HISTORY)... 0.9s
  [LGB-1] scoring 366,816 Zeilen (FULL HISTORY)... 12.1s
  [LGB-2] scoring 366,816 Zeilen (FULL HISTORY)... 11.6s
  [LGB-3] scoring 366,816 Zeilen (FULL HISTORY)... 11.5s
  [RF] scoring 366,816 Zeilen (FULL HISTORY)... 4.0s
  [ET] scoring 366,816 Zeilen (FULL HISTORY)... 3.8s
  [LR] scoring 366,816 Zeilen (FULL HISTORY)... 0.2s
  Meta-Matrix Shape: (366816, 20)
  Scoring done.

30874 historical signals across all tickers.
4133 signals in FINAL holdout (unbiased / OOS analysis).
Download 192 Ticker (2018-01-01 … 2026-05-19) …
Train-Label-Eval: return_window=10, rally_threshold=0

$AIXA.DE: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)
$AIXA.DE: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)

1 Failed download:
['AIXA.DE']: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)
$AIXA.DE: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)
$AIXA.DE: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)

1 Failed download:
['AIXA.DE']: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)
$AIXA.DE: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)
$AIXA.DE: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)

1 Failed download:
['AIXA.DE']: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)
$AOF.DE: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)
$AOF.DE: possibly delisted; no price data found  (1d 2026-04-03 -> 2026-04-07)

1 Failed download:
['AOF.DE']: possibly delisted; no

  600 charts generated.
  Warnung: Kursnachzug (yfinance): 25 fehlgeschlagene Abrufe (23 eindeutige Ticker) — Charts enden am letzten df_features-Tag. Ticker: ['1U1.DE', 'AIXA.DE', 'AOF.DE', 'CRM', 'DOW', 'ELG.DE', 'ETH-USD', 'G24.DE', 'GFT.DE', 'GXI.DE', 'HLAG.DE', 'HYQ.DE', 'IBM', 'LYB', 'NDX1.DE', 'NEM.DE', 'SMHN.DE', 'SOL-USD', 'UNH', 'UTDI.DE', 'VBK.DE', 'WAF.DE', 'YSN.DE']
[SCORING_ONLY] PR-Kurven übersprungen (keine Trainings-Labels in dieser Session).
Gemini: KI-Analyse (CSV-Upload + Google Search) …

docs/index.html   59,702,195 bytes
docs/signals.json 30874 signals total (4133 FINAL OOS)

Open docs/index.html in a browser to preview.

Git: committed and pushed — "Daily signal update 2026-04-10 06:15 (30874 signals, 369 in last 30 days)"


In [68]:
# Cell 18 — Artefakt manuell speichern (optional)
# Nach vollem Training wird automatisch in Cell 14 (Phase 5) gespeichert; Cell 17 sichert nach, falls nötig.
# Diese Zelle nur, wenn du ohne Cell 14/17 erneut auf Platte schreiben willst (gleicher Kernel, Modelle im RAM).

import sys
from pathlib import Path

_root18 = Path.cwd().resolve()
if str(_root18) not in sys.path:
    sys.path.insert(0, str(_root18))

import lib.scoring_persist as scoring_persist

Path("models").mkdir(parents=True, exist_ok=True)
scoring_persist.save_scoring_artifacts(globals(), Path("models/scoring_artifacts.joblib"))
globals()["_SCORING_ARTIFACT_SAVED_THIS_SESSION"] = True

Gespeichert: models\scoring_artifacts.joblib  (threshold=0.7100)


In [69]:
from google import genai
import os

# Client initialisieren
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# Modelle auflisten
print("Verfügbare Modelle im neuen SDK:")
for model in client.models.list():
    # Wir filtern nach Modellen, die Text generieren können
    print(f"Name: {model.name} | Support: {model.supported_actions}")

Verfügbare Modelle im neuen SDK:
Name: models/gemini-2.5-flash | Support: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.5-pro | Support: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.0-flash | Support: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.0-flash-001 | Support: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.0-flash-lite-001 | Support: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.0-flash-lite | Support: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Name: models/gemini-2.5-flash-preview-tts | Support: ['countTokens', 'generateContent']
Name: models/gemini-2.5-pro-preview-tts | Support: ['countTokens', 'generateContent', 'batchGenerateContent']
Name: models/ge